# Gemma 4 E4B — Blackwell LoRA Train (iterate loop)

Run-read-fix loop on `GEMMA_BLACKWELL_POOL` (RTX PRO 6000 Blackwell, single GPU).

**Order:** env probe -> pull model + unpack dataset -> model-load smoke (correct class) -> short smoke train -> batch sweep -> full train + save.

Notes verified from stage artifacts:
- Model is `Gemma4ForConditionalGeneration` (multimodal) -> must NOT load via `AutoModelForCausalLM`.
- Dataset is a `.tar.zst` that unpacks to `train/*.jsonl`, `eval/*.jsonl` -> must pull+unpack and point paths to globs.

In [ ]:
# Cell 1 - Environment & GPU probe
import sys, platform, json, subprocess
print("python:", sys.version.split()[0], "|", platform.platform())

def ver(mod):
    try:
        import importlib
        m = importlib.import_module(mod)
        return getattr(m, "__version__", "unknown")
    except Exception as e:
        return f"MISSING ({type(e).__name__})"

info = {m: ver(m) for m in ["torch","transformers","peft","trl","datasets","accelerate","liger_kernel","bitsandbytes","zstandard"]}
print(json.dumps(info, indent=2))

try:
    import torch
    print("cuda_available:", torch.cuda.is_available(), "| device_count:", torch.cuda.device_count(), "| torch.version.cuda:", torch.version.cuda)
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name} | sm_{p.major}{p.minor} | {p.total_memory/1e9:.1f} GB")
except Exception as e:
    print("torch probe error:", repr(e))

try:
    out = subprocess.run(["nvidia-smi","--query-gpu=name,driver_version,memory.total","--format=csv,noheader"], capture_output=True, text=True)
    print("nvidia-smi:", out.stdout.strip() or out.stderr.strip())
except Exception as e:
    print("nvidia-smi error:", repr(e))

In [ ]:
# Cell 2 - Does current transformers support gemma4? + pip/internet capability
import transformers, subprocess, sys
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print("transformers:", transformers.__version__)
print("gemma-ish model_types known:", [k for k in CONFIG_MAPPING_NAMES if "gemma" in k])
print("gemma4 supported:", "gemma4" in CONFIG_MAPPING_NAMES)

# quick outbound-internet / pip probe
try:
    r = subprocess.run([sys.executable,"-m","pip","install","--quiet","--dry-run","zstandard"], capture_output=True, text=True, timeout=90)
    print("pip dry-run rc:", r.returncode)
    print((r.stdout or "")[-500:])
    print((r.stderr or "")[-500:])
except Exception as e:
    print("pip probe error:", repr(e))

In [ ]:
# Cell 3 - Offline install from stage wheelhouse (NO internet). Keep runtime torch 2.9.1+cu128.
import os, sys, subprocess, glob, time, importlib
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session = get_active_session()

STAGE = "@ML_DB.MODELS.FUZZ_DATASET_STAGE/gemma4_e4b_smart_pack/offline_env_v2/blackwell_fast10_trl_liger_rslora_20260605"
CORE = Path("/tmp/wh_core"); CORE.mkdir(parents=True, exist_ok=True)

if not glob.glob(str(CORE)+"/**/*.whl", recursive=True):
    print("GET wheelhouse_core ...", flush=True)
    session.file.get(f"{STAGE}/wheelhouse_core", str(CORE))
whls = glob.glob(str(CORE)+"/**/*.whl", recursive=True)
print(f"core wheels: {len(whls)}")

def pip_install(*pkgs):
    cmd = [sys.executable,"-m","pip","install","--no-index","--find-links",str(CORE)] + list(pkgs)
    print("$", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("STDOUT tail:\n", (r.stdout or "")[-1500:])
    if r.returncode != 0:
        print("STDERR tail:\n", (r.stderr or "")[-2000:])
    print("rc =", r.returncode)
    return r.returncode

# transformers 5.x REQUIRED for gemma4. No -U, no torch in list -> keep installed torch 2.9.1+cu128.
rc = pip_install("transformers==5.10.1", "trl", "liger-kernel")
print("install rc:", rc)

# re-verify in a fresh interpreter (avoid stale modules in this kernel)
chk = subprocess.run([sys.executable,"-c",
    "import transformers,trl,liger_kernel,torch;"
    "from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES;"
    "print('transformers',transformers.__version__,'trl',trl.__version__,'liger',liger_kernel.__version__);"
    "print('torch',torch.__version__,'cuda',torch.version.cuda);"
    "print('gemma4 supported:', 'gemma4' in CONFIG_MAPPING_NAMES)"], capture_output=True, text=True)
print(chk.stdout); print(chk.stderr[-800:])

In [ ]:
# Cell 4 - Verify upgraded stack in-kernel (after restart)
import transformers, trl, liger_kernel, torch, peft, accelerate, datasets
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
import importlib.metadata as md
print("transformers", transformers.__version__)
print("trl", trl.__version__)
print("liger-kernel", md.version("liger-kernel"))
print("peft", peft.__version__, "| accelerate", accelerate.__version__, "| datasets", datasets.__version__)
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device_count", torch.cuda.device_count())
print("gemma4 supported:", "gemma4" in CONFIG_MAPPING_NAMES)
print("gemma4 model->classes:")
from transformers.models.auto.modeling_auto import MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES as M
print("  gemma4 in ImageTextToText map:", "gemma4" in M)

In [ ]:
# Cell 5 - Pull base model + dataset, unpack dataset tar.zst -> train/eval jsonl
import os, glob, time, json, tarfile, shutil
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session = get_active_session()
STAGE = "@ML_DB.MODELS.FUZZ_DATASET_STAGE/gemma4_e4b_smart_pack"

# 1) base model (~16GB)
BASE = Path("/tmp/gemma4_base"); BASE.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(BASE)+"/**/*.safetensors", recursive=True):
    print("GET base model (~16GB)...", flush=True); t=time.time()
    session.file.get(f"{STAGE}/models/google__gemma-4-E4B-it", str(BASE))
    print("model GET done in %.0fs" % (time.time()-t))
cfgs = glob.glob(str(BASE)+"/**/config.json", recursive=True)
MODEL_DIR = str(Path(cfgs[0]).parent) if cfgs else str(BASE)
print("MODEL_DIR:", MODEL_DIR)

# 2) dataset archive
DS = Path("/tmp/ds"); DS.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(DS)+"/*.tar.zst"):
    print("GET dataset archive...", flush=True)
    session.file.get(f"{STAGE}/datasets/quality4_fuzzing_real_20260604_174450", str(DS))
arch = glob.glob(str(DS)+"/*.tar.zst")[0]
print("archive:", arch, os.path.getsize(arch), "bytes")

# 3) unpack via pyarrow zstd codec (no zstd CLI / no zstandard lib available)
import pyarrow as pa
UNP = Path("/tmp/ds_unpacked")
if UNP.exists(): shutil.rmtree(UNP)
UNP.mkdir(parents=True, exist_ok=True)
with pa.OSFile(arch, "rb") as src:
    with pa.CompressedInputStream(src, "zstd") as stream:
        with tarfile.open(fileobj=stream, mode="r|") as tf:
            tf.extractall(UNP)
print("unpacked top:", sorted(os.listdir(UNP)))

trains = sorted(glob.glob(str(UNP)+"/**/train/*.jsonl", recursive=True)) or sorted(glob.glob(str(UNP)+"/**/train*.jsonl", recursive=True))
evals  = sorted(glob.glob(str(UNP)+"/**/eval/*.jsonl", recursive=True))  or sorted(glob.glob(str(UNP)+"/**/eval*.jsonl", recursive=True))
def count_lines(fs):
    n=0
    for f in fs:
        with open(f) as fh: n+=sum(1 for _ in fh)
    return n
ntr, nev = count_lines(trains), count_lines(evals)
print("train files:", len(trains), "| train rows:", ntr)
print("eval files:", len(evals), "| eval rows:", nev)
TRAIN_GLOB = (str(Path(trains[0]).parent)+"/*.jsonl") if trains else ""
EVAL_GLOB  = (str(Path(evals[0]).parent)+"/*.jsonl") if evals else ""
print("TRAIN_GLOB:", TRAIN_GLOB); print("EVAL_GLOB:", EVAL_GLOB)
if trains:
    with open(trains[0]) as fh: print("sample keys:", list(json.loads(fh.readline()).keys()))

In [ ]:
# Cell 6 - Model-load smoke (correct class) + module inspect + liger gemma4 support
import time, torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForImageTextToText

cfg = AutoConfig.from_pretrained(MODEL_DIR)
print("model_type:", cfg.model_type, "| arch:", cfg.architectures)
print("sub-configs present:", [k for k in ["text_config","vision_config","audio_config"] if hasattr(cfg,k)])

# liger support check
try:
    from liger_kernel.transformers.monkey_patch import MODEL_TYPE_TO_APPLY_LIGER_FN as LIGER_MAP
    print("liger-supported model_types:", sorted(LIGER_MAP.keys()))
    print("liger supports gemma4:", "gemma4" in LIGER_MAP, "| gemma3:", "gemma3" in LIGER_MAP)
except Exception as e:
    print("liger map check error:", repr(e))

tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
print("tokenizer:", type(tok).__name__, "| pad:", tok.pad_token, "| eos:", tok.eos_token)

t=time.time()
model = AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
print("loaded", type(model).__name__, "in %.0fs" % (time.time()-t))
print("total params (B): %.3f" % (sum(p.numel() for p in model.parameters())/1e9))
print("top-level children:", [n for n,_ in model.named_children()])
# inspect where the language tower lives + linear names
import collections
lin = collections.Counter()
for n,m in model.named_modules():
    if m.__class__.__name__ == "Linear":
        lin[n.split('.')[-1]] += 1
print("linear leaf names:", dict(lin))
# show a couple of language layer paths
lm_paths = [n for n,_ in model.named_modules() if n.endswith('q_proj')][:3]
print("sample q_proj paths:", lm_paths)

In [ ]:
# Cell 7 - Locate language tower + count LoRA targets under it (text-only scope)
import re
lm_prefixes = sorted({n for n,_ in model.named_modules() if n.endswith('language_model')})
print("language_model module paths:", lm_prefixes)
for nm,_ in model.named_modules():
    if nm.endswith('language_model'):
        print("language_model class:", _.__class__.__name__)
        try: print("language_model.config.model_type:", _.config.model_type)
        except Exception as e: print("no sub-config:", e)
        break
PROJ = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGET_REGEX = r".*language_model\..*\.(" + "|".join(PROJ) + r")$"
matched = [n for n,m in model.named_modules() if m.__class__.__name__=="Linear" and re.fullmatch(TARGET_REGEX, n)]
vision_hits = [n for n in matched if 'vision' in n or 'audio' in n]
print("LoRA target Linear count (language only):", len(matched))
print("any vision/audio leaked into targets:", len(vision_hits))
print("examples:", matched[:3], "...", matched[-1:])
print("TARGET_REGEX =", TARGET_REGEX)

In [ ]:
# Cell 8 - Write corrected DDP train script to /tmp (used by torchrun subprocess)
TRAIN_PY = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]; EVAL_GLOB=os.environ.get("EVAL_GLOB","")
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out")
MAX_LEN=int(os.environ.get("MAX_LEN","2048")); BATCH=int(os.environ.get("BATCH","4"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","1")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","1e-4"))
LORA_R=int(os.environ.get("LORA_R","32")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","64"))
USE_RSLORA=os.environ.get("USE_RSLORA","0")=="1"; USE_PACKING=os.environ.get("USE_PACKING","1")=="1"
USE_LIGER=os.environ.get("USE_LIGER","0")=="1"; GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
ALL_LINEAR=os.environ.get("ALL_LINEAR","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0"))
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True

if USE_LIGER:
    try:
        from liger_kernel.transformers.monkey_patch import _apply_liger_kernel
        _apply_liger_kernel("gemma4_text")
        if local_rank==0: print("LIGER: applied to gemma4_text", flush=True)
    except Exception as e:
        if local_rank==0: print("LIGER: patch failed, continuing without ->", repr(e), flush=True)

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
model.config.use_cache=False

PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
if ALL_LINEAR:
    TARGETS=r".*language_model\..*_proj$"
else:
    TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
lora=LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
                bias="none", target_modules=TARGETS, use_rslora=USE_RSLORA)
model=get_peft_model(model, lora)
if local_rank==0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

df={"train":sorted(glob.glob(TRAIN_GLOB))}
if EVAL_GLOB and glob.glob(EVAL_GLOB): df["validation"]=sorted(glob.glob(EVAL_GLOB))
ds=load_dataset("json", data_files=df)

def safe_cfg(**kw):
    allowed=set(inspect.signature(SFTConfig.__init__).parameters)
    drop=[k for k in kw if k not in allowed]
    if drop and local_rank==0: print("SFTConfig dropping unsupported:", drop, flush=True)
    return SFTConfig(**{k:v for k,v in kw.items() if k in allowed})

args=safe_cfg(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=2,
    save_steps=100000, save_total_limit=1, gradient_checkpointing=GRAD_CKPT, ddp_find_unused_parameters=False,
    dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", max_length=MAX_LEN,
    dataset_text_field="text", packing=USE_PACKING, eval_strategy="no", include_num_input_tokens_seen=True)

kw=dict(model=model, args=args, train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)

t0=time.time(); res=trainer.train(); dt=time.time()-t0
if local_rank==0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0)
    mem=torch.cuda.max_memory_allocated()/1e9
    summary={"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),
             "steps":trainer.state.global_step,"max_vram_gb":round(mem,1),
             "tokens_seen":ntok,"tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None,
             "world_size":int(os.environ.get("WORLD_SIZE","1"))}
    print("SUMMARY "+json.dumps(summary), flush=True)
    if os.environ.get("SAVE_MODEL","0")=="1":
        trainer.save_model(OUT_DIR); tok.save_pretrained(OUT_DIR); print("saved to", OUT_DIR, flush=True)
'''
with open("/tmp/train_gemma4_blackwell.py","w") as f: f.write(TRAIN_PY)
print("wrote /tmp/train_gemma4_blackwell.py", len(TRAIN_PY), "bytes")

In [ ]:
# Cell 9 - launch helper (torchrun subprocess). Re-run cell 8 first if script changed.
import os, sys, subprocess, time

def run_train(nproc=2, extra_env=None, timeout=3600):
    env=os.environ.copy()
    env.update({
        "MODEL_DIR": MODEL_DIR, "TRAIN_GLOB": TRAIN_GLOB, "EVAL_GLOB": EVAL_GLOB,
        "HF_HUB_OFFLINE":"1", "TRANSFORMERS_OFFLINE":"1", "TOKENIZERS_PARALLELISM":"false",
        "PYTORCH_ALLOC_CONF":"expandable_segments:True",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True", "NCCL_DEBUG":"WARN",
        "OMP_NUM_THREADS":"12", "CUDA_DEVICE_MAX_CONNECTIONS":"1",
    })
    if extra_env: env.update({k:str(v) for k,v in extra_env.items()})
    cmd=[sys.executable,"-m","torch.distributed.run","--standalone",f"--nproc_per_node={nproc}",
         "/tmp/train_gemma4_blackwell.py"]
    keys=["MAX_LEN","BATCH","GRAD_ACC","MAX_STEPS","EPOCHS","LORA_R","USE_PACKING","USE_LIGER","USE_RSLORA","GRADIENT_CHECKPOINTING","ALL_LINEAR"]
    print("$ torchrun --nproc_per_node=%d ; cfg=" % nproc, {k:env[k] for k in keys if k in env})
    t0=time.time()
    p=subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=timeout)
    out=(p.stdout or "")+"\n"+(p.stderr or "")
    summ=[l for l in out.splitlines() if l.startswith("SUMMARY") or "trainable params" in l or l.startswith("LIGER") or "OutOfMemory" in l]
    print("\n".join(summ[-8:]) if summ else out[-2000:])
    print(f"[wall {time.time()-t0:.0f}s, rc={p.returncode}]")
    return p.returncode, out

In [ ]:
# Cell 9b - Fix: peft 0.17.1 incompatible with transformers 5.10.1 -> upgrade peft from wheelhouse
import sys, subprocess
r = subprocess.run([sys.executable,"-m","pip","install","--no-index","--find-links","/tmp/wh_core","-U","peft"],
                   capture_output=True, text=True)
print((r.stdout or "")[-600:]); print((r.stderr or "")[-600:]); print("rc", r.returncode)
# verify peft<->transformers import in fresh interpreter
chk = subprocess.run([sys.executable,"-c",
  "import peft,transformers; from peft import LoraConfig,get_peft_model; "
  "print('peft',peft.__version__,'transformers',transformers.__version__,'-> import OK')"],
  capture_output=True, text=True)
print(chk.stdout); print(chk.stderr[-800:])

In [ ]:
# Cell 9c - inspect exact error + dataset text field health
import re, json, glob
m = re.search(r'(Error|Exception)[^\n]*', out)
for line in out.splitlines():
    if 'Error' in line or 'endswith' in line or 'NoneType' in line or 'has no attribute' in line:
        print(line)
print("-----")
# check text field types across train
bad=0; total=0; types={}
for f in glob.glob(TRAIN_GLOB)+glob.glob(EVAL_GLOB):
    with open(f) as fh:
        for ln in fh:
            if not ln.strip(): continue
            total+=1; ex=json.loads(ln); t=ex.get('text')
            types[type(t).__name__]=types.get(type(t).__name__,0)+1
            if not isinstance(t,str) or not t: bad+=1
print('total rows:',total,'| text types:',types,'| bad(non-str/empty):',bad)

In [ ]:
# Cell 9d - inspect schema of text=None rows vs text=str rows
import json, glob, collections
none_keys=collections.Counter(); str_keys=collections.Counter(); none_example=None; sg_none=collections.Counter()
for f in glob.glob(TRAIN_GLOB)+glob.glob(EVAL_GLOB):
    with open(f) as fh:
        for ln in fh:
            if not ln.strip(): continue
            ex=json.loads(ln); t=ex.get('text')
            if isinstance(t,str) and t: str_keys.update(ex.keys())
            else:
                none_keys.update(ex.keys()); sg_none.update([ex.get('source_group')])
                if none_example is None: none_example=ex
print('keys in text=None rows:', dict(none_keys))
print('keys in text=str rows :', dict(str_keys))
print('source_group of None rows:', dict(sg_none))
print('--- sample None-text row (truncated) ---')
if none_example:
    for k,v in none_example.items():
        s=json.dumps(v, ensure_ascii=False) if not isinstance(v,str) else v
        print(f'  {k}: {s[:300]}')

In [ ]:
# Cell 10 - Normalize mixed dataset -> single 'text' column (chat rows via chat_template)
import json, glob, os
from pathlib import Path
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def merge_system(msgs):
    out=[]; carry=""
    for m in msgs:
        if m.get('role')=='system': carry += (m.get('content','')+"\n\n"); continue
        if m.get('role')=='user' and carry:
            out.append({'role':'user','content':carry+m.get('content','')}); carry=""
        else: out.append(m)
    if carry and out: out[0]['content']=carry+out[0].get('content','')
    return out

def to_text(ex):
    t = ex.get('text')
    if isinstance(t,str) and t.strip(): return t
    msgs = ex.get('messages')
    if msgs:
        if isinstance(msgs,str): msgs=json.loads(msgs)
        try:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        except Exception:
            return tok.apply_chat_template(merge_system(msgs), tokenize=False, add_generation_prompt=False)
    return None

CLEAN=Path('/tmp/ds_clean'); (CLEAN/'train').mkdir(parents=True,exist_ok=True); (CLEAN/'eval').mkdir(parents=True,exist_ok=True)
def convert(globpat, dst):
    n=0; fail=0
    with open(dst,'w',encoding='utf-8') as w:
        for f in glob.glob(globpat):
            with open(f) as fh:
                for ln in fh:
                    if not ln.strip(): continue
                    ex=json.loads(ln); txt=to_text(ex)
                    if not (isinstance(txt,str) and txt.strip()): fail+=1; continue
                    w.write(json.dumps({'text':txt}, ensure_ascii=False)+"\n"); n+=1
    return n, fail

ntr,ftr = convert(TRAIN_GLOB, CLEAN/'train'/'part.jsonl')
nev,fev = convert(EVAL_GLOB,  CLEAN/'eval'/'part.jsonl') if EVAL_GLOB else (0,0)
print('clean train rows:',ntr,'(failed',ftr,') | clean eval rows:',nev,'(failed',fev,')')
TRAIN_GLOB=str(CLEAN/'train'/'*.jsonl'); EVAL_GLOB=str(CLEAN/'eval'/'*.jsonl')
print('NEW TRAIN_GLOB:',TRAIN_GLOB); print('NEW EVAL_GLOB:',EVAL_GLOB)
# sanity: show length stats in tokens
import numpy as np
lens=[]
with open(CLEAN/'train'/'part.jsonl') as fh:
    for i,ln in enumerate(fh):
        if i>=400: break
        lens.append(len(tok(json.loads(ln)['text']).input_ids))
lens=np.array(lens); print('token len (n=%d): mean %.0f p50 %.0f p90 %.0f max %d'%(len(lens),lens.mean(),np.percentile(lens,50),np.percentile(lens,90),lens.max()))

In [ ]:
# Cell 10b - extract SUMMARY + trainable params from last run
for line in out.splitlines():
    if line.startswith('SUMMARY') or 'trainable params' in line or 'LIGER' in line:
        print(line)

In [ ]:
# Cell 11 - 2-GPU DDP + Liger(text-tower) test @ seq4096/batch8/r32
rc, out = run_train(nproc=2, extra_env={
    "MAX_LEN":"4096","BATCH":"8","GRAD_ACC":"1","MAX_STEPS":"15",
    "LORA_R":"32","LORA_ALPHA":"64","USE_PACKING":"1","USE_LIGER":"1",
    "OUT_DIR":"/tmp/out_2gpu","SAVE_MODEL":"0"})
print("rc:", rc)

In [ ]:
# Cell 12 - Batch/seq sweep (2-GPU, packing, liger, r32, grad-ckpt) - FITTING configs
import json
def parse_summary(out):
    for l in out.splitlines():
        if l.startswith("SUMMARY"):
            try: return json.loads(l[len("SUMMARY"):].strip())
            except: return None
    return None

# tokens/step/GPU = seq*batch shown for intuition (logits memory ~ tokens*vocab)
sweep = [
    dict(MAX_LEN=2048, BATCH=4, GRADIENT_CHECKPOINTING=1),   # 8192 tok
    dict(MAX_LEN=4096, BATCH=2, GRADIENT_CHECKPOINTING=1),   # 8192 tok
    dict(MAX_LEN=2048, BATCH=8, GRADIENT_CHECKPOINTING=1),   # 16384 tok
    dict(MAX_LEN=4096, BATCH=4, GRADIENT_CHECKPOINTING=1),   # 16384 tok
    dict(MAX_LEN=2048, BATCH=12, GRADIENT_CHECKPOINTING=1),  # 24576 tok
]
results=[]
for cfg in sweep:
    base=dict(LORA_R="32",LORA_ALPHA="64",USE_PACKING="1",USE_LIGER="1",MAX_STEPS="12",GRAD_ACC="1",
              OUT_DIR="/tmp/out_sweep",SAVE_MODEL="0")
    base.update({k:str(v) for k,v in cfg.items()})
    rc,out = run_train(nproc=2, extra_env=base, timeout=1200)
    s=parse_summary(out); oom=("OutOfMemory" in out)
    row=dict(seq=cfg["MAX_LEN"],batch=cfg["BATCH"],tok_step=cfg["MAX_LEN"]*cfg["BATCH"],rc=rc,oom=oom,
             tok_s=(s or {}).get("tokens_per_s"),vram=(s or {}).get("max_vram_gb"),loss=(s or {}).get("train_loss"))
    results.append(row); print("=>", row)
print("\n==== SWEEP TABLE (2-GPU) ====")
print("%-6s %-6s %-9s %-4s %-6s %-12s %-8s %-8s"%("seq","batch","tok/step","rc","oom","tokens/s","vram","loss"))
for r in results:
    print("%-6s %-6s %-9s %-4s %-6s %-12s %-8s %-8s"%(r["seq"],r["batch"],r["tok_step"],r["rc"],r["oom"],r["tok_s"],r["vram"],r["loss"]))
ok=[r for r in results if r["rc"]==0 and r["tok_s"]]
if ok:
    best=max(ok,key=lambda r:r["tok_s"]) ; print("\nBEST throughput:",best)

In [ ]:
# Cell 13 - Probe NO-checkpointing small-token configs (avoid recompute) @ 20 steps
sweep2 = [
    dict(MAX_LEN=4096, BATCH=2, GRADIENT_CHECKPOINTING=1),  # current best ref
    dict(MAX_LEN=4096, BATCH=1, GRADIENT_CHECKPOINTING=0),  # 4096 tok, no recompute
    dict(MAX_LEN=2048, BATCH=2, GRADIENT_CHECKPOINTING=0),  # 4096 tok, no recompute
    dict(MAX_LEN=3072, BATCH=2, GRADIENT_CHECKPOINTING=0),  # 6144 tok, no recompute
]
res2=[]
for cfg in sweep2:
    base=dict(LORA_R="32",LORA_ALPHA="64",USE_PACKING="1",USE_LIGER="1",MAX_STEPS="20",GRAD_ACC="1",
              OUT_DIR="/tmp/out_sweep2",SAVE_MODEL="0")
    base.update({k:str(v) for k,v in cfg.items()})
    rc,out = run_train(nproc=2, extra_env=base, timeout=900)
    s=parse_summary(out); oom=("OutOfMemory" in out)
    row=dict(seq=cfg["MAX_LEN"],batch=cfg["BATCH"],ckpt=cfg["GRADIENT_CHECKPOINTING"],rc=rc,oom=oom,
             tok_s=(s or {}).get("tokens_per_s"),vram=(s or {}).get("max_vram_gb"),loss=(s or {}).get("train_loss"))
    res2.append(row); print("=>", row)
print("\n==== NO-CKPT SWEEP ====")
print("%-6s %-6s %-5s %-4s %-6s %-10s %-7s %-7s"%("seq","batch","ckpt","rc","oom","tokens/s","vram","loss"))
for r in res2:
    print("%-6s %-6s %-5s %-4s %-6s %-10s %-7s %-7s"%(r["seq"],r["batch"],r["ckpt"],r["rc"],r["oom"],r["tok_s"],r["vram"],r["loss"]))
okall=[r for r in res2 if r["rc"]==0 and r["tok_s"]]
if okall: print("\nBEST:", max(okall,key=lambda r:r["tok_s"]))

In [ ]:
# Cell 14 - FULL TRAINING (2-GPU, seq4096/batch1 no-ckpt, liger, r64+rsLoRA, 2 epochs) + save adapter
rc, out = run_train(nproc=2, extra_env={
    "MAX_LEN":"4096", "BATCH":"1", "GRAD_ACC":"8", "GRADIENT_CHECKPOINTING":"0",
    "USE_PACKING":"1", "USE_LIGER":"1",
    "LORA_R":"64", "LORA_ALPHA":"128", "USE_RSLORA":"1",
    "LR":"7e-5", "EPOCHS":"2", "MAX_STEPS":"-1",
    "OUT_DIR":"/tmp/gemma4_lora_main", "SAVE_MODEL":"1"
}, timeout=5400)
print("FULL TRAIN rc:", rc)
# show progress lines (loss curve) + summary
import re
loss_lines=[l for l in out.splitlines() if re.search(r"'loss':", l)]
print("\n".join(loss_lines[-15:]))
for l in out.splitlines():
    if l.startswith("SUMMARY") or l.startswith("saved to"): print(l)

In [ ]:
# Cell 15 - list saved adapter + inference sanity (base + trained LoRA) via 1-GPU subprocess
import os, glob
ADAPTER="/tmp/gemma4_lora_main"
print("adapter files:", sorted(os.listdir(ADAPTER)))

GEN = r'''
import os, torch
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import PeftModel
MODEL_DIR=os.environ["MODEL_DIR"]; ADAPTER=os.environ["ADAPTER"]
tok=AutoTokenizer.from_pretrained(MODEL_DIR)
m=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to("cuda:0")
m=PeftModel.from_pretrained(m, ADAPTER).to("cuda:0"); m.eval()
msgs=[{"role":"user","content":"Sinh 1 seed fuzzing cho torch.nn.functional.conv2d va de xuat oracle phat hien crash. Tra loi ngan gon."}]
enc=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
enc={k:v.to("cuda:0") for k,v in enc.items()}
with torch.no_grad():
    o=m.generate(**enc, max_new_tokens=220, do_sample=False)
print("=== GENERATION ===")
print(tok.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))
'''
with open("/tmp/gen_test.py","w") as f: f.write(GEN)
import subprocess, sys
env=os.environ.copy(); env.update({"MODEL_DIR":MODEL_DIR,"ADAPTER":ADAPTER,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/gen_test.py"], env=env, capture_output=True, text=True, timeout=900)
print(p.stdout[-2500:]); print("ERR:", p.stderr[-600:])

In [ ]:
# Cell 16 - PUT trained LoRA adapter back to stage
import os, glob, json
ADAPTER="/tmp/gemma4_lora_main"
DEST="@ML_DB.MODELS.FUZZ_DATASET_STAGE/gemma4_e4b_smart_pack/trained/gemma4_e4b_lora_r64_rslora_blackwell_20260605"

# write a small training summary alongside the adapter
summary={
  "base_model":"google__gemma-4-E4B-it (Gemma4ForConditionalGeneration)",
  "load_class":"AutoModelForImageTextToText",
  "method":"BF16 LoRA DDP(2x RTX PRO 6000 Blackwell) + TRL SFT packing + Liger(gemma4_text) + rsLoRA",
  "lora":{"r":64,"alpha":128,"rslora":True,"target":"language_model.*_proj (text tower only)","trainable_params":"139.5M (1.73%)"},
  "train":{"seq":4096,"per_device_batch":1,"grad_acc":8,"world_size":2,"epochs":2,"lr":"7e-5","packing":True,"grad_ckpt":False},
  "result":{"train_loss_final":"~0.55","mean_token_accuracy":"~0.85","tokens_per_s":7100,"train_seconds":670,"max_vram_gb":63.8},
  "dataset":"quality4_fuzzing_real_20260604_174450 (train 4750 / eval 67, normalized text+messages)"
}
with open(os.path.join(ADAPTER,"training_summary.json"),"w") as f: json.dump(summary,f,ensure_ascii=False,indent=2)

# upload top-level adapter files (skip the checkpoint-74 dir)
files=[p for p in glob.glob(ADAPTER+"/*") if os.path.isfile(p)]
print("uploading", len(files), "files ->", DEST)
for p in files:
    session.file.put(p, DEST, auto_compress=False, overwrite=True)
    print("  PUT", os.path.basename(p), os.path.getsize(p))
print("DONE")

In [ ]:
# Cell 17 - Inspect liger internals to find a fused-linear-CE forward usable for the gemma4 wrapper
import inspect, liger_kernel
from liger_kernel.transformers import monkey_patch as mp
apply_fns = [n for n in dir(mp) if n.startswith('apply_liger_kernel_to_')]
print('apply fns:', apply_fns)
print('has gemma3 wrapper fn:', 'apply_liger_kernel_to_gemma3' in apply_fns)
print('has gemma4_text fn   :', 'apply_liger_kernel_to_gemma4_text' in apply_fns)
print('has gemma3_text fn   :', 'apply_liger_kernel_to_gemma3_text' in apply_fns)
# show signature + source head of the gemma3 (multimodal wrapper) apply fn as a template for gemma4
for fn in ['apply_liger_kernel_to_gemma3','apply_liger_kernel_to_gemma4_text']:
    if hasattr(mp, fn):
        print('\n==== ', fn, inspect.signature(getattr(mp, fn)))
        src = inspect.getsource(getattr(mp, fn))
        print(src[:1600])

In [ ]:
# Cell 18 - Deep inspect: gemma4_text apply source, wrapper loss path, FLCE signature
import inspect
from liger_kernel.transformers import monkey_patch as mp
print('=== apply_liger_kernel_to_gemma4_text FULL ===')
print(inspect.getsource(mp.apply_liger_kernel_to_gemma4_text))
print('\n=== does liger ship a model.gemma4 lce_forward? ===')
try:
    import importlib
    g4 = importlib.import_module('liger_kernel.transformers.model.gemma4')
    print('gemma4 model module members:', [n for n in dir(g4) if 'forward' in n or 'lce' in n.lower()])
except Exception as e:
    print('no liger model.gemma4 module:', repr(e))

In [ ]:
# Cell 19 - check causal_forward signature + write NEW FLCE train script (separate file)
import inspect, importlib
g4 = importlib.import_module('liger_kernel.transformers.model.gemma4')
print('causal_forward signature:')
print(inspect.signature(g4.causal_forward))

FLCE_PY = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from liger_kernel.transformers.model.gemma4 import causal_forward as gemma4_causal_forward

MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]; EVAL_GLOB=os.environ.get("EVAL_GLOB","")
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_flce")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); BATCH=int(os.environ.get("BATCH","4"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","1")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
USE_RSLORA=os.environ.get("USE_RSLORA","1")=="1"; USE_PACKING=os.environ.get("USE_PACKING","1")=="1"
GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0"))
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
model.config.use_cache=False
# *** KEY: patch the wrapper forward with liger fused-linear-CE forward (no full logits) ***
type(model).forward = gemma4_causal_forward
if local_rank==0: print("FLCE: patched", type(model).__name__, ".forward = gemma4 causal_forward", flush=True)

PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
lora=LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
                bias="none", target_modules=TARGETS, use_rslora=USE_RSLORA)
model=get_peft_model(model, lora)
if local_rank==0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

df={"train":sorted(glob.glob(TRAIN_GLOB))}
ds=load_dataset("json", data_files=df)

def safe_cfg(**kw):
    allowed=set(inspect.signature(SFTConfig.__init__).parameters)
    return SFTConfig(**{k:v for k,v in kw.items() if k in allowed})
args=safe_cfg(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=2,
    save_steps=100000, save_total_limit=1, gradient_checkpointing=GRAD_CKPT, ddp_find_unused_parameters=False,
    dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", max_length=MAX_LEN,
    dataset_text_field="text", packing=USE_PACKING, eval_strategy="no", include_num_input_tokens_seen=True)
kw=dict(model=model, args=args, train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if local_rank==0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),
        "steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,
        "tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None,"world_size":int(os.environ.get("WORLD_SIZE","1"))}), flush=True)
    if os.environ.get("SAVE_MODEL","0")=="1":
        trainer.save_model(OUT_DIR); tok.save_pretrained(OUT_DIR); print("saved to", OUT_DIR, flush=True)
'''
with open("/tmp/train_gemma4_flce.py","w") as f: f.write(FLCE_PY)
print("wrote /tmp/train_gemma4_flce.py", len(FLCE_PY), "bytes")

def run_flce(nproc=2, extra_env=None, timeout=2400):
    import sys, subprocess
    env=os.environ.copy()
    env.update({"MODEL_DIR":MODEL_DIR,"TRAIN_GLOB":TRAIN_GLOB,"EVAL_GLOB":EVAL_GLOB,
        "HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","TOKENIZERS_PARALLELISM":"false",
        "PYTORCH_ALLOC_CONF":"expandable_segments:True","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "NCCL_DEBUG":"WARN","OMP_NUM_THREADS":"12"})
    if extra_env: env.update({k:str(v) for k,v in extra_env.items()})
    cmd=[sys.executable,"-m","torch.distributed.run","--standalone",f"--nproc_per_node={nproc}","/tmp/train_gemma4_flce.py"]
    keys=["MAX_LEN","BATCH","GRAD_ACC","MAX_STEPS","EPOCHS","LORA_R","USE_PACKING","USE_RSLORA","GRADIENT_CHECKPOINTING"]
    print("$ torchrun --nproc=%d cfg="%nproc, {k:env[k] for k in keys if k in env})
    p=subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=timeout)
    o=(p.stdout or "")+"\n"+(p.stderr or "")
    keep=[l for l in o.splitlines() if l.startswith("SUMMARY") or "trainable params" in l or l.startswith("FLCE") or "OutOfMemory" in l or "Error" in l]
    print("\n".join(keep[-10:]) if keep else o[-2500:]); print("rc=",p.returncode)
    return p.returncode, o

In [ ]:
# Cell 20 - FLCE correctness smoke (2-GPU, seq4096/batch1, 8 steps)
rc, o = run_flce(nproc=2, extra_env={"MAX_LEN":"4096","BATCH":"1","GRAD_ACC":"1","MAX_STEPS":"8",
    "LORA_R":"64","LORA_ALPHA":"128","USE_RSLORA":"1","USE_PACKING":"1","GRADIENT_CHECKPOINTING":"0"})
print("FLCE smoke rc:", rc)

In [ ]:
# Cell 21 - rewrite FLCE script with config fix (copy text_config attrs to top-level config)
FLCE_PY2 = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from liger_kernel.transformers.model.gemma4 import causal_forward as gemma4_causal_forward

MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_flce")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); BATCH=int(os.environ.get("BATCH","4"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","1")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
USE_RSLORA=os.environ.get("USE_RSLORA","1")=="1"; USE_PACKING=os.environ.get("USE_PACKING","1")=="1"
GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0"))
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
model.config.use_cache=False
# FIX: expose text_config scalars on the top-level config (FLCE forward reads self.config.hidden_size/vocab_size)
tc=getattr(model.config,"text_config",None)
for a in ["hidden_size","vocab_size","pad_token_id","final_logit_softcapping"]:
    if tc is not None and hasattr(tc,a): setattr(model.config,a,getattr(tc,a))
if local_rank==0: print("CONFIG hidden_size=",getattr(model.config,"hidden_size",None),"vocab_size=",getattr(model.config,"vocab_size",None), flush=True)
# KEY: patch wrapper forward with liger fused-linear-CE forward (no full logits materialized)
type(model).forward = gemma4_causal_forward
if local_rank==0: print("FLCE: patched", type(model).__name__, flush=True)

PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
lora=LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
                bias="none", target_modules=TARGETS, use_rslora=USE_RSLORA)
model=get_peft_model(model, lora)
if local_rank==0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

ds=load_dataset("json", data_files={"train":sorted(glob.glob(TRAIN_GLOB))})
def safe_cfg(**kw):
    allowed=set(inspect.signature(SFTConfig.__init__).parameters)
    return SFTConfig(**{k:v for k,v in kw.items() if k in allowed})
args=safe_cfg(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=2,
    save_steps=100000, save_total_limit=1, gradient_checkpointing=GRAD_CKPT, ddp_find_unused_parameters=False,
    dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", max_length=MAX_LEN,
    dataset_text_field="text", packing=USE_PACKING, eval_strategy="no", include_num_input_tokens_seen=True)
kw=dict(model=model, args=args, train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if local_rank==0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),
        "steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,
        "tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None,"world_size":int(os.environ.get("WORLD_SIZE","1"))}), flush=True)
    if os.environ.get("SAVE_MODEL","0")=="1":
        trainer.save_model(OUT_DIR); tok.save_pretrained(OUT_DIR); print("saved to", OUT_DIR, flush=True)
'''
with open("/tmp/train_gemma4_flce.py","w") as f: f.write(FLCE_PY2)
print("rewrote /tmp/train_gemma4_flce.py", len(FLCE_PY2), "bytes (run_flce from Cell 19 reuses this path)")

In [ ]:
# Cell 22 - print FLCE traceback to locate the NoneType subscript
lines=o.splitlines()
idx=[i for i,l in enumerate(lines) if 'Traceback' in l or 'subscriptable' in l or 'causal_forward' in l or 'gemma4' in l.lower()]
start=max(0,(idx[0]-1)) if idx else max(0,len(lines)-60)
for l in lines[start:start+60]:
    print(l)

In [ ]:
# Cell 23 - deepest frames of FLCE error
for l in o.splitlines()[-45:]:
    print(l)

In [ ]:
# Cell 24 - FLCE script v3: custom Trainer.compute_loss uses model's FLCE loss (logits=None)
FLCE_PY3 = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from liger_kernel.transformers.model.gemma4 import causal_forward as gemma4_causal_forward

MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_flce")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); BATCH=int(os.environ.get("BATCH","4"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","1")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
USE_RSLORA=os.environ.get("USE_RSLORA","1")=="1"; USE_PACKING=os.environ.get("USE_PACKING","1")=="1"
GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0"))
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
model.config.use_cache=False
tc=getattr(model.config,"text_config",None)
for a in ["hidden_size","vocab_size","pad_token_id","final_logit_softcapping"]:
    if tc is not None and hasattr(tc,a): setattr(model.config,a,getattr(tc,a))
type(model).forward = gemma4_causal_forward
if local_rank==0: print("FLCE: patched", type(model).__name__, "hidden=",model.config.hidden_size, flush=True)

PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
lora=LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
                bias="none", target_modules=TARGETS, use_rslora=USE_RSLORA)
model=get_peft_model(model, lora)
if local_rank==0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

ds=load_dataset("json", data_files={"train":sorted(glob.glob(TRAIN_GLOB))})
def safe_cfg(**kw):
    allowed=set(inspect.signature(SFTConfig.__init__).parameters)
    return SFTConfig(**{k:v for k,v in kw.items() if k in allowed})
args=safe_cfg(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=2,
    save_steps=100000, save_total_limit=1, gradient_checkpointing=GRAD_CKPT, ddp_find_unused_parameters=False,
    dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", max_length=MAX_LEN,
    dataset_text_field="text", packing=USE_PACKING, eval_strategy="no", include_num_input_tokens_seen=True)

class FLCETrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if num_items_in_batch is not None: inputs["num_items_in_batch"]=num_items_in_batch
        out=model(**inputs)
        loss=out.loss if hasattr(out,"loss") else out["loss"]
        return (loss, out) if return_outputs else loss

kw=dict(model=model, args=args, train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=FLCETrainer(**kw)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if local_rank==0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),
        "steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,
        "tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None,"world_size":int(os.environ.get("WORLD_SIZE","1"))}), flush=True)
    if os.environ.get("SAVE_MODEL","0")=="1":
        trainer.save_model(OUT_DIR); tok.save_pretrained(OUT_DIR); print("saved to", OUT_DIR, flush=True)
'''
with open("/tmp/train_gemma4_flce.py","w") as f: f.write(FLCE_PY3)
print("rewrote /tmp/train_gemma4_flce.py v3", len(FLCE_PY3), "bytes")

In [ ]:
# Cell 25 - read causal_forward source; find exact config attrs it needs
import inspect, importlib
g4 = importlib.import_module('liger_kernel.transformers.model.gemma4')
src = inspect.getsource(g4.causal_forward)
import re
for i,l in enumerate(src.splitlines()):
    if 'config' in l or 'vocab' in l or 'hidden' in l or 'self.' in l and ('size' in l or 'norm' in l or 'softcap' in l):
        print(i, l)

In [ ]:
# Cell 26 - print causal_forward body lines 78-120 to see FLCE-vs-logits branch condition
lines = inspect.getsource(g4.causal_forward).splitlines()
for i in range(78,121):
    if i < len(lines): print(i, lines[i])

In [ ]:
# Cell 27 - FLCE script v4: force skip_logits=True (FLCE branch) + set model.vocab_size + debug keys
FLCE_PY4 = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from liger_kernel.transformers.model.gemma4 import causal_forward as gemma4_causal_forward

MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_flce")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); BATCH=int(os.environ.get("BATCH","4"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","1")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
USE_RSLORA=os.environ.get("USE_RSLORA","1")=="1"; USE_PACKING=os.environ.get("USE_PACKING","1")=="1"
GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0"))
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
model.config.use_cache=False
tc=getattr(model.config,"text_config",None)
for a in ["hidden_size","vocab_size","pad_token_id","final_logit_softcapping"]:
    if tc is not None and hasattr(tc,a): setattr(model.config,a,getattr(tc,a))
model.vocab_size=getattr(model.config,"vocab_size")  # used by liger non-FLCE path; set for safety
type(model).forward = gemma4_causal_forward
if local_rank==0: print("FLCE: patched; hidden=",model.config.hidden_size,"vocab=",model.vocab_size, flush=True)

PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
lora=LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
                bias="none", target_modules=TARGETS, use_rslora=USE_RSLORA)
model=get_peft_model(model, lora)
if local_rank==0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

ds=load_dataset("json", data_files={"train":sorted(glob.glob(TRAIN_GLOB))})
def safe_cfg(**kw):
    allowed=set(inspect.signature(SFTConfig.__init__).parameters)
    return SFTConfig(**{k:v for k,v in kw.items() if k in allowed})
args=safe_cfg(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=2,
    save_steps=100000, save_total_limit=1, gradient_checkpointing=GRAD_CKPT, ddp_find_unused_parameters=False,
    dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", max_length=MAX_LEN,
    dataset_text_field="text", packing=USE_PACKING, eval_strategy="no", include_num_input_tokens_seen=True)

class FLCETrainer(SFTTrainer):
    _dbg=False
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if not FLCETrainer._dbg and int(os.environ.get("LOCAL_RANK","0"))==0:
            print("INPUT KEYS:", sorted(inputs.keys()), flush=True); FLCETrainer._dbg=True
        if num_items_in_batch is not None: inputs["num_items_in_batch"]=num_items_in_batch
        inputs["skip_logits"]=True  # force liger fused-linear-CE (no full logits)
        out=model(**inputs)
        loss=out.loss if hasattr(out,"loss") else out["loss"]
        return (loss, out) if return_outputs else loss

kw=dict(model=model, args=args, train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=FLCETrainer(**kw)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if local_rank==0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),
        "steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,
        "tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None,"world_size":int(os.environ.get("WORLD_SIZE","1"))}), flush=True)
    if os.environ.get("SAVE_MODEL","0")=="1":
        trainer.save_model(OUT_DIR); tok.save_pretrained(OUT_DIR); print("saved to", OUT_DIR, flush=True)
'''
with open("/tmp/train_gemma4_flce.py","w") as f: f.write(FLCE_PY4)
print("rewrote v4", len(FLCE_PY4), "bytes")

In [ ]:
# Cell 28 - deepest frames of v4 error + did config.vocab_size actually stick?
for l in o.splitlines()[-40:]:
    print(l)

In [ ]:
# Cell 29 - show 25 lines before the last vocab_size AttributeError to find the frame
lines=o.splitlines()
pos=[i for i,l in enumerate(lines) if "has no attribute 'vocab_size'" in l]
if pos:
    p=pos[-1]
    for l in lines[max(0,p-25):p+1]:
        print(l)
else:
    print('not found; tail:'); print("\n".join(lines[-30:]))

In [ ]:
# Cell 30 - patch on-disk base config.json: expose top-level vocab_size/hidden_size (for PEFT save check)
import json, os
cfg_path=os.path.join(MODEL_DIR,"config.json")
with open(cfg_path) as f: cfg=json.load(f)
tc=cfg.get("text_config",{})
before={k:cfg.get(k) for k in ["vocab_size","hidden_size"]}
for k in ["vocab_size","hidden_size"]:
    if k not in cfg and k in tc: cfg[k]=tc[k]
with open(cfg_path,"w") as f: json.dump(cfg,f,indent=2)
print("patched", cfg_path)
print("before:", before, "| after: vocab_size=",cfg.get("vocab_size"),"hidden_size=",cfg.get("hidden_size"))

In [ ]:
# Cell 31 - FLCE big-batch sweep (seq4096, no-ckpt) - the payoff test
flce_sweep=[
    dict(MAX_LEN=4096, BATCH=4),
    dict(MAX_LEN=4096, BATCH=8),
    dict(MAX_LEN=4096, BATCH=12),
    dict(MAX_LEN=4096, BATCH=16),
]
fres=[]
for cfg in flce_sweep:
    base=dict(LORA_R="64",LORA_ALPHA="128",USE_RSLORA="1",USE_PACKING="1",MAX_STEPS="12",GRAD_ACC="1",
              GRADIENT_CHECKPOINTING="0",OUT_DIR="/tmp/out_flce_sweep",SAVE_MODEL="0")
    base.update({k:str(v) for k,v in cfg.items()})
    rc,o2 = run_flce(nproc=2, extra_env=base, timeout=1200)
    s=None
    for l in o2.splitlines():
        if l.startswith("SUMMARY"):
            import json as _j; s=_j.loads(l[7:].strip())
    oom="OutOfMemory" in o2
    row=dict(seq=cfg["MAX_LEN"],batch=cfg["BATCH"],tok_step=cfg["MAX_LEN"]*cfg["BATCH"],rc=rc,oom=oom,
             tok_s=(s or {}).get("tokens_per_s"),vram=(s or {}).get("max_vram_gb"),loss=(s or {}).get("train_loss"))
    fres.append(row); print("=>",row)
print("\n==== FLCE BIG-BATCH (2-GPU, seq4096) ====")
print("%-6s %-6s %-9s %-4s %-6s %-10s %-7s"%("seq","batch","tok/step","rc","oom","tokens/s","vram"))
for r in fres:
    print("%-6s %-6s %-9s %-4s %-6s %-10s %-7s"%(r["seq"],r["batch"],r["tok_step"],r["rc"],r["oom"],r["tok_s"],r["vram"]))
okf=[r for r in fres if r["rc"]==0 and r["tok_s"]]
if okf: print("\nBEST FLCE:", max(okf,key=lambda r:r["tok_s"]))

In [ ]:
# Cell 32 - FLCE decisive comparison: no-ckpt batch2, and ckpt+big-batch (vs 7100 baseline)
import json as _j
combo=[
    dict(MAX_LEN=4096, BATCH=2,  GRADIENT_CHECKPOINTING=0),  # FLCE lets batch2 fit no-ckpt?
    dict(MAX_LEN=4096, BATCH=8,  GRADIENT_CHECKPOINTING=1),  # FLCE+ckpt big batch
    dict(MAX_LEN=4096, BATCH=16, GRADIENT_CHECKPOINTING=1),  # FLCE+ckpt bigger
]
cres=[]
for cfg in combo:
    base=dict(LORA_R="64",LORA_ALPHA="128",USE_RSLORA="1",USE_PACKING="1",MAX_STEPS="16",GRAD_ACC="1",
              OUT_DIR="/tmp/out_flce_combo",SAVE_MODEL="0")
    base.update({k:str(v) for k,v in cfg.items()})
    rc,oc = run_flce(nproc=2, extra_env=base, timeout=1500)
    s=None
    for l in oc.splitlines():
        if l.startswith("SUMMARY"): s=_j.loads(l[7:].strip())
    cres.append(dict(seq=cfg["MAX_LEN"],batch=cfg["BATCH"],ckpt=cfg["GRADIENT_CHECKPOINTING"],rc=rc,
        oom=("OutOfMemory" in oc),tok_s=(s or {}).get("tokens_per_s"),vram=(s or {}).get("max_vram_gb")))
    print("=>",cres[-1])
print("\n==== FLCE combo vs baseline non-FLCE(7100 tok/s grad_acc8, 5134 batch1) ====")
for r in cres: print(r)

In [ ]:
# Cell 33 - check for orphaned torchrun/training procs + GPU memory after the interrupt
import subprocess
print("===== nvidia-smi (mem + compute procs) =====")
print(subprocess.run(["nvidia-smi","--query-gpu=index,memory.used,memory.total,utilization.gpu","--format=csv"],capture_output=True,text=True).stdout)
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory,process_name","--format=csv"],capture_output=True,text=True).stdout or "(no compute apps)")
print("===== python / torchrun processes =====")
ps=subprocess.run(["bash","-lc","ps -eo pid,ppid,etime,rss,comm,args | grep -E 'torch.distributed.run|train_gemma4|torchrun' | grep -v grep"],capture_output=True,text=True)
print(ps.stdout or "(none found - clean, no orphans)")

In [ ]:
# Cell 34 - parametrized A/B train script (ATTN_IMPL / USE_PACKING / TORCH_COMPILE) + verbose runner
AB_PY = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_ab")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); BATCH=int(os.environ.get("BATCH","1"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","8")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
USE_RSLORA=os.environ.get("USE_RSLORA","1")=="1"; USE_PACKING=os.environ.get("USE_PACKING","1")=="1"
USE_LIGER=os.environ.get("USE_LIGER","1")=="1"; GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
ATTN_IMPL=os.environ.get("ATTN_IMPL","sdpa"); TORCH_COMPILE=os.environ.get("TORCH_COMPILE","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0")); R0=(local_rank==0)
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True

if USE_LIGER:
    try:
        from liger_kernel.transformers.monkey_patch import _apply_liger_kernel
        _apply_liger_kernel("gemma4_text")
        if R0: print("LIGER: applied to gemma4_text", flush=True)
    except Exception as e:
        if R0: print("LIGER off:", repr(e), flush=True)

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
t_load=time.time()
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation=ATTN_IMPL)
model.config.use_cache=False
if R0: print(f"MODEL loaded attn_impl={ATTN_IMPL} in {time.time()-t_load:.0f}s", flush=True)

PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
lora=LoraConfig(task_type=TaskType.CAUSAL_LM,r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.0,bias="none",target_modules=TARGETS,use_rslora=USE_RSLORA)
model=get_peft_model(model,lora)
if R0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

ds=load_dataset("json", data_files={"train":sorted(glob.glob(TRAIN_GLOB))})
def safe_cfg(**kw):
    allowed=set(inspect.signature(SFTConfig.__init__).parameters)
    drop=[k for k in kw if k not in allowed]
    if drop and R0: print("drop unsupported SFTConfig:",drop,flush=True)
    return SFTConfig(**{k:v for k,v in kw.items() if k in allowed})
cfg=dict(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=1,
    save_steps=10**9, save_total_limit=1, gradient_checkpointing=GRAD_CKPT, ddp_find_unused_parameters=False,
    dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", max_length=MAX_LEN,
    dataset_text_field="text", packing=USE_PACKING, eval_strategy="no", include_num_input_tokens_seen=True)
if TORCH_COMPILE:
    cfg.update(torch_compile=True, torch_compile_backend="inductor", torch_compile_mode="max-autotune-no-cudagraphs")
args=safe_cfg(**cfg)
kw=dict(model=model,args=args,train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
if R0: print(f"CONFIG packing={USE_PACKING} attn={ATTN_IMPL} compile={TORCH_COMPILE} liger={USE_LIGER} ckpt={GRAD_CKPT} | num_train_rows={len(trainer.train_dataset)}", flush=True)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if R0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"packing":USE_PACKING,"attn":ATTN_IMPL,"compile":TORCH_COMPILE,
        "train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),"steps":trainer.state.global_step,
        "max_vram_gb":round(mem,1),"tokens_seen":ntok,"tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None}), flush=True)
'''
with open("/tmp/train_gemma4_ab.py","w") as f: f.write(AB_PY)

def run_ab(extra_env, nproc=2, timeout=2400, show_steps=True):
    import sys, subprocess
    env=os.environ.copy()
    env.update({"MODEL_DIR":MODEL_DIR,"TRAIN_GLOB":TRAIN_GLOB,"EVAL_GLOB":EVAL_GLOB,
        "HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","TOKENIZERS_PARALLELISM":"false",
        "PYTORCH_ALLOC_CONF":"expandable_segments:True","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "NCCL_DEBUG":"WARN","OMP_NUM_THREADS":"12"})
    env.update({k:str(v) for k,v in extra_env.items()})
    cmd=[sys.executable,"-m","torch.distributed.run","--standalone",f"--nproc_per_node={nproc}","/tmp/train_gemma4_ab.py"]
    print("RUN:", {k:extra_env[k] for k in extra_env}); t=time.time()
    p=subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=timeout)
    o=(p.stdout or "")+"\n"+(p.stderr or "")
    for l in o.splitlines():
        if ("'loss':" in l and show_steps) or l.startswith(("SUMMARY","CONFIG","MODEL","LIGER")) or "trainable params" in l or "OutOfMemory" in l or "Error:" in l:
            print(l)
    print(f"[wall {time.time()-t:.0f}s rc={p.returncode}]")
    return p.returncode, o
print("wrote /tmp/train_gemma4_ab.py + run_ab() ready")

In [ ]:
# Cell 34 - parametrized A/B train script (ATTN_IMPL / USE_PACKING / TORCH_COMPILE) + verbose runner
AB_PY = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_ab")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); BATCH=int(os.environ.get("BATCH","1"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","8")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
USE_RSLORA=os.environ.get("USE_RSLORA","1")=="1"; USE_PACKING=os.environ.get("USE_PACKING","1")=="1"
USE_LIGER=os.environ.get("USE_LIGER","1")=="1"; GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
ATTN_IMPL=os.environ.get("ATTN_IMPL","sdpa"); TORCH_COMPILE=os.environ.get("TORCH_COMPILE","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0")); R0=(local_rank==0)
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True

if USE_LIGER:
    try:
        from liger_kernel.transformers.monkey_patch import _apply_liger_kernel
        _apply_liger_kernel("gemma4_text")
        if R0: print("LIGER: applied to gemma4_text", flush=True)
    except Exception as e:
        if R0: print("LIGER off:", repr(e), flush=True)

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
t_load=time.time()
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation=ATTN_IMPL)
model.config.use_cache=False
if R0: print(f"MODEL loaded attn_impl={ATTN_IMPL} in {time.time()-t_load:.0f}s", flush=True)

PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
lora=LoraConfig(task_type=TaskType.CAUSAL_LM,r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.0,bias="none",target_modules=TARGETS,use_rslora=USE_RSLORA)
model=get_peft_model(model,lora)
if R0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

ds=load_dataset("json", data_files={"train":sorted(glob.glob(TRAIN_GLOB))})
def safe_cfg(**kw):
    allowed=set(inspect.signature(SFTConfig.__init__).parameters)
    drop=[k for k in kw if k not in allowed]
    if drop and R0: print("drop unsupported SFTConfig:",drop,flush=True)
    return SFTConfig(**{k:v for k,v in kw.items() if k in allowed})
cfg=dict(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=1,
    save_steps=10**9, save_total_limit=1, gradient_checkpointing=GRAD_CKPT, ddp_find_unused_parameters=False,
    dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none", max_length=MAX_LEN,
    dataset_text_field="text", packing=USE_PACKING, eval_strategy="no", include_num_input_tokens_seen=True)
if TORCH_COMPILE:
    cfg.update(torch_compile=True, torch_compile_backend="inductor", torch_compile_mode="max-autotune-no-cudagraphs")
args=safe_cfg(**cfg)
kw=dict(model=model,args=args,train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
if R0: print(f"CONFIG packing={USE_PACKING} attn={ATTN_IMPL} compile={TORCH_COMPILE} liger={USE_LIGER} ckpt={GRAD_CKPT} | num_train_rows={len(trainer.train_dataset)}", flush=True)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if R0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"packing":USE_PACKING,"attn":ATTN_IMPL,"compile":TORCH_COMPILE,
        "train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),"steps":trainer.state.global_step,
        "max_vram_gb":round(mem,1),"tokens_seen":ntok,"tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None}), flush=True)
'''
with open("/tmp/train_gemma4_ab.py","w") as f: f.write(AB_PY)

def run_ab(extra_env, nproc=2, timeout=2400, show_steps=True):
    import sys, subprocess
    env=os.environ.copy()
    env.update({"MODEL_DIR":MODEL_DIR,"TRAIN_GLOB":TRAIN_GLOB,"EVAL_GLOB":EVAL_GLOB,
        "HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","TOKENIZERS_PARALLELISM":"false",
        "PYTORCH_ALLOC_CONF":"expandable_segments:True","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "NCCL_DEBUG":"WARN","OMP_NUM_THREADS":"12"})
    env.update({k:str(v) for k,v in extra_env.items()})
    cmd=[sys.executable,"-m","torch.distributed.run","--standalone",f"--nproc_per_node={nproc}","/tmp/train_gemma4_ab.py"]
    print("RUN:", {k:extra_env[k] for k in extra_env}); t=time.time()
    p=subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=timeout)
    o=(p.stdout or "")+"\n"+(p.stderr or "")
    for l in o.splitlines():
        if ("'loss':" in l and show_steps) or l.startswith(("SUMMARY","CONFIG","MODEL","LIGER")) or "trainable params" in l or "OutOfMemory" in l or "Error:" in l:
            print(l)
    print(f"[wall {time.time()-t:.0f}s rc={p.returncode}]")
    return p.returncode, o
print("wrote /tmp/train_gemma4_ab.py + run_ab() ready")

In [ ]:
# Cell 35 - probe FlashAttention feasibility offline (flash_attn pkg + HF kernels)
import importlib.util as u, subprocess, sys
print("flash_attn installed:", u.find_spec("flash_attn") is not None)
print("kernels (HF) installed:", u.find_spec("kernels") is not None)
# try a tiny offline load with kernels FA2 (expected to fail w/o internet) - quick, no train
test = ("import os,torch;os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1';"
        "from transformers import AutoModelForImageTextToText as M;"
        "m=M.from_pretrained(os.environ['MODEL_DIR'],dtype=torch.bfloat16,attn_implementation='kernels-community/flash-attn2');"
        "print('FA2-kernels OK')")
import os
env=os.environ.copy(); env.update({"MODEL_DIR":MODEL_DIR,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"-c",test], env=env, capture_output=True, text=True, timeout=300)
print("stdout:", p.stdout.strip()[-400:])
print("stderr:", (p.stderr or "").strip()[-600:])
print("=> FA2 via kernels feasible offline:", "FA2-kernels OK" in p.stdout)

In [ ]:
# Cell 36 - A/B: correctness+speed across attn/packing (seq4096/batch1/grad_acc8, 20 steps, verbose)
common=dict(MAX_LEN="4096",BATCH="1",GRAD_ACC="8",MAX_STEPS="20",LORA_R="64",LORA_ALPHA="128",
            USE_RSLORA="1",USE_LIGER="1",GRADIENT_CHECKPOINTING="0")
trials={
  "C_pack_sdpa":      dict(USE_PACKING="1", ATTN_IMPL="sdpa"),
  "A_pack_flash2":    dict(USE_PACKING="1", ATTN_IMPL="flash_attention_2"),
  "B_nopack_sdpa":    dict(USE_PACKING="0", ATTN_IMPL="sdpa"),
}
import json as _j
summ={}
for name,extra in trials.items():
    print("\n############ TRIAL", name, "############")
    env=dict(common); env.update(extra); env["OUT_DIR"]="/tmp/out_ab_"+name
    rc,o = run_ab(env, nproc=2, timeout=1800, show_steps=True)
    for l in o.splitlines():
        if l.startswith("SUMMARY"): summ[name]=_j.loads(l[7:].strip())
print("\n==================== A/B SUMMARY ====================")
print("%-16s %-8s %-7s %-10s %-9s %-9s"%("trial","packing","attn","tokens/s","vram","final_loss"))
for name in trials:
    s=summ.get(name,{})
    print("%-16s %-8s %-7s %-10s %-9s %-9s"%(name,s.get("packing"),(s.get("attn") or "")[:6],s.get("tokens_per_s"),s.get("max_vram_gb"),s.get("train_loss")))

In [ ]:
# Cell 37 - test packing=True + flex_attention (clean block-diagonal mask, no head_dim limit)
common=dict(MAX_LEN="4096",BATCH="1",GRAD_ACC="8",MAX_STEPS="20",LORA_R="64",LORA_ALPHA="128",
            USE_RSLORA="1",USE_LIGER="1",GRADIENT_CHECKPOINTING="0")
import json as _j
env=dict(common); env.update(USE_PACKING="1", ATTN_IMPL="flex_attention", OUT_DIR="/tmp/out_flex")
rc,o = run_ab(env, nproc=2, timeout=1800, show_steps=True)
flex=None
for l in o.splitlines():
    if l.startswith("SUMMARY"): flex=_j.loads(l[7:].strip())
print("\n==== flex_attention result ====")
print(flex if flex else "(no summary - check error above)")
print("compare: C_pack_sdpa was 6832 tok/s @ loss 1.30")

In [ ]:
# Cell 38 - decisive contamination unit test: does sdpa isolate packed docs given only reset position_ids?
CT = r'''
import os, torch
from transformers import AutoModelForImageTextToText
MODEL_DIR=os.environ["MODEL_DIR"]
m=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa").to("cuda:0").eval()
dev="cuda:0"
a=torch.tensor([[5,6,7,8,9,10]],device=dev)          # doc1
b=torch.tensor([[100,101,102,103]],device=dev)        # doc2
with torch.no_grad():
    # doc2 standalone
    out_b=m(input_ids=b).logits[0]
    # packed: [doc1, doc2], position_ids reset at doc2, attention_mask all ones (packing w/o block mask)
    ids=torch.cat([a,b],dim=1)
    pos=torch.tensor([[0,1,2,3,4,5, 0,1,2,3]],device=dev)
    am=torch.ones_like(ids)
    out_packed=m(input_ids=ids, attention_mask=am, position_ids=pos).logits[0]
    b_in_packed=out_packed[a.shape[1]:, :]   # logits over doc2 region
    diff=(b_in_packed.float()-out_b.float()).abs().max().item()
    rel=diff/ (out_b.float().abs().max().item()+1e-6)
print("MAXABS_DIFF doc2 standalone vs packed:", round(diff,4), "| relative:", round(rel,4))
print("VERDICT:", "CONTAMINATED (doc2 sees doc1)" if diff>1e-2 else "ISOLATED/clean")
'''
import os, subprocess, sys
with open("/tmp/contam_test.py","w") as f: f.write(CT)
env=os.environ.copy(); env.update({"MODEL_DIR":MODEL_DIR,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/contam_test.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-1500:]); print("ERR:", (p.stderr or '')[-500:])

In [ ]:
# Cell 39 - inspect the ACTUAL batch TRL produces with packing=True (does it build a block-diagonal mask?)
INS = r'''
import os, glob, torch, inspect
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTTrainer, SFTConfig
MODEL_DIR=os.environ["MODEL_DIR"]; TRAIN_GLOB=os.environ["TRAIN_GLOB"]
tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
ds=load_dataset("json", data_files={"train":sorted(glob.glob(TRAIN_GLOB))})
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters); return SFTConfig(**{k:v for k,v in kw.items() if k in al})
args=sc(output_dir="/tmp/insp", per_device_train_batch_size=1, max_length=4096, dataset_text_field="text", packing=True, report_to="none", dataloader_num_workers=0)
# minimal model stub not needed for dataloader; pass a tiny config-less trainer via model=None? SFTTrainer needs model. Load real.
from transformers import AutoModelForImageTextToText
m=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16)
kw=dict(model=m, args=args, train_dataset=ds["train"])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
tr=SFTTrainer(**kw)
dl=tr.get_train_dataloader()
b=next(iter(dl))
print("BATCH KEYS:", list(b.keys()))
for k,v in b.items():
    if torch.is_tensor(v): print(f"  {k}: shape={tuple(v.shape)} dtype={v.dtype}")
if "attention_mask" in b:
    am=b["attention_mask"]; print("attention_mask ndim:", am.ndim, "unique(sample):", torch.unique(am)[:8].tolist())
if "position_ids" in b:
    pid=b["position_ids"][0]; print("position_ids head:", pid[:20].tolist(), "... resets(count of zeros):", int((pid==0).sum()))
'''
import os, subprocess, sys
with open("/tmp/inspect_batch.py","w") as f: f.write(INS)
env=os.environ.copy(); env.update({"MODEL_DIR":MODEL_DIR,"TRAIN_GLOB":TRAIN_GLOB,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/inspect_batch.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-2000:]); print("ERR:", (p.stderr or '')[-700:])

In [ ]:
# Cell 40 - INSPECT loss mask + chat-template assistant-mask support (decides masking strategy)
INSP = r'''
import os, glob, json, time
from transformers import AutoTokenizer
def ts(m): print(time.strftime("[%H:%M:%S]"), m, flush=True)
MODEL_DIR=os.environ["MODEL_DIR"]; RAW_GLOB=os.environ["RAW_GLOB"]
tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token

ts("1) chat_template assistant-mask support?")
has_gen = "{% generation %}" in (tok.chat_template or "") or "generation" in (tok.chat_template or "")
print("   chat_template has {% generation %} block:", has_gen)
msg=[{"role":"user","content":"hỏi A"},{"role":"assistant","content":"đáp B"}]
try:
    enc=tok.apply_chat_template(msg, return_dict=True, return_assistant_tokens_mask=True, tokenize=True)
    am=enc.get("assistant_masks")
    print("   return_assistant_tokens_mask -> assistant_masks present:", am is not None,
          "| #assistant_tokens:", (sum(am) if am else None), "/", (len(am) if am else None))
except Exception as e:
    print("   assistant mask NOT supported:", repr(e))

ts("2) quantify wasted learning if full-text on messages rows")
sys_user=0; asst=0; n=0
for f in glob.glob(RAW_GLOB):
    for ln in open(f):
        if not ln.strip():continue
        ex=json.loads(ln); ms=ex.get("messages")
        if not ms: continue
        n+=1
        for m in ms:
            t=len(tok(m.get("content","")).input_ids)
            if m["role"]=="assistant": asst+=t
            else: sys_user+=t
        if n>=300: break
    if n>=300: break
tot=sys_user+asst
print(f"   messages rows sampled: {n} | assistant tokens: {asst} ({100*asst/max(tot,1):.1f}%) | system+user: {sys_user} ({100*sys_user/max(tot,1):.1f}%)")
print("   => with text-field full-loss, ~%.0f%% of messages-row tokens are NON-answer being learned"%(100*sys_user/max(tot,1)))
'''
import os, subprocess, sys
RAW_GLOB="/tmp/ds_unpacked/train/*.jsonl"
with open("/tmp/inspect_mask.py","w") as f: f.write(INSP)
env=os.environ.copy(); env.update({"MODEL_DIR":MODEL_DIR,"RAW_GLOB":RAW_GLOB,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1"})
p=subprocess.run([sys.executable,"/tmp/inspect_mask.py"], env=env, capture_output=True, text=True, timeout=400)
print(p.stdout); print("ERR:",(p.stderr or '')[-600:])

In [ ]:
# Cell 41 - build unified prompt/completion dataset from RAW mixed data (for completion_only_loss)
import json, glob, os, time
from pathlib import Path
def ts(m): print(time.strftime("[%H:%M:%S]"), m, flush=True)

def merge_system(ms):
    out=[]; carry=""
    for m in ms:
        if m.get("role")=="system": carry+=(m.get("content","")+"\n\n"); continue
        if m.get("role")=="user" and carry: out.append({"role":"user","content":carry+m.get("content","")}); carry=""
        else: out.append(m)
    if carry and out: out[0]["content"]=carry+out[0].get("content","")
    return out

def to_pc(ex):
    t=ex.get("text")
    if isinstance(t,str) and t.strip():
        return {"prompt":"", "completion": t}          # raw corpus -> learn fully
    ms=ex.get("messages")
    if ms:
        if isinstance(ms,str): ms=json.loads(ms)
        ms=merge_system(ms)
        # last assistant = completion; everything before = prompt
        last_a=max([i for i,m in enumerate(ms) if m.get("role")=="assistant"], default=None)
        if last_a is None: return None
        comp=ms[last_a].get("content","")
        pre=ms[:last_a]
        prompt=tok.apply_chat_template(pre, tokenize=False, add_generation_prompt=True)
        if not comp.strip(): return None
        return {"prompt":prompt, "completion":comp}
    return None

RAW="/tmp/ds_unpacked"; PC=Path("/tmp/ds_pc"); (PC/"train").mkdir(parents=True,exist_ok=True); (PC/"eval").mkdir(parents=True,exist_ok=True)
def convert(globpat,dst):
    n=0;skip=0;pc_msg=0;pc_txt=0
    with open(dst,"w",encoding="utf-8") as w:
        for f in glob.glob(globpat):
            for ln in open(f):
                if not ln.strip():continue
                ex=json.loads(ln); pc=to_pc(ex)
                if pc is None: skip+=1; continue
                if pc["prompt"]=="": pc_txt+=1
                else: pc_msg+=1
                w.write(json.dumps(pc,ensure_ascii=False)+"\n"); n+=1
    return n,skip,pc_msg,pc_txt
ts("converting train..."); ntr=convert(f"{RAW}/train/*.jsonl", PC/"train"/"part.jsonl")
ts("converting eval...");  nev=convert(f"{RAW}/eval/*.jsonl",  PC/"eval"/"part.jsonl")
print("train (rows,skip,from_msg,from_text):",ntr)
print("eval  (rows,skip,from_msg,from_text):",nev)
PC_TRAIN_GLOB=str(PC/"train"/"*.jsonl"); PC_EVAL_GLOB=str(PC/"eval"/"*.jsonl")
print("PC_TRAIN_GLOB:",PC_TRAIN_GLOB)
# peek one converted messages-row
for ln in open(PC/"train"/"part.jsonl"):
    r=json.loads(ln)
    if r["prompt"]:
        print("\nSAMPLE prompt[:200]:",r["prompt"][:200].replace(chr(10),"\\n"))
        print("SAMPLE completion[:200]:",r["completion"][:200].replace(chr(10),"\\n")); break

In [ ]:
# Cell 42 - VERIFY completion_only_loss masking on real batches (prompt=-100, only completion learned)
VER = r'''
import os, glob, time, inspect, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText
from trl import SFTTrainer, SFTConfig
def ts(m): print(time.strftime("[%H:%M:%S]"), m, flush=True)
MODEL_DIR=os.environ["MODEL_DIR"]; PC_TRAIN=os.environ["PC_TRAIN"]
tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
ds=load_dataset("json", data_files={"train":sorted(glob.glob(PC_TRAIN))})
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters)
    drop=[k for k in kw if k not in al]
    if drop: print("   (SFTConfig unsupported, dropped):", drop)
    return SFTConfig(**{k:v for k,v in kw.items() if k in al})
args=sc(output_dir="/tmp/ver", per_device_train_batch_size=2, max_length=4096, packing=False,
        completion_only_loss=True, group_by_length=True, report_to="none", dataloader_num_workers=0)
ts("loading model (for trainer)")
m=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16)
kw=dict(model=m,args=args,train_dataset=ds["train"]) 
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
tr=SFTTrainer(**kw)
ts("pulling 3 batches and checking masks")
dl=tr.get_train_dataloader(); it=iter(dl)
for bi in range(3):
    b=next(it)
    lab=b["labels"]; ids=b["input_ids"]
    learned=(lab!=-100); masked=(lab==-100)
    print(f"  batch{bi}: shape={tuple(ids.shape)} | learned tokens={int(learned.sum())} ({100*learned.float().mean():.1f}%) | masked(-100)={int(masked.sum())}")
    # show that masked region == prompt: decode first learned token boundary of row0
    row=lab[0]; first_learn=(row!=-100).nonzero()
    if len(first_learn):
        j=int(first_learn[0])
        print(f"    row0 first-learned idx={j}; prompt(decoded,last40 before)=...{tok.decode(ids[0][max(0,j-40):j])!r}")
        print(f"    row0 completion start(decoded,40)= {tok.decode(ids[0][j:j+40])!r}")
'''
import os, subprocess, sys
with open("/tmp/verify_mask.py","w") as f: f.write(VER)
env=os.environ.copy(); env.update({"MODEL_DIR":MODEL_DIR,"PC_TRAIN":PC_TRAIN_GLOB,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/verify_mask.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-2500:]); print("ERR:",(p.stderr or '')[-600:])

In [ ]:
# Cell 43 - CLEAN train script (prompt/completion + completion_only_loss, packing=False, sdpa) + timestamped runner
CLEAN_PY = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_DIR=os.environ["MODEL_DIR"]; PC_TRAIN=os.environ["PC_TRAIN"]; PC_EVAL=os.environ.get("PC_EVAL","")
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_clean")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); BATCH=int(os.environ.get("BATCH","2"))
GRAD_ACC=int(os.environ.get("GRAD_ACC","8")); EPOCHS=float(os.environ.get("EPOCHS","2"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
USE_RSLORA=os.environ.get("USE_RSLORA","1")=="1"; USE_LIGER=os.environ.get("USE_LIGER","1")=="1"
GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
local_rank=int(os.environ.get("LOCAL_RANK","0")); R0=(local_rank==0)
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
def ts(m):
    if R0: print(time.strftime("[%H:%M:%S]"), m, flush=True)

if USE_LIGER:
    try:
        from liger_kernel.transformers.monkey_patch import _apply_liger_kernel
        _apply_liger_kernel("gemma4_text"); ts("LIGER applied to gemma4_text")
    except Exception as e: ts("LIGER off: "+repr(e))

tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
model.config.use_cache=False
PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
model=get_peft_model(model, LoraConfig(task_type=TaskType.CAUSAL_LM,r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.0,bias="none",target_modules=TARGETS,use_rslora=USE_RSLORA))
if R0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

df={"train":sorted(glob.glob(PC_TRAIN))}
if PC_EVAL and glob.glob(PC_EVAL): df["validation"]=sorted(glob.glob(PC_EVAL))
ds=load_dataset("json", data_files=df)
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters); return SFTConfig(**{k:v for k,v in kw.items() if k in al})
args=sc(output_dir=OUT_DIR, bf16=True, tf32=True, per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR, warmup_ratio=0.03, num_train_epochs=EPOCHS, max_steps=MAX_STEPS, lr_scheduler_type="cosine",
    optim="adamw_torch_fused", logging_steps=1, save_steps=10**9, save_total_limit=1, gradient_checkpointing=GRAD_CKPT,
    ddp_find_unused_parameters=False, dataloader_num_workers=4, dataloader_pin_memory=True, report_to="none",
    max_length=MAX_LEN, packing=False, completion_only_loss=True, eval_strategy="no", include_num_input_tokens_seen=True)

class TS(TrainerCallback):
    def on_log(self, a,s,c, logs=None, **k):
        if logs and R0 and "loss" in logs:
            tps=logs.get("train_tokens_per_second")
            print(time.strftime("[%H:%M:%S]"), f"step={s.global_step} loss={logs.get('loss')} acc={logs.get('mean_token_accuracy')} tok/s={tps} seen={logs.get('num_input_tokens_seen')}", flush=True)

kw=dict(model=model,args=args,train_dataset=ds["train"],callbacks=[TS()])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
if R0: print(f"CONFIG packing=False completion_only=True attn=sdpa liger={USE_LIGER} ckpt={GRAD_CKPT} batch={BATCH} gacc={GRAD_ACC} rows={len(trainer.train_dataset)}", flush=True)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if R0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"batch":BATCH,"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),
        "steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,
        "tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None}), flush=True)
    if os.environ.get("SAVE_MODEL","0")=="1":
        trainer.save_model(OUT_DIR); tok.save_pretrained(OUT_DIR); print("saved to",OUT_DIR, flush=True)
'''
with open("/tmp/train_gemma4_clean.py","w") as f: f.write(CLEAN_PY)

def run_clean(extra_env, nproc=2, timeout=3600):
    import sys, subprocess
    env=os.environ.copy()
    env.update({"MODEL_DIR":MODEL_DIR,"PC_TRAIN":PC_TRAIN_GLOB,"PC_EVAL":PC_EVAL_GLOB,
        "HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","TOKENIZERS_PARALLELISM":"false",
        "PYTORCH_ALLOC_CONF":"expandable_segments:True","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "NCCL_DEBUG":"WARN","OMP_NUM_THREADS":"12"})
    env.update({k:str(v) for k,v in extra_env.items()})
    cmd=[sys.executable,"-m","torch.distributed.run","--standalone",f"--nproc_per_node={nproc}","/tmp/train_gemma4_clean.py"]
    print(time.strftime("[%H:%M:%S]"),"START", {k:extra_env.get(k) for k in ["BATCH","GRAD_ACC","MAX_STEPS","GRADIENT_CHECKPOINTING","EPOCHS","SAVE_MODEL"]}); t=time.time()
    p=subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=timeout)
    o=(p.stdout or "")+"\n"+(p.stderr or "")
    for l in o.splitlines():
        if l.startswith(("[","SUMMARY","CONFIG","saved")) or "trainable params" in l or "OutOfMemory" in l or "Error:" in l: print(l)
    print(time.strftime("[%H:%M:%S]"),f"END wall={time.time()-t:.0f}s rc={p.returncode}")
    return p.returncode, o
print("clean script + run_clean() ready")

In [ ]:
# Cell 44 - CLEAN sweep: batch 1/2/3/4 (packing=False, completion_only_loss, sdpa, no ckpt), 20 steps
import json as _j
clean_res={}
for B in [1,2,3,4]:
    rc,o = run_clean(dict(BATCH=B,GRAD_ACC=8,MAX_STEPS=20,GRADIENT_CHECKPOINTING=0,EPOCHS=1,
                          OUT_DIR=f"/tmp/out_clean_b{B}",SAVE_MODEL=0), nproc=2, timeout=1200)
    s=None
    for l in o.splitlines():
        if l.startswith("SUMMARY"): s=_j.loads(l[7:].strip())
    clean_res[B]=dict(rc=rc,oom=("OutOfMemory" in o),tok_s=(s or {}).get("tokens_per_s"),
                      vram=(s or {}).get("max_vram_gb"),loss=(s or {}).get("train_loss"))
    print("  =>",B,clean_res[B])
print("\n==== CLEAN SWEEP (packing=False, completion_only_loss, sdpa) ====")
print("%-6s %-5s %-6s %-10s %-7s %-8s"%("batch","rc","oom","tokens/s","vram","loss"))
for B,r in clean_res.items():
    print("%-6s %-5s %-6s %-10s %-7s %-8s"%(B,r["rc"],r["oom"],r["tok_s"],r["vram"],r["loss"]))
ok=[(B,r) for B,r in clean_res.items() if r["rc"]==0 and r["tok_s"]]
if ok:
    bestB=max(ok,key=lambda x:x[1]["tok_s"]); print("\nBEST clean batch:",bestB[0],bestB[1])

In [ ]:
# Cell 45 - clean + grad-ckpt to allow bigger batch (does it beat batch1 no-ckpt 2930 tok/s?)
import json as _j
for B in [4,8]:
    rc,o = run_clean(dict(BATCH=B,GRAD_ACC=4,MAX_STEPS=16,GRADIENT_CHECKPOINTING=1,EPOCHS=1,
                          OUT_DIR=f"/tmp/out_clean_ckpt_b{B}",SAVE_MODEL=0), nproc=2, timeout=1500)
    s=None
    for l in o.splitlines():
        if l.startswith("SUMMARY"): s=_j.loads(l[7:].strip())
    print("  => ckpt batch",B, (s or {"oom":"OutOfMemory" in o}))
print("\nReference: clean batch1 no-ckpt = 2930 tok/s")

In [ ]:
# Cell 46 - SYNTHESIS: FLCE + completion_only + packing=False (clean AND lets batch grow). Test batch 4/8/12
FLCE_CLEAN = r'''
import os, json, glob, time, inspect
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from liger_kernel.transformers.model.gemma4 import causal_forward as g4cf
MODEL_DIR=os.environ["MODEL_DIR"]; PC_TRAIN=os.environ["PC_TRAIN"]; PC_EVAL=os.environ.get("PC_EVAL","")
OUT_DIR=os.environ.get("OUT_DIR","/tmp/out_fc"); MAX_LEN=int(os.environ.get("MAX_LEN","4096"))
BATCH=int(os.environ.get("BATCH","4")); GRAD_ACC=int(os.environ.get("GRAD_ACC","4")); EPOCHS=float(os.environ.get("EPOCHS","1"))
MAX_STEPS=int(os.environ.get("MAX_STEPS","-1")); LR=float(os.environ.get("LR","7e-5"))
LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
GRAD_CKPT=os.environ.get("GRADIENT_CHECKPOINTING","0")=="1"
R0=(int(os.environ.get("LOCAL_RANK","0"))==0)
if torch.cuda.is_available(): torch.cuda.set_device(int(os.environ.get("LOCAL_RANK","0")))
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
tok=AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, attn_implementation="sdpa")
model.config.use_cache=False
tc=getattr(model.config,"text_config",None)
for a in ["hidden_size","vocab_size","pad_token_id","final_logit_softcapping"]:
    if tc is not None and hasattr(tc,a): setattr(model.config,a,getattr(tc,a))
model.vocab_size=model.config.vocab_size
type(model).forward=g4cf
if R0: print("FLCE+clean patched",flush=True)
PROJ=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
TARGETS=r".*language_model\..*\.("+"|".join(PROJ)+r")$"
model=get_peft_model(model, LoraConfig(task_type=TaskType.CAUSAL_LM,r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.0,bias="none",target_modules=TARGETS,use_rslora=True))
if R0: model.print_trainable_parameters()
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
df={"train":sorted(glob.glob(PC_TRAIN))}
if PC_EVAL and glob.glob(PC_EVAL): df["validation"]=sorted(glob.glob(PC_EVAL))
ds=load_dataset("json", data_files=df)
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters); return SFTConfig(**{k:v for k,v in kw.items() if k in al})
args=sc(output_dir=OUT_DIR,bf16=True,tf32=True,per_device_train_batch_size=BATCH,gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,warmup_ratio=0.03,num_train_epochs=EPOCHS,max_steps=MAX_STEPS,lr_scheduler_type="cosine",
    optim="adamw_torch_fused",logging_steps=1,save_steps=10**9,save_total_limit=1,gradient_checkpointing=GRAD_CKPT,
    ddp_find_unused_parameters=False,dataloader_num_workers=4,report_to="none",max_length=MAX_LEN,packing=False,
    completion_only_loss=True,eval_strategy="no",include_num_input_tokens_seen=True)
class FC(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if num_items_in_batch is not None: inputs["num_items_in_batch"]=num_items_in_batch
        inputs["skip_logits"]=True
        out=model(**inputs); loss=out.loss if hasattr(out,"loss") else out["loss"]
        return (loss,out) if return_outputs else loss
class TS(TrainerCallback):
    def on_log(self,a,s,c,logs=None,**k):
        if logs and R0 and "loss" in logs: print(time.strftime("[%H:%M:%S]"),f"step={s.global_step} loss={logs.get('loss')} acc={logs.get('mean_token_accuracy')} tok/s={logs.get('train_tokens_per_second')}",flush=True)
kw=dict(model=model,args=args,train_dataset=ds["train"],callbacks=[TS()])
sig=inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters: kw["processing_class"]=tok
elif "tokenizer" in sig.parameters: kw["tokenizer"]=tok
trainer=FC(**kw)
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if R0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    print("SUMMARY "+json.dumps({"batch":BATCH,"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),
        "steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,
        "tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None}),flush=True)
    if os.environ.get("SAVE_MODEL","0")=="1":
        trainer.save_model(OUT_DIR); tok.save_pretrained(OUT_DIR); print("saved to",OUT_DIR,flush=True)
'''
with open("/tmp/train_gemma4_flce_clean.py","w") as f: f.write(FLCE_CLEAN)
import os, subprocess, sys, time, json as _j
def run_fc(extra, nproc=2, timeout=2400):
    env=os.environ.copy()
    env.update({"MODEL_DIR":MODEL_DIR,"PC_TRAIN":PC_TRAIN_GLOB,"PC_EVAL":PC_EVAL_GLOB,"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1",
        "TOKENIZERS_PARALLELISM":"false","PYTORCH_ALLOC_CONF":"expandable_segments:True","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","OMP_NUM_THREADS":"12"})
    env.update({k:str(v) for k,v in extra.items()})
    print(time.strftime("[%H:%M:%S]"),"START",{k:extra.get(k) for k in ["BATCH","GRAD_ACC","MAX_STEPS","GRADIENT_CHECKPOINTING","EPOCHS","SAVE_MODEL"]}); t=time.time()
    p=subprocess.run([sys.executable,"-m","torch.distributed.run","--standalone",f"--nproc_per_node={nproc}","/tmp/train_gemma4_flce_clean.py"],env=env,capture_output=True,text=True,timeout=timeout)
    o=(p.stdout or "")+"\n"+(p.stderr or "")
    for l in o.splitlines():
        if l.startswith(("[","SUMMARY","FLCE","saved")) or "trainable params" in l or "OutOfMemory" in l or "Error:" in l: print(l)
    print(time.strftime("[%H:%M:%S]"),f"END wall={time.time()-t:.0f}s rc={p.returncode}"); return p.returncode,o
for B in [4,8,12]:
    rc,o=run_fc(dict(BATCH=B,GRAD_ACC=4,MAX_STEPS=16,GRADIENT_CHECKPOINTING=0,EPOCHS=1,OUT_DIR=f"/tmp/fc_b{B}",SAVE_MODEL=0),timeout=1200)
    s=None
    for l in o.splitlines():
        if l.startswith("SUMMARY"): s=_j.loads(l[7:].strip())
    print("  =>",B,(s or {"oom":"OutOfMemory" in o}))
print("\nRef: clean batch1 no-ckpt(non-FLCE)=2930 tok/s; FLCE removes logits so batch should grow")

In [ ]:
# Cell 47 - FULL CLEAN TRAIN (packing=False, completion_only, sdpa, liger, r64 rsLoRA, batch1, 2 epochs) -> NEW path
import time
print("Expect ~50-55 min for 2 epochs at ~2930 tok/s. Live per-step timestamps below.")
rc, o = run_clean(dict(BATCH=1, GRAD_ACC=8, EPOCHS=2, MAX_STEPS=-1, GRADIENT_CHECKPOINTING=0,
                       LORA_R=64, LORA_ALPHA=128, USE_RSLORA=1, LR="7e-5",
                       OUT_DIR="/tmp/gemma4_lora_clean_nopack", SAVE_MODEL=1), nproc=2, timeout=6000)
print("CLEAN FULL TRAIN rc:", rc)
for l in o.splitlines():
    if l.startswith("SUMMARY") or l.startswith("saved"): print(l)

In [ ]:
# Cell 47 - FULL CLEAN TRAIN (packing=False, completion_only, sdpa, liger, r64 rsLoRA, batch1, 2 epochs) -> NEW path
import time
print("Expect ~50-55 min for 2 epochs at ~2930 tok/s. Live per-step timestamps below.")
rc, o = run_clean(dict(BATCH=1, GRAD_ACC=8, EPOCHS=2, MAX_STEPS=-1, GRADIENT_CHECKPOINTING=0,
                       LORA_R=64, LORA_ALPHA=128, USE_RSLORA=1, LR="7e-5",
                       OUT_DIR="/tmp/gemma4_lora_clean_nopack", SAVE_MODEL=1), nproc=2, timeout=6000)
print("CLEAN FULL TRAIN rc:", rc)
for l in o.splitlines():
    if l.startswith("SUMMARY") or l.startswith("saved"): print(l)

In [ ]:
# Cell 48 - cheap Unsloth feasibility probe (import offline + gemma4 support + multi-gpu signal)
import importlib.util as u, subprocess, sys, os
print("unsloth installed:", u.find_spec("unsloth") is not None)
test = (
  "import os;os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1';\n"
  "import unsloth, importlib.metadata as md\n"
  "print('unsloth', md.version('unsloth'))\n"
  "src=''\n"
  "try:\n"
  "    import unsloth.models as M, inspect, glob\n"
  "    files=glob.glob(os.path.dirname(M.__file__)+'/*.py')\n"
  "    txt=''.join(open(f,encoding='utf-8',errors='ignore').read() for f in files)\n"
  "    print('mentions gemma4:', 'gemma4' in txt.lower() or 'gemma-4' in txt.lower() or 'gemma 4' in txt.lower())\n"
  "    print('mentions gemma3:', 'gemma3' in txt.lower())\n"
  "    print('multi-GPU guard present:', ('num_gpus' in txt) or ('device_count' in txt) or ('multi' in txt.lower() and 'gpu' in txt.lower()))\n"
  "except Exception as e:\n"
  "    print('introspect err', repr(e))\n"
)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"-c",test], env=env, capture_output=True, text=True, timeout=400)
print("STDOUT:\n", p.stdout)
print("STDERR tail:\n", (p.stderr or "")[-1200:])

In [ ]:
# Cell 49 - build MESSAGE-ONLY prompt/completion dataset (drop raw-text rows; keep SFT instruction rows)
import json, glob, os
from pathlib import Path
MSG=Path("/tmp/ds_pc_msg"); (MSG/"train").mkdir(parents=True,exist_ok=True); (MSG/"eval").mkdir(parents=True,exist_ok=True)
def filt(src_glob, dst):
    n=0
    with open(dst,"w",encoding="utf-8") as w:
        for f in glob.glob(src_glob):
            for ln in open(f):
                if not ln.strip(): continue
                r=json.loads(ln)
                if isinstance(r.get("prompt"),str) and r["prompt"].strip():  # message-derived rows only
                    w.write(json.dumps(r,ensure_ascii=False)+"\n"); n+=1
    return n
ntr=filt("/tmp/ds_pc/train/*.jsonl", MSG/"train"/"part.jsonl")
nev=filt("/tmp/ds_pc/eval/*.jsonl",  MSG/"eval"/"part.jsonl")
MSG_TRAIN_GLOB=str(MSG/"train"/"*.jsonl"); MSG_EVAL_GLOB=str(MSG/"eval"/"*.jsonl")
print("message-only train rows:",ntr,"| eval rows:",nev)
print("MSG_TRAIN_GLOB:",MSG_TRAIN_GLOB)
# token-length stats of prompt vs completion on msg-only (why max_len matters)
import numpy as np
pl=[]; cl=[]
with open(MSG/"train"/"part.jsonl") as fh:
    for i,ln in enumerate(fh):
        if i>=400: break
        r=json.loads(ln); pl.append(len(tok(r["prompt"]).input_ids)); cl.append(len(tok(r["completion"]).input_ids))
pl=np.array(pl); cl=np.array(cl)
print("prompt tokens: p50 %.0f p90 %.0f max %d"%(np.percentile(pl,50),np.percentile(pl,90),pl.max()))
print("completion tokens: p50 %.0f p90 %.0f max %d"%(np.percentile(cl,50),np.percentile(cl,90),cl.max()))
print("=> prompt+completion p90:", int(np.percentile(pl+cl,90)), "| how many exceed 2048:", int((pl+cl>2048).sum()),"/400", "| exceed 3072:", int((pl+cl>3072).sum()),"/400")

In [ ]:
# Cell 50 - Phase 1 benchmark (message-only, completion_only, packing=False): r32/r64 x max_len 2048/3072, 40 steps
import json as _j
bench={}
for R in [32,64]:
    for ML in [2048,3072]:
        rc,o = run_clean(dict(PC_TRAIN=MSG_TRAIN_GLOB, PC_EVAL=MSG_EVAL_GLOB,
                              BATCH=1, GRAD_ACC=8, MAX_STEPS=40, GRADIENT_CHECKPOINTING=0,
                              LORA_R=R, LORA_ALPHA=2*R, USE_RSLORA=1, MAX_LEN=ML,
                              OUT_DIR=f"/tmp/bench_r{R}_ml{ML}", SAVE_MODEL=0), nproc=2, timeout=1200)
        s=None
        for l in o.splitlines():
            if l.startswith("SUMMARY"): s=_j.loads(l[7:].strip())
        bench[(R,ML)]=dict(rc=rc,oom=("OutOfMemory" in o),tok_s=(s or {}).get("tokens_per_s"),
                           vram=(s or {}).get("max_vram_gb"),loss=(s or {}).get("train_loss"))
        print("  =>",f"r{R} ml{ML}",bench[(R,ML)])
print("\n==== PHASE-1 BENCH (msg-only, completion_only) ====")
print("%-6s %-7s %-5s %-6s %-10s %-7s %-8s"%("r","max_len","rc","oom","tokens/s","vram","loss"))
for (R,ML),v in bench.items():
    print("%-6s %-7s %-5s %-6s %-10s %-7s %-8s"%(R,ML,v["rc"],v["oom"],v["tok_s"],v["vram"],v["loss"]))
ok=[(k,v) for k,v in bench.items() if v["rc"]==0 and v["tok_s"]]
if ok: print("\nFastest:", max(ok,key=lambda x:x[1]["tok_s"]))

In [ ]:
# Cell 51 - Phase 2 PILOT (150 steps, r64, max_len3072, message-only, completion_only) -> save new adapter
rc, o = run_clean(dict(PC_TRAIN=MSG_TRAIN_GLOB, PC_EVAL=MSG_EVAL_GLOB,
                       BATCH=1, GRAD_ACC=8, MAX_STEPS=150, GRADIENT_CHECKPOINTING=0,
                       LORA_R=64, LORA_ALPHA=128, USE_RSLORA=1, MAX_LEN=3072, LR="7e-5",
                       OUT_DIR="/tmp/gemma4_lora_clean_msgonly_r64", SAVE_MODEL=1), nproc=2, timeout=2400)
print("PILOT rc:", rc)
for l in o.splitlines():
    if l.startswith("SUMMARY") or l.startswith("saved"): print(l)

In [ ]:
# Cell 52 - Phase 3: inference compare base vs DIRTY adapter vs CLEAN adapter on held-out eval prompts
CMP = r'''
import os, json, glob, torch
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import PeftModel
MODEL_DIR=os.environ["MODEL_DIR"]; EVAL=os.environ["PC_EVAL"]
DIRTY=os.environ["DIRTY"]; CLEAN=os.environ["CLEAN"]
tok=AutoTokenizer.from_pretrained(MODEL_DIR)
prompts=[]
for f in glob.glob(EVAL):
    for ln in open(f):
        r=json.loads(ln); prompts.append((r["prompt"], r["completion"]))
prompts=prompts[:3]
def gen(model, prompt):
    enc=tok(prompt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        o=model.generate(**enc, max_new_tokens=160, do_sample=False)
    return tok.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
for tag, adapter in [("BASE",None),("DIRTY",DIRTY),("CLEAN",CLEAN)]:
    m=AutoModelForImageTextToText.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to("cuda:0").eval()
    if adapter: m=PeftModel.from_pretrained(m, adapter).to("cuda:0").eval()
    print("\n"+"="*30, tag, "="*30)
    for i,(p,ref) in enumerate(prompts):
        out=gen(m,p)
        print(f"--- prompt{i} (user tail): ...{p[-160:].strip()[-120:]!r}")
        print(f"    [{tag}] {out[:360]!r}")
        if tag=="CLEAN" : print(f"    [REF]  {ref[:200]!r}")
    del m; torch.cuda.empty_cache()
'''
import os, subprocess, sys
with open("/tmp/cmp_infer.py","w") as f: f.write(CMP)
env=os.environ.copy(); env.update({"MODEL_DIR":MODEL_DIR,"PC_EVAL":MSG_EVAL_GLOB,
   "DIRTY":"/tmp/gemma4_lora_main","CLEAN":"/tmp/gemma4_lora_clean_msgonly_r64",
   "HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/cmp_infer.py"], env=env, capture_output=True, text=True, timeout=1200)
print(p.stdout[-4000:]); print("ERR:",(p.stderr or '')[-400:])

In [ ]:
# Cell 53 - GET Unsloth wheels + install --no-deps (protect TRL stack) + import probe in subprocess
import os, glob, subprocess, sys, time
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
OPT="@ML_DB.MODELS.FUZZ_DATASET_STAGE/gemma4_e4b_smart_pack/offline_env_v2/blackwell_fast10_trl_liger_rslora_20260605/wheelhouse_optional"
WH=Path("/tmp/wh_unsloth"); WH.mkdir(parents=True,exist_ok=True)
wheels=["unsloth-2026.6.1-py3-none-any.whl","unsloth_zoo-2026.6.1-py3-none-any.whl",
  "cut_cross_entropy-25.1.1-py3-none-any.whl","tyro-1.0.13-py3-none-any.whl",
  "typeguard-4.5.2-py3-none-any.whl","docstring_parser-0.18.0-py3-none-any.whl",
  "xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl"]
for w in wheels:
    if not (WH/w).exists():
        try: session.file.get(f"{OPT}/{w}", str(WH))
        except Exception as e: print("GET fail",w,repr(e))
print("got:", sorted(p.name for p in WH.glob('*.whl')))
# install WITHOUT deps so we don't disturb torch/transformers/trl/peft already working
r=subprocess.run([sys.executable,"-m","pip","install","--no-index","--no-deps","--find-links",str(WH),
   "unsloth","unsloth_zoo","cut_cross_entropy","tyro","typeguard","docstring_parser"],capture_output=True,text=True)
print((r.stdout or '')[-700:]); print((r.stderr or '')[-500:]); print("install rc:",r.returncode)
# probe import in FRESH subprocess (keep kernel clean)
probe=("import os;os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1';\n"
  "import time;t=time.time()\n"
  "try:\n"
  "    import unsloth; import importlib.metadata as md\n"
  "    print('unsloth', md.version('unsloth'),'import OK in %.0fs'%(time.time()-t))\n"
  "    from unsloth import FastModel, FastLanguageModel; print('FastModel import OK')\n"
  "except Exception as e:\n"
  "    import traceback; traceback.print_exc()\n")
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"-c",probe],env=env,capture_output=True,text=True,timeout=600)
print("\n--- IMPORT PROBE ---"); print(p.stdout[-2000:]); print("ERR:",(p.stderr or '')[-2000:])

In [ ]:
# Cell 54 - add bitsandbytes (Unsloth hard-requires it) + re-probe import
import os, subprocess, sys
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
OPT="@ML_DB.MODELS.FUZZ_DATASET_STAGE/gemma4_e4b_smart_pack/offline_env_v2/blackwell_fast10_trl_liger_rslora_20260605/wheelhouse_optional"
WH=Path("/tmp/wh_unsloth")
bnb="bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl"
if not (WH/bnb).exists(): session.file.get(f"{OPT}/{bnb}", str(WH))
r=subprocess.run([sys.executable,"-m","pip","install","--no-index","--no-deps","--find-links",str(WH),"bitsandbytes"],capture_output=True,text=True)
print((r.stdout or '')[-300:]); print("bnb install rc:",r.returncode)
probe=("import os;os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1'\n"
  "import time;t=time.time()\n"
  "try:\n"
  "    import unsloth; import importlib.metadata as md\n"
  "    from unsloth import FastModel, FastLanguageModel\n"
  "    print('unsloth',md.version('unsloth'),'+ FastModel import OK in %.0fs'%(time.time()-t))\n"
  "    import torch; print('torch',torch.__version__,'gpus',torch.cuda.device_count())\n"
  "except Exception as e:\n"
  "    import traceback; traceback.print_exc()\n")
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"-c",probe],env=env,capture_output=True,text=True,timeout=600)
print("\n--- IMPORT PROBE 2 ---"); print(p.stdout[-1500:]); print("ERR:",(p.stderr or '')[-1500:])

In [ ]:
# Cell 55 - DECISIVE: can Unsloth FastModel load gemma4 on Blackwell (head_dim>256)? + LoRA + 1 forward
LOAD = r'''
import os, time, traceback
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch
try:
    from unsloth import FastModel
    t=time.time()
    model, tok = FastModel.from_pretrained(
        model_name="/tmp/gemma4_base",
        max_seq_length=2048, dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False)
    print("LOAD OK in %.0fs ->"%(time.time()-t), type(model).__name__)
    model = FastModel.get_peft_model(model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth")
    print("get_peft_model OK")
    ids = tok("Sinh seed fuzzing cho torch.add.", return_tensors="pt").to("cuda:0")
    ids["labels"]=ids["input_ids"].clone()
    out = model(**ids)
    print("FORWARD OK, loss=", float(out.loss))
    out.loss.backward(); print("BACKWARD OK -> Unsloth trains gemma4 on this GPU")
except Exception as e:
    print("UNSLOTH FAILED:"); traceback.print_exc()
'''
import os, subprocess, sys
with open("/tmp/unsloth_load.py","w") as f: f.write(LOAD)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_load.py"], env=env, capture_output=True, text=True, timeout=900)
print(p.stdout[-2500:]); print("ERR:",(p.stderr or '')[-2500:])

In [ ]:
# Cell 56 - debug why Unsloth rejects local gemma4 config
DBG = r'''
import os, inspect, traceback
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
P="/tmp/gemma4_base"
print("exists:", os.path.isdir(P), "| config.json:", os.path.exists(P+"/config.json"))
print("dir:", sorted(os.listdir(P)))
from transformers import AutoConfig
try:
    c=AutoConfig.from_pretrained(P); print("AutoConfig OK model_type=", c.model_type)
except Exception as e: traceback.print_exc()
import unsloth_zoo.hf_utils as hu
src=inspect.getsource(hu.get_transformers_model_type)
print("\n=== get_transformers_model_type source (head) ===")
print(src[:2200])
'''
import os, subprocess, sys
with open("/tmp/unsloth_dbg.py","w") as f: f.write(DBG)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_dbg.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-3000:]); print("ERR:",(p.stderr or '')[-800:])

In [ ]:
# Cell 57 - read get_transformers_model_type to find the offline-rejection cause
DBG2 = r'''
import os, inspect
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import unsloth  # sets env flag so unsloth_zoo can import
import unsloth_zoo.hf_utils as hu
print(inspect.getsource(hu.get_transformers_model_type))
'''
import os, subprocess, sys
with open("/tmp/unsloth_dbg2.py","w") as f: f.write(DBG2)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_dbg2.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-3500:]); print("ERR:",(p.stderr or '')[-500:])

In [ ]:
# Cell 58 - read HEAD of get_transformers_model_type (the offline-rejection branch)
DBG3 = r'''
import os, inspect
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import unsloth
import unsloth_zoo.hf_utils as hu
src=inspect.getsource(hu.get_transformers_model_type)
print(src[:1900])
'''
import os, subprocess, sys
with open("/tmp/unsloth_dbg3.py","w") as f: f.write(DBG3)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_dbg3.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-2400:]); print("ERR:",(p.stderr or '')[-400:])

In [ ]:
# Cell 59 - read Unsloth loader around config fetch (line ~1090-1145) to find why config=None offline
DBG4 = r'''
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import unsloth, unsloth.models.loader as L
src=open(L.__file__).read().splitlines()
for i in range(1085,1145):
    if i < len(src): print(i+1, src[i])
'''
import os, subprocess, sys
with open("/tmp/unsloth_dbg4.py","w") as f: f.write(DBG4)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_dbg4.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-3200:]); print("ERR:",(p.stderr or '')[-400:])

In [ ]:
# Cell 60 - reproduce Unsloth's exact AutoConfig call to capture the real failure
REPRO = r'''
import os, traceback
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import unsloth  # apply unsloth patches first (it patches transformers)
from transformers import AutoConfig
for lfo in (False, True):
    try:
        c=AutoConfig.from_pretrained("/tmp/gemma4_base", token=None, revision=None,
                                     trust_remote_code=False, local_files_only=lfo)
        print(f"local_files_only={lfo}: OK model_type=", c.model_type)
    except Exception as e:
        print(f"local_files_only={lfo}: FAILED ->", repr(e)[:300])
'''
import os, subprocess, sys
with open("/tmp/unsloth_repro.py","w") as f: f.write(REPRO)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_repro.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-1800:]); print("ERR:",(p.stderr or '')[-1200:])

In [ ]:
# Cell 61 - probe REAL internet egress from the GPU service (no offline flags)
import subprocess, sys, socket
def dns(host):
    try: return socket.gethostbyname(host)
    except Exception as e: return f"FAIL {e}"
for h in ["pypi.org","files.pythonhosted.org","huggingface.co","github.com"]:
    print(f"DNS {h}:", dns(h))
# try an actual pip index reach (no --no-index), short timeout
r=subprocess.run([sys.executable,"-m","pip","download","--no-deps","-d","/tmp/netchk","pip","--timeout","15"],
                 capture_output=True,text=True)
print("pip download rc:", r.returncode)
print((r.stdout or '')[-500:]); print((r.stderr or '')[-700:])

In [ ]:
# Cell 62 - GET Unsloth model pack + reconstruct model.safetensors from parts
import os, glob, json, shutil, time
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
PACK="@ML_DB.MODELS.FUZZ_DATASET_STAGE/unsloth_gemma4_e4b_offline_pack_20260605"
B=Path("/tmp/uns_pack")
for sub in ["meta","model_files","model_parts"]:
    d=B/sub; d.mkdir(parents=True,exist_ok=True)
    if not any(d.iterdir()):
        print(time.strftime('[%H:%M:%S]'),"GET",sub,"...",flush=True)
        session.file.get(f"{PACK}/{sub}", str(d))
print("parts:", sorted(p.name for p in (B/'model_parts').glob('*')))
man=json.load(open(B/"meta"/"model_manifest.json"))
MD=Path("/tmp/unsloth_gemma4"); shutil.rmtree(MD,ignore_errors=True); MD.mkdir(parents=True)
for ent in man["files"]:
    out=MD/ent["rel"]; out.parent.mkdir(parents=True,exist_ok=True)
    if ent["parts"]:
        with open(out,"wb") as w:
            for part in ent["parts"]:
                with open(B/"model_parts"/part,"rb") as r: shutil.copyfileobj(r,w,64*1024*1024)
    else:
        shutil.copy2(B/"model_files"/ent["safe"], out)
print("reconstructed files:", sorted(os.listdir(MD)))
st=MD/"model.safetensors"
print("model.safetensors GB:", round(st.stat().st_size/1e9,2) if st.exists() else "MISSING")
# check num_kv_shared_layers in the unsloth config
cfg=json.load(open(MD/"config.json")); tc=cfg.get('text_config',{})
print("unsloth config model_type:", cfg.get('model_type'), "| text num_kv_shared_layers:", tc.get('num_kv_shared_layers'))

In [ ]:
# Cell 63 - DECISIVE: load Unsloth-prepared gemma4 via FastLanguageModel + LoRA + forward/backward
LOAD = r'''
import os, time, traceback
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch
MD="/tmp/unsloth_gemma4"
try:
    from unsloth import FastLanguageModel
    t=time.time()
    model, tok = FastLanguageModel.from_pretrained(
        model_name=MD, max_seq_length=2048, dtype=torch.bfloat16,
        load_in_4bit=False, full_finetuning=False, local_files_only=True)
    print("LOAD OK in %.0fs ->"%(time.time()-t), type(model).__name__)
    model = FastLanguageModel.get_peft_model(model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=0)
    print("get_peft_model OK")
    enc=tok("Sinh seed fuzzing cho torch.add va de xuat oracle.", return_tensors="pt").to("cuda:0")
    enc["labels"]=enc["input_ids"].clone()
    out=model(**enc); print("FORWARD OK loss=", float(out.loss))
    out.loss.backward(); print("BACKWARD OK -> UNSLOTH TRAINS GEMMA4 \u2713")
    print("max_vram_gb=", round(torch.cuda.max_memory_allocated()/1e9,1))
except Exception as e:
    print("UNSLOTH LOAD/TRAIN FAILED:"); traceback.print_exc()
'''
import os, subprocess, sys
with open("/tmp/unsloth_load2.py","w") as f: f.write(LOAD)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_load2.py"], env=env, capture_output=True, text=True, timeout=1200)
print(p.stdout[-3000:]); print("ERR:",(p.stderr or '')[-3000:])

In [ ]:
# Cell 64 - Try Unsloth on the GOOGLE base model FIRST (per request): FastLanguageModel on /tmp/gemma4_base
LOADG = r'''
import os, time, traceback
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch
MD="/tmp/gemma4_base"   # GOOGLE original model
import json
cfg=json.load(open(MD+"/config.json")); tc=cfg.get("text_config",{})
print("GOOGLE config num_kv_shared_layers =", tc.get("num_kv_shared_layers"))
try:
    from unsloth import FastLanguageModel
    t=time.time()
    model, tok = FastLanguageModel.from_pretrained(
        model_name=MD, max_seq_length=2048, dtype=torch.bfloat16,
        load_in_4bit=False, full_finetuning=False, local_files_only=True)
    print("LOAD OK in %.0fs ->"%(time.time()-t), type(model).__name__)
    model = FastLanguageModel.get_peft_model(model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=0)
    enc=tok("Sinh seed fuzzing cho torch.add.", return_tensors="pt").to("cuda:0"); enc["labels"]=enc["input_ids"].clone()
    out=model(**enc); print("FORWARD OK loss=", float(out.loss)); out.loss.backward(); print("BACKWARD OK -> Unsloth trains GOOGLE gemma4")
except Exception as e:
    print("UNSLOTH + GOOGLE MODEL FAILED:"); traceback.print_exc()
'''
import os, subprocess, sys, time
with open("/tmp/unsloth_google.py","w") as f: f.write(LOADG)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
print(time.strftime('[%H:%M:%S]'),"start (unsloth import ~80s + load ~30s)",flush=True)
p=subprocess.run([sys.executable,"/tmp/unsloth_google.py"], env=env, capture_output=True, text=True, timeout=1200)
print(p.stdout[-2500:]); print("ERR:",(p.stderr or '')[-2500:])

In [ ]:
# Cell 65 - DIAGNOSE real cause: full traceback of AutoConfig under unsloth patch (Google + Unsloth repo) + versions
DIAG = r'''
import os, traceback, time
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import importlib.metadata as md
print("=== versions (before unsloth) ===")
for p in ["unsloth","unsloth_zoo","transformers","torch","triton","bitsandbytes","xformers","peft","trl"]:
    try: print(" ",p, md.version(p))
    except Exception as e: print(" ",p,"MISSING")
print("\n=== import unsloth (patches transformers) ===", flush=True)
import unsloth
from transformers import AutoConfig
for name,path in [("GOOGLE","/tmp/gemma4_base"),("UNSLOTH","/tmp/unsloth_gemma4")]:
    print(f"\n=== AutoConfig {name} ({path}) ===")
    try:
        c=AutoConfig.from_pretrained(path, local_files_only=True)
        print("  OK model_type=", c.model_type)
    except Exception:
        traceback.print_exc()
'''
import os, subprocess, sys, time
with open("/tmp/unsloth_diag.py","w") as f: f.write(DIAG)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
print(time.strftime('[%H:%M:%S]'),"running diag (unsloth import ~80s)...",flush=True)
p=subprocess.run([sys.executable,"/tmp/unsloth_diag.py"], env=env, capture_output=True, text=True, timeout=600)
print(p.stdout[-2500:]); print("--- STDERR (tracebacks) ---"); print((p.stderr or '')[-3500:])

In [ ]:
# Cell 66 - read unsloth_zoo gemma4 temporary patch to find disable switch / the __getattr__ bug
RD = r'''
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import unsloth   # satisfies the import guard
import importlib
g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
src=open(g.__file__).read().splitlines()
for i in range(90,160):
    if i < len(src): print(i+1, src[i])
print("\n=== UNSLOTH_* env switches in temporary_patches ===")
import glob, re, os as _os
base=_os.path.dirname(g.__file__)
hits=set()
for f in glob.glob(base+"/*.py"):
    for ln in open(f,encoding="utf-8",errors="ignore"):
        for m in re.findall(r"UNSLOTH_[A-Z0-9_]+", ln): hits.add(m)
print(sorted(hits))
'''
import os, subprocess, sys
with open("/tmp/unsloth_rd.py","w") as f: f.write(RD)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
p=subprocess.run([sys.executable,"/tmp/unsloth_rd.py"], env=env, capture_output=True, text=True, timeout=300)
print(p.stdout[-3800:]); print("ERR:",(p.stderr or '')[-600:])

In [ ]:
# Cell 67 - WORKAROUND: neutralize unsloth_zoo gemma4 proxy, then load GOOGLE model via FastModel
WA = r'''
import os, time, traceback, importlib
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, unsloth
# neutralize the buggy KV-shared proxy: forward all attrs to the real config (drop the num_kv guard)
try:
    g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
    P=getattr(g,"_Gemma4KVSharedSafeProxy",None)
    if P is not None:
        def _getattr(self,name): return getattr(object.__getattribute__(self,"_real"), name)
        P.__getattr__=_getattr
        print("patched _Gemma4KVSharedSafeProxy.__getattr__ to passthrough")
    else:
        print("proxy class not found (name changed)")
except Exception as e:
    print("could not patch proxy:", repr(e))
from transformers import AutoConfig
try:
    c=AutoConfig.from_pretrained("/tmp/gemma4_base", local_files_only=True)
    print("AutoConfig GOOGLE now OK, model_type=", c.model_type)
except Exception:
    traceback.print_exc()
try:
    from unsloth import FastModel
    t=time.time()
    model, tok = FastModel.from_pretrained(model_name="/tmp/gemma4_base", max_seq_length=2048,
        dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False, local_files_only=True)
    print("FastModel LOAD OK in %.0fs ->"%(time.time()-t), type(model).__name__)
    model=FastModel.get_peft_model(model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=0)
    enc=tok("Sinh seed fuzzing cho torch.add.", return_tensors="pt").to("cuda:0"); enc["labels"]=enc["input_ids"].clone()
    out=model(**enc); print("FORWARD OK loss=", float(out.loss)); out.loss.backward(); print("BACKWARD OK -> UNSLOTH TRAINS GOOGLE GEMMA4")
except Exception:
    traceback.print_exc()
'''
import os, subprocess, sys, time
with open("/tmp/unsloth_wa.py","w") as f: f.write(WA)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
print(time.strftime('[%H:%M:%S]'),"running workaround (import ~80s + load)...",flush=True)
p=subprocess.run([sys.executable,"/tmp/unsloth_wa.py"], env=env, capture_output=True, text=True, timeout=1200)
print(p.stdout[-3000:]); print("--- STDERR ---"); print((p.stderr or '')[-3000:])

In [ ]:
# Cell 68 - proxy patch + FastLanguageModel (text-only) GOOGLE model -> clear VERDICT (loss/backward/vram)
WA2 = r'''
import os, time, traceback, importlib, warnings, json
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, unsloth
g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
P=getattr(g,"_Gemma4KVSharedSafeProxy",None)
if P is not None:
    P.__getattr__=lambda self,name: getattr(object.__getattribute__(self,"_real"), name)
verdict={}
try:
    from unsloth import FastLanguageModel
    from transformers import AutoTokenizer
    t=time.time()
    model, tok = FastLanguageModel.from_pretrained(model_name="/tmp/gemma4_base", max_seq_length=2048,
        dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False, local_files_only=True)
    verdict["load_s"]=round(time.time()-t,0)
    model=FastLanguageModel.get_peft_model(model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=0)
    at=AutoTokenizer.from_pretrained("/tmp/gemma4_base", use_fast=True)
    ids=at("Sinh seed fuzzing cho torch.add va de xuat oracle.", return_tensors="pt").input_ids.to("cuda:0")
    out=model(input_ids=ids, labels=ids.clone())
    verdict["loss"]=round(out.loss.item(),4)
    out.loss.backward()
    gnorm=sum((p.grad.norm().item() for p in model.parameters() if p.grad is not None))
    verdict["backward_ok"]=True; verdict["grad_sum_norm"]=round(gnorm,3)
    verdict["max_vram_gb"]=round(torch.cuda.max_memory_allocated()/1e9,1)
    verdict["status"]="PASS - Unsloth trains GOOGLE gemma4 on Blackwell"
except Exception as e:
    verdict["status"]="FAIL"; verdict["err"]=repr(e)[:300]; traceback.print_exc()
print("\n=== UNSLOTH VERDICT ==="); print(json.dumps(verdict, indent=2))
'''
import os, subprocess, sys, time
with open("/tmp/unsloth_wa2.py","w") as f: f.write(WA2)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0"})
print(time.strftime('[%H:%M:%S]'),"running (import ~80s + load)...",flush=True)
p=subprocess.run([sys.executable,"/tmp/unsloth_wa2.py"], env=env, capture_output=True, text=True, timeout=1200)
print(p.stdout[-1500:]); print("--- tail stderr ---"); print((p.stderr or '')[-600:])

In [ ]:
# Cell 69 - UNSLOTH pilot (STREAMING real-time output, 20 steps)
UNS = r'''
import os, glob, time, inspect, importlib, warnings, json, sys
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"; os.environ["TOKENIZERS_PARALLELISM"]="false"
def log(m): print(time.strftime("[%H:%M:%S]")+" "+m, flush=True)
log("importing torch + unsloth (~60-90s, patching transformers)...")
import torch, unsloth
g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
P=getattr(g,"_Gemma4KVSharedSafeProxy",None)
if P is not None: P.__getattr__=lambda self,name: getattr(object.__getattribute__(self,"_real"), name)
log("unsloth imported + proxy patched. Loading model (16GB, ~30s)...")
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from datasets import load_dataset
MAX_LEN=int(os.environ.get("MAX_LEN","3072")); BATCH=int(os.environ.get("BATCH","8")); GRAD_ACC=int(os.environ.get("GRAD_ACC","4")); STEPS=int(os.environ.get("MAX_STEPS","20"))
model, tok = FastLanguageModel.from_pretrained(model_name="/tmp/gemma4_base", max_seq_length=MAX_LEN, dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False, local_files_only=True)
log("model loaded. Attaching LoRA r64...")
model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, lora_dropout=0.0, bias="none", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth", use_rslora=True, random_state=0)
log("tokenizing dataset...")
ds=load_dataset("json", data_files={"train":sorted(glob.glob("/tmp/ds_pc_msg/train/*.jsonl"))})
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters); return SFTConfig(**{k:v for k,v in kw.items() if k in al})
args=sc(output_dir="/tmp/uns_pilot", per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, learning_rate=7e-5, warmup_ratio=0.03, max_steps=STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=1, save_steps=10**9, report_to="none", max_length=MAX_LEN, packing=False, completion_only_loss=True, bf16=True, include_num_input_tokens_seen=True, dataloader_num_workers=2)
class TS(TrainerCallback):
    def on_log(self,a,s,c,logs=None,**k):
        if logs and "loss" in logs: log(f"step {s.global_step}/{STEPS}  loss={logs.get('loss')}  acc={logs.get('mean_token_accuracy')}  tok/s={logs.get('train_tokens_per_second')}")
kw=dict(model=model,args=args,train_dataset=ds["train"],callbacks=[TS()])
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters: kw["processing_class"]=tok
else: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
log(f"START training: batch={BATCH} gacc={GRAD_ACC} maxlen={MAX_LEN} rows={len(trainer.train_dataset)}")
t0=time.time(); res=trainer.train(); dt=time.time()-t0
ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
log("SUMMARY "+json.dumps({"engine":"unsloth-1gpu","batch":BATCH,"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),"steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,"tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None}))
'''
import os, subprocess, sys, time
with open("/tmp/unsloth_pilot.py","w") as f: f.write(UNS)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","CUDA_VISIBLE_DEVICES":"0","PYTHONUNBUFFERED":"1","MAX_LEN":"3072","BATCH":"8","GRAD_ACC":"4","MAX_STEPS":"20"})
print(time.strftime('[%H:%M:%S]'),"launching (output streams live below)...",flush=True)
proc=subprocess.Popen([sys.executable,"-u","/tmp/unsloth_pilot.py"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    s=line.rstrip()
    if s.startswith("[") or "SUMMARY" in s or "Unsloth" in s or "Error" in s or "OutOfMemory" in s or "Traceback" in s:
        print(s, flush=True)
proc.wait()
print("[done] rc=", proc.returncode)

In [ ]:
# Cell 70 - UNSLOTH 2-GPU (torchrun DDP) + higher batch, STREAMING real-time, 20 steps
UNS2 = r'''
import os, glob, time, inspect, importlib, warnings, json
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"; os.environ["TOKENIZERS_PARALLELISM"]="false"
LR=int(os.environ.get("LOCAL_RANK","0")); R0=(LR==0)
def log(m):
    if R0: print(time.strftime("[%H:%M:%S]")+" "+m, flush=True)
log("importing torch+unsloth (~80s)...")
import torch
if torch.cuda.is_available(): torch.cuda.set_device(LR)
import unsloth
g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
P=getattr(g,"_Gemma4KVSharedSafeProxy",None)
if P is not None: P.__getattr__=lambda self,name: getattr(object.__getattribute__(self,"_real"), name)
log("loading model on each rank (16GB)...")
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from datasets import load_dataset
MAX_LEN=int(os.environ.get("MAX_LEN","3072")); BATCH=int(os.environ.get("BATCH","8")); GRAD_ACC=int(os.environ.get("GRAD_ACC","2")); STEPS=int(os.environ.get("MAX_STEPS","20"))
model, tok = FastLanguageModel.from_pretrained(model_name="/tmp/gemma4_base", max_seq_length=MAX_LEN, dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False, local_files_only=True)
log("attaching LoRA r64...")
model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, lora_dropout=0.0, bias="none", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth", use_rslora=True, random_state=0)
ds=load_dataset("json", data_files={"train":sorted(glob.glob("/tmp/ds_pc_msg/train/*.jsonl"))})
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters); return SFTConfig(**{k:v for k,v in kw.items() if k in al})
args=sc(output_dir="/tmp/uns_pilot2", per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, learning_rate=7e-5, warmup_ratio=0.03, max_steps=STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=1, save_steps=10**9, report_to="none", max_length=MAX_LEN, packing=False, completion_only_loss=True, bf16=True, include_num_input_tokens_seen=True, dataloader_num_workers=2, ddp_find_unused_parameters=False)
class TS(TrainerCallback):
    def on_log(self,a,s,c,logs=None,**k):
        if logs and "loss" in logs and R0: log(f"step {s.global_step}/{STEPS} loss={logs.get('loss')} acc={logs.get('mean_token_accuracy')} tok/s={logs.get('train_tokens_per_second')}")
kw=dict(model=model,args=args,train_dataset=ds["train"],callbacks=[TS()])
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters: kw["processing_class"]=tok
else: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
log(f"START 2-GPU: per_gpu_batch={BATCH} gacc={GRAD_ACC} world={os.environ.get('WORLD_SIZE')} maxlen={MAX_LEN}")
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if R0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    log("SUMMARY "+json.dumps({"engine":"unsloth-2gpu","per_gpu_batch":BATCH,"world":int(os.environ.get('WORLD_SIZE','1')),"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),"steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,"tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None}))
'''
import os, subprocess, sys, time
with open("/tmp/unsloth_pilot2.py","w") as f: f.write(UNS2)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","PYTHONUNBUFFERED":"1","MAX_LEN":"3072","BATCH":"10","GRAD_ACC":"2","MAX_STEPS":"20","NCCL_DEBUG":"WARN"})
print(time.strftime('[%H:%M:%S]'),"launch torchrun nproc=2 (streams live)...",flush=True)
proc=subprocess.Popen([sys.executable,"-u","-m","torch.distributed.run","--standalone","--nproc_per_node=2","/tmp/unsloth_pilot2.py"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    s=line.rstrip()
    if s.startswith("[") or "SUMMARY" in s or "Error" in s or "OutOfMemory" in s or "Traceback" in s or "not supported" in s or "multi" in s.lower() or "NCCL" in s: print(s, flush=True)
proc.wait(); print("[done] rc=", proc.returncode)

In [ ]:
# Cell 71 - LOAD UNSLOTH MODEL ONCE IN KERNEL (no more reloading per cell). 1 GPU.
import os, time, importlib, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"; os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
def log(m): print(time.strftime("[%H:%M:%S]")+" "+m, flush=True)
log("importing unsloth in kernel (one-time ~80s)...")
import torch, unsloth
g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
P=getattr(g,"_Gemma4KVSharedSafeProxy",None)
if P is not None: P.__getattr__=lambda self,name: getattr(object.__getattribute__(self,"_real"), name)
log("loading gemma4 (16GB)...")
from unsloth import FastLanguageModel
uns_model, uns_tok = FastLanguageModel.from_pretrained(model_name="/tmp/gemma4_base", max_seq_length=4096,
    dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False, local_files_only=True)
uns_model = FastLanguageModel.get_peft_model(uns_model, r=64, lora_alpha=128, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", use_rslora=True, random_state=0)
log("MODEL READY in kernel (uns_model, uns_tok). Re-runnable train cells will NOT reload.")
import json; print(json.dumps({"vram_after_load_gb": round(torch.cuda.max_memory_allocated()/1e9,1)}))

In [ ]:
# Cell 72 - UNSLOTH 2-GPU DDP with vision/audio FROZEN (fix for 'unused params') - STREAMING, 20 steps
DDP = r'''
import os, glob, time, inspect, importlib, warnings, json
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"; os.environ["TOKENIZERS_PARALLELISM"]="false"
LR=int(os.environ.get("LOCAL_RANK","0")); R0=(LR==0)
def log(m):
    if R0: print(time.strftime("[%H:%M:%S]")+" "+m, flush=True)
log("import torch+unsloth (~80s)...")
import torch
if torch.cuda.is_available(): torch.cuda.set_device(LR)
import unsloth
g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
P=getattr(g,"_Gemma4KVSharedSafeProxy",None)
if P is not None: P.__getattr__=lambda self,name: getattr(object.__getattribute__(self,"_real"), name)
log("load model (16GB) each rank...")
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from datasets import load_dataset
MAX_LEN=int(os.environ.get("MAX_LEN","3072")); BATCH=int(os.environ.get("BATCH","8")); GRAD_ACC=int(os.environ.get("GRAD_ACC","1")); STEPS=int(os.environ.get("MAX_STEPS","20")); RANK=int(os.environ.get("LORA_R","32"))
model, tok = FastLanguageModel.from_pretrained(model_name="/tmp/gemma4_base", max_seq_length=MAX_LEN, dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False, local_files_only=True)
model = FastLanguageModel.get_peft_model(model, r=RANK, lora_alpha=2*RANK, lora_dropout=0.0, bias="none", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth", use_rslora=True, random_state=0)
# *** FIX: freeze any trainable params in vision/audio branches (not used in text-only loss) ***
bad_keys=["vision_tower","audio_tower","embed_vision","embed_audio","vision_model","audio_model"]
frozen=0
for name,p in model.named_parameters():
    if p.requires_grad and any(k in name for k in bad_keys): p.requires_grad_(False); frozen+=1
bad_trainables=[n for n,p in model.named_parameters() if p.requires_grad and any(k in n for k in bad_keys)]
n_train=sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"froze {frozen} vision/audio tensors | remaining bad_trainables={len(bad_trainables)} | trainable params={n_train/1e6:.1f}M")
assert len(bad_trainables)==0, bad_trainables[:20]
ds=load_dataset("json", data_files={"train":sorted(glob.glob("/tmp/ds_pc_msg/train/*.jsonl"))})
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters); return SFTConfig(**{k:v for k,v in kw.items() if k in al})
args=sc(output_dir="/tmp/uns_ddp", per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, learning_rate=7e-5, warmup_ratio=0.03, max_steps=STEPS, lr_scheduler_type="cosine", optim="adamw_torch_fused", logging_steps=1, save_steps=10**9, report_to="none", max_length=MAX_LEN, packing=False, completion_only_loss=True, bf16=True, include_num_input_tokens_seen=True, dataloader_num_workers=2, ddp_find_unused_parameters=False)
class TS(TrainerCallback):
    def on_log(self,a,s,c,logs=None,**k):
        if logs and "loss" in logs and R0: log(f"step {s.global_step}/{STEPS} loss={logs.get('loss')} acc={logs.get('mean_token_accuracy')} tok/s={logs.get('train_tokens_per_second')}")
kw=dict(model=model,args=args,train_dataset=ds["train"],callbacks=[TS()])
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters: kw["processing_class"]=tok
else: kw["tokenizer"]=tok
trainer=SFTTrainer(**kw)
log(f"START DDP: world={os.environ.get('WORLD_SIZE')} per_gpu_batch={BATCH} gacc={GRAD_ACC} r={RANK} maxlen={MAX_LEN}")
t0=time.time(); res=trainer.train(); dt=time.time()-t0
if R0:
    ntok=getattr(trainer.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    log("SUMMARY "+json.dumps({"engine":"unsloth-2gpu-ddp","per_gpu_batch":BATCH,"world":int(os.environ.get('WORLD_SIZE','1')),"train_seconds":round(dt,1),"train_loss":round(float(res.training_loss),4),"steps":trainer.state.global_step,"max_vram_gb":round(mem,1),"tokens_seen":ntok,"tokens_per_s":round(ntok/dt,0) if (ntok and dt) else None}))
'''
import os, subprocess, sys, time
with open("/tmp/train_unsloth_ddp.py","w") as f: f.write(DDP)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","PYTHONUNBUFFERED":"1","CUDA_VISIBLE_DEVICES":"0,1","MAX_LEN":"3072","BATCH":"8","GRAD_ACC":"1","LORA_R":"32","MAX_STEPS":"20","NCCL_DEBUG":"WARN"})
print(time.strftime('[%H:%M:%S]'),"launch torchrun nproc=2 (streams live; ~4min load then train)...",flush=True)
proc=subprocess.Popen([sys.executable,"-u","-m","torch.distributed.run","--standalone","--nproc_per_node=2","/tmp/train_unsloth_ddp.py"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    s=line.rstrip()
    if s.startswith("[") or "SUMMARY" in s or "Error" in s or "OutOfMemory" in s or "did not receive grad" in s or "AssertionError" in s or "Traceback" in s: print(s, flush=True)
proc.wait(); print("[done] rc=", proc.returncode)

In [ ]:
# Cell 73 - UNSLOTH 2-GPU DDP BATCH SWEEP (load ONCE, sweep batches in same process), streaming
SWEEP = r'''
import os, glob, time, inspect, importlib, warnings, json
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"; os.environ["TOKENIZERS_PARALLELISM"]="false"
LR=int(os.environ.get("LOCAL_RANK","0")); R0=(LR==0)
def log(m):
    if R0: print(time.strftime("[%H:%M:%S]")+" "+m, flush=True)
log("import torch+unsloth (~80s, ONE-TIME)...")
import torch
if torch.cuda.is_available(): torch.cuda.set_device(LR)
import unsloth
g=importlib.import_module("unsloth_zoo.temporary_patches.gemma4")
P=getattr(g,"_Gemma4KVSharedSafeProxy",None)
if P is not None: P.__getattr__=lambda self,name: getattr(object.__getattribute__(self,"_real"), name)
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from datasets import load_dataset
MAX_LEN=3072
log("load model ONCE (16GB)...")
model, tok = FastLanguageModel.from_pretrained(model_name="/tmp/gemma4_base", max_seq_length=MAX_LEN, dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False, local_files_only=True)
model = FastLanguageModel.get_peft_model(model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth", use_rslora=True, random_state=0)
bad=["vision_tower","audio_tower","embed_vision","embed_audio","vision_model","audio_model"]
for n,p in model.named_parameters():
    if p.requires_grad and any(k in n for k in bad): p.requires_grad_(False)
log("model ready. sweeping batches...")
ds=load_dataset("json", data_files={"train":sorted(glob.glob("/tmp/ds_pc_msg/train/*.jsonl"))})
def sc(**kw):
    al=set(inspect.signature(SFTConfig.__init__).parameters); return SFTConfig(**{k:v for k,v in kw.items() if k in al})
def run_batch(B, steps=12):
    torch.cuda.reset_peak_memory_stats()
    args=sc(output_dir="/tmp/uns_sweep", per_device_train_batch_size=B, gradient_accumulation_steps=1,
        learning_rate=7e-5, warmup_ratio=0.0, max_steps=steps, lr_scheduler_type="constant", optim="adamw_torch_fused",
        logging_steps=100, save_steps=10**9, report_to="none", max_length=MAX_LEN, packing=False,
        completion_only_loss=True, bf16=True, include_num_input_tokens_seen=True, dataloader_num_workers=2, ddp_find_unused_parameters=False)
    kw=dict(model=model,args=args,train_dataset=ds["train"])
    if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters: kw["processing_class"]=tok
    else: kw["tokenizer"]=tok
    tr=SFTTrainer(**kw)
    t0=time.time(); res=tr.train(); dt=time.time()-t0
    ntok=getattr(tr.state,"num_input_tokens_seen",0); mem=torch.cuda.max_memory_allocated()/1e9
    tps=round(ntok/dt,0) if (ntok and dt) else None
    log(f"RESULT per_gpu_batch={B}: tok/s/rank={tps} (~{(tps or 0)*2:.0f} aggregate) vram={mem:.1f}GB s/step={dt/steps:.2f}")
for B in [12,16,24]:
    try: run_batch(B)
    except Exception as e:
        log(f"per_gpu_batch={B} FAILED: {repr(e)[:140]}"); torch.cuda.empty_cache()
log("sweep done")
'''
import os, subprocess, sys, time
with open("/tmp/unsloth_sweep.py","w") as f: f.write(SWEEP)
env=os.environ.copy(); env.update({"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","PYTHONUNBUFFERED":"1","CUDA_VISIBLE_DEVICES":"0,1","NCCL_DEBUG":"WARN"})
print(time.strftime('[%H:%M:%S]'),"launch torchrun nproc=2 sweep (one load ~4min, then batches stream)...",flush=True)
proc=subprocess.Popen([sys.executable,"-u","-m","torch.distributed.run","--standalone","--nproc_per_node=2","/tmp/unsloth_sweep.py"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    s=line.rstrip()
    if s.startswith("[") or "RESULT" in s or "FAILED" in s or "OutOfMemory" in s or "Error" in s or "Traceback" in s: print(s, flush=True)
proc.wait(); print("[done] rc=", proc.returncode)

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=index,name,memory.total,driver_version","--format=csv"], capture_output=True, text=True).stdout)
print("torch:", torch.__version__, "| cuda avail:", torch.cuda.is_available(), "| device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    cap = torch.cuda.get_device_capability(i)
    print(f"  GPU{i}: {torch.cuda.get_device_name(i)} sm_{cap[0]}{cap[1]}")

In [ ]:
import os, glob
log=open('/tmp/ddp_log.txt').read()
i=log.find('"engine"'); print('SUMMARY:\n', log[log.rfind('{',0,i):log.find('}',i)+1] if i>0 else 'n/a')
out='/tmp/gemma4_google_unsloth_2gpu_lora'
print('\nadapter files:', sorted(os.path.basename(p) for p in glob.glob(out+'/*')))

In [ ]:
# Patch packaged DDP script (inner text tokenizer + enable grad-checkpointing) + run 2-GPU DDP smoke 20
import os, sys, glob, subprocess, time
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
PACK="@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605"
DS=Path("/tmp/gemma4_quality4_dataset"); DS.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(DS)+"/train.jsonl"):
    session.sql(f"GET {PACK}/dataset_quality4 file://{DS} PATTERN='.*[.]jsonl' PARALLEL=16").collect()
TRAIN=str(DS/"train.jsonl")
SRC="/tmp/google_gemma4_unsloth_pack/scripts/train_unsloth_gemma4_2gpu_ddp.py"
if not os.path.exists(SRC): SRC="/tmp/pack_meta/scripts/train_unsloth_gemma4_2gpu_ddp.py"
src=open(SRC).read()
old="if tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token"
fix=old+"\n# PATCH: Gemma4 FastLanguageModel returns multimodal Processor -> use inner text tokenizer\nif hasattr(tokenizer, \"tokenizer\"):\n    tokenizer = tokenizer.tokenizer\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token"
assert old in src, "anchor1 not found"
src=src.replace(old,fix,1)
oldgc='use_gradient_checkpointing=False'
assert oldgc in src, "anchor2 not found"
src=src.replace(oldgc,'use_gradient_checkpointing="unsloth"',1)
SCRIPT="/tmp/train_ddp_patched.py"; open(SCRIPT,"w").write(src)
print("patched -> grad ckpt unsloth + inner tokenizer")
env=os.environ.copy()
env.update({"PYTHONPATH":"/tmp/hubfix:"+env.get("PYTHONPATH",""),"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1",
  "PYTHONUNBUFFERED":"1","CUDA_VISIBLE_DEVICES":"0,1","NCCL_DEBUG":"WARN","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
  "MODEL_DIR":"/tmp/google_gemma4_e4b_model","DATA_PATH":TRAIN,"OUT_DIR":"/tmp/gemma4_google_unsloth_2gpu_lora",
  "MAX_LEN":"3072","BATCH":"8","GRAD_ACC":"1","LORA_R":"32","MAX_STEPS":"20"})
print(time.strftime('[%H:%M:%S]'),"launch torchrun nproc=2 (load ~6min then train)...",flush=True)
lines=[]
proc=subprocess.Popen([sys.executable,"-u","-m","torch.distributed.run","--standalone","--nproc_per_node=2",SCRIPT],
  env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    lines.append(line); s=line.rstrip()
    if s.startswith("[") or any(k in s for k in ("SUMMARY","froze","START DDP","'loss'","engine","out_dir","OutOfMemory","Error","Traceback")):
        print(s, flush=True)
proc.wait(); open("/tmp/ddp_log.txt","w").write("".join(lines))
print("[done] rc=", proc.returncode)
if proc.returncode!=0: print("".join(lines[-40:]))

In [ ]:
# [SETUP] Rebuild env on fresh /tmp (after suspend): installer + hub fix + drop datasets5 + pyarrow guard
import runpy, os, sys, subprocess, glob, time
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
def log(m): print(time.strftime('[%H:%M:%S] ')+m, flush=True)
t0=time.time()
PACK='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605'
# 1) get installer script
os.makedirs('/tmp/pack_meta', exist_ok=True)
session.file.get(f'{PACK}/meta','/tmp/pack_meta/meta')
log('got installer; running (GET pack ~16GB + offline pip install + reconstruct model)...')
try:
    runpy.run_path('/tmp/pack_meta/meta/SNOWFLAKE_INSTALL_GOOGLE_GEMMA4_UNSLOTH_FROM_STAGE.py', run_name='__main__')
except Exception as e:
    log('installer probe note (expected pre-hubfix): '+repr(e)[:160])
W='/tmp/google_gemma4_unsloth_pack/wheels'
# 2) hub fix
log('installing huggingface_hub 1.18.0 + deps (offline)...')
r=subprocess.run([sys.executable,'-m','pip','install','--no-index',f'--find-links={W}','--upgrade','--no-deps',
    'huggingface_hub','filelock','fsspec','packaging','pyyaml','requests','tqdm','typing_extensions'],capture_output=True,text=True)
log('hub install rc='+str(r.returncode))
# 3) drop datasets 5.0.0 -> use runtime datasets 4.0.0 (pyarrow compat)
r=subprocess.run([sys.executable,'-m','pip','uninstall','-y','datasets'],capture_output=True,text=True)
log('datasets5 uninstall rc='+str(r.returncode))
# 4) pyarrow extension-type guard sitecustomize (for all subprocs incl torchrun)
os.makedirs('/tmp/hubfix', exist_ok=True)
open('/tmp/hubfix/sitecustomize.py','w').write(
'try:\n import pyarrow as _pa\n _o=_pa.register_extension_type\n def _s(*a,**k):\n  try:\n   return _o(*a,**k)\n  except _pa.lib.ArrowKeyError:\n   return None\n _pa.register_extension_type=_s\nexcept Exception:\n pass\n')
md=os.path.exists('/tmp/google_gemma4_e4b_model/model.safetensors')
log(f'DONE setup in {time.time()-t0:.0f}s | model.safetensors present={md}')

In [ ]:
# [SWEEP] Maximize 2-GPU Blackwell utilization: load once, grid MAX_LEN x BATCH, measure tok/s + peak VRAM (real-time stream)
import os, sys, subprocess, time
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
PACK='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605'
DS=Path('/tmp/gemma4_quality4_dataset'); DS.mkdir(parents=True, exist_ok=True)
import glob
if not glob.glob(str(DS)+'/train.jsonl'):
    session.sql(f"GET {PACK}/dataset_quality4 file://{DS} PATTERN='.*[.]jsonl' PARALLEL=16").collect()
TRAIN=str(DS/'train.jsonl')
SCRIPT=r'''
import os,sys,json,time,gc
os.environ.setdefault("HF_HUB_OFFLINE","1");os.environ.setdefault("TRANSFORMERS_OFFLINE","1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True");os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
import torch
LR=int(os.environ.get("LOCAL_RANK","0"));WORLD=int(os.environ.get("WORLD_SIZE","1"));R0=(LR==0)
torch.cuda.set_device(LR)
def log(m):
    if R0:print(time.strftime("[%H:%M:%S] ")+m,flush=True)
import unsloth
import unsloth_zoo.temporary_patches.gemma4 as g4
if hasattr(g4,"_Gemma4KVSharedSafeProxy"):
    g4._Gemma4KVSharedSafeProxy.__getattr__=lambda self,name:getattr(object.__getattribute__(self,"_real"),name)
from unsloth import FastLanguageModel
from transformers import Trainer,TrainingArguments
from datasets import Dataset
MODEL_DIR="/tmp/google_gemma4_e4b_model";DATA=os.environ["DATA_PATH"];CAP=4096
log("loading model once (cap=%d, ~3-6min)..."%CAP)
model,tok=FastLanguageModel.from_pretrained(model_name=MODEL_DIR,max_seq_length=CAP,load_in_4bit=False,load_in_16bit=True,full_finetuning=False,local_files_only=True)
if hasattr(tok,"tokenizer"):tok=tok.tokenizer
if tok.pad_token is None:tok.pad_token=tok.eos_token
model=FastLanguageModel.get_peft_model(model,r=32,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=64,lora_dropout=0.0,bias="none",use_rslora=True,use_gradient_checkpointing="unsloth")
for n,p in model.named_parameters():
    if any(k in n for k in ["vision_tower","audio_tower","embed_vision","embed_audio"]):p.requires_grad_(False)
log("model ready trainable=%.1fM | GPUs=%d"%(sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6,WORLD))
rows=[]
with open(DATA) as f:
    for line in f:
        try:
            t=json.loads(line).get("text") or ""
            if t:rows.append(t)
        except:pass
log("dataset rows=%d"%len(rows))
class Coll:
    def __init__(self,tok):self.t=tok
    def __call__(self,feats):
        m=max(len(f["input_ids"]) for f in feats);pid=self.t.pad_token_id or 0
        ii=[];ll=[];am=[]
        for f in feats:
            ids=f["input_ids"];lab=f["labels"];pad=m-len(ids)
            ii.append(ids+[pid]*pad);ll.append(lab+[-100]*pad);am.append([1]*len(ids)+[0]*pad)
        return {"input_ids":torch.tensor(ii),"labels":torch.tensor(ll),"attention_mask":torch.tensor(am)}
coll=Coll(tok)
dscache={}
def make_ds(maxlen):
    if maxlen in dscache:return dscache[maxlen]
    log("  tokenizing dataset for maxlen=%d ..."%maxlen)
    def tk(b):
        out=tok(b["text"],add_special_tokens=False,truncation=True,max_length=maxlen)
        out["labels"]=[x[:] for x in out["input_ids"]]
        return out
    d=Dataset.from_dict({"text":rows}).map(tk,batched=True,remove_columns=["text"])
    dscache[maxlen]=d;return d
GRID=[(3072,8),(2048,16),(2048,24),(3072,16),(4096,12),(4096,16),(2048,32),(3072,24),(3072,32)]
K=int(os.environ.get("K","15"))
results=[]
log("==== START SWEEP: %d configs, %d steps each (eff_batch=batch x %d GPUs) ===="%(len(GRID),K,WORLD))
for i,(ML,B) in enumerate(GRID):
    ds=make_ds(ML)
    log("CONFIG %d/%d  maxlen=%-4d batch=%-2d  eff_batch=%-3d  running %d steps..."%(i+1,len(GRID),ML,B,B*WORLD,K))
    torch.cuda.reset_peak_memory_stats();torch.cuda.empty_cache()
    args=TrainingArguments(output_dir="/tmp/sweep_out",per_device_train_batch_size=B,gradient_accumulation_steps=1,max_steps=K,logging_steps=K+1,warmup_steps=2,learning_rate=7e-5,optim="adamw_torch_fused",bf16=True,report_to="none",save_strategy="no",dataloader_num_workers=2,include_num_input_tokens_seen=True,ddp_find_unused_parameters=False,gradient_checkpointing=False)
    tr=Trainer(model=model,args=args,train_dataset=ds,data_collator=coll)
    ok=True;t0=time.time();res=None
    try:
        res=tr.train()
    except torch.cuda.OutOfMemoryError:
        ok=False
    dt=time.time()-t0
    if ok:
        tokr=getattr(tr.state,"num_input_tokens_seen",0);agg=tokr*WORLD;vram=torch.cuda.max_memory_allocated()/1e9;tps=agg/dt if dt else 0
        rec={"maxlen":ML,"batch":B,"eff_batch":B*WORLD,"tok_s_2gpu":round(tps),"peak_vram_gb":round(vram,1),"loss":round(float(res.training_loss),3),"sec":round(dt,1)}
        results.append(rec)
        log("   -> OK  tok/s(2gpu)=%-6d  peak_vram=%-5.1fGB  loss=%.3f  (%.1fs)"%(tps,vram,float(res.training_loss),dt))
    else:
        log("   -> OOM at maxlen=%d batch=%d  (stop pushing higher; keeping prior results)"%(ML,B))
    del tr;gc.collect();torch.cuda.empty_cache()
    if not ok:break
if R0:
    results.sort(key=lambda r:r["tok_s_2gpu"],reverse=True)
    print("\n===== SWEEP RANKED (highest throughput first) =====",flush=True)
    for r in results:print(json.dumps(r),flush=True)
    if results:print("\nBEST_CONFIG "+json.dumps(results[0]),flush=True)
'''
open('/tmp/sweep_ddp.py','w').write(SCRIPT)
env=os.environ.copy()
env.update({'PYTHONPATH':'/tmp/hubfix:'+env.get('PYTHONPATH',''),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0,1','NCCL_DEBUG':'WARN','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','DATA_PATH':TRAIN,'K':'15'})
print(time.strftime('[%H:%M:%S]'),'launch sweep torchrun nproc=2 (model load ~3-6min, then each config streams)...',flush=True)
proc=subprocess.Popen([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/sweep_ddp.py'],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
for line in proc.stdout:
    lines.append(line);s=line.rstrip()
    if s.startswith('[') or any(k in s for k in ('CONFIG','->','RANKED','BEST_CONFIG','OOM','maxlen','Error','Traceback','OutOfMemory')):
        print(s,flush=True)
proc.wait();open('/tmp/sweep_log.txt','w').write(''.join(lines))
print('[done] rc=',proc.returncode)
if proc.returncode!=0:print(''.join(lines[-30:]))

In [ ]:
# [TRAIN-BEST] Sustained 2-GPU DDP @ maxlen=3072 batch=24 (eff=48) | runs until YOU interrupt | autosave every 100 steps
# Real-time timestamped stream: per-step loss + tok/s(2gpu) + peak VRAM. Full dataset (text + messages).
import os, sys, subprocess, time, glob
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
PACK='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605'
DS=Path('/tmp/gemma4_quality4_dataset'); DS.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(DS)+'/train.jsonl'):
    session.sql(f"GET {PACK}/dataset_quality4 file://{DS} PATTERN='.*[.]jsonl' PARALLEL=16").collect()
TRAIN=str(DS/'train.jsonl')
SCRIPT=r'''
import os,sys,json,time,gc
os.environ.setdefault("HF_HUB_OFFLINE","1");os.environ.setdefault("TRANSFORMERS_OFFLINE","1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True");os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
import torch
LR=int(os.environ.get("LOCAL_RANK","0"));WORLD=int(os.environ.get("WORLD_SIZE","1"));R0=(LR==0)
torch.cuda.set_device(LR)
def log(m):
    if R0:print(time.strftime("[%H:%M:%S] ")+m,flush=True)
import unsloth
import unsloth_zoo.temporary_patches.gemma4 as g4
if hasattr(g4,"_Gemma4KVSharedSafeProxy"):
    g4._Gemma4KVSharedSafeProxy.__getattr__=lambda self,name:getattr(object.__getattribute__(self,"_real"),name)
from unsloth import FastLanguageModel
from transformers import Trainer,TrainingArguments,TrainerCallback
from datasets import Dataset
MODEL_DIR="/tmp/google_gemma4_e4b_model";DATA=os.environ["DATA_PATH"]
ML=int(os.environ.get("MAX_LEN","3072"));B=int(os.environ.get("BATCH","24"));STEPS=int(os.environ.get("MAX_STEPS","1000000"))
OUT=os.environ.get("OUT_DIR","/tmp/gemma4_best_run")
SYS=os.environ.get("SYSTEM_PROMPT","Ban la AI chuyen sinh seed, mutate fuzz script va danh gia oracle cho PyTorch/TensorFlow.")
log("loading model once (cap=%d, ~3-6min)..."%ML)
model,tok=FastLanguageModel.from_pretrained(model_name=MODEL_DIR,max_seq_length=ML,load_in_4bit=False,load_in_16bit=True,full_finetuning=False,local_files_only=True)
if hasattr(tok,"tokenizer"):tok=tok.tokenizer
if tok.pad_token is None:tok.pad_token=tok.eos_token
model=FastLanguageModel.get_peft_model(model,r=32,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=64,lora_dropout=0.0,bias="none",use_rslora=True,use_gradient_checkpointing="unsloth")
for n,p in model.named_parameters():
    if any(k in n for k in ["vision_tower","audio_tower","embed_vision","embed_audio"]):p.requires_grad_(False)
log("trainable=%.1fM | GPUs=%d | maxlen=%d batch=%d eff_batch=%d"%(sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6,WORLD,ML,B,B*WORLD))
# build full-text from text OR messages OR prompt/completion
def row_text(ex):
    t=ex.get("text")
    if t:return str(t)
    msgs=ex.get("messages")
    if msgs:
        if isinstance(msgs,str):
            try:msgs=json.loads(msgs)
            except:return ""
        try:return tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=False)
        except:return ""
    p=ex.get("prompt") or ex.get("instruction") or ex.get("input");c=ex.get("completion") or ex.get("output") or ex.get("response")
    if p is not None and c is not None:
        return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":str(p)},{"role":"assistant","content":str(c)}],tokenize=False,add_generation_prompt=False)
    return ""
texts=[]
with open(DATA) as f:
    for line in f:
        try:
            tt=row_text(json.loads(line))
            if tt:texts.append(tt)
        except:pass
log("dataset rows=%d"%len(texts))
def tk(b):
    out=tok(b["text"],add_special_tokens=False,truncation=True,max_length=ML);out["labels"]=[x[:] for x in out["input_ids"]];return out
ds=Dataset.from_dict({"text":texts}).map(tk,batched=True,remove_columns=["text"])
class Coll:
    def __init__(self,t):self.t=t
    def __call__(self,feats):
        m=max(len(f["input_ids"]) for f in feats);pid=self.t.pad_token_id or 0
        ii=[];ll=[];am=[]
        for f in feats:
            ids=f["input_ids"];lab=f["labels"];pad=m-len(ids)
            ii.append(ids+[pid]*pad);ll.append(lab+[-100]*pad);am.append([1]*len(ids)+[0]*pad)
        return {"input_ids":torch.tensor(ii),"labels":torch.tensor(ll),"attention_mask":torch.tensor(am)}
class Stream(TrainerCallback):
    def __init__(self):self.lt=time.time();self.ltok=0
    def on_log(self,a,s,c,logs=None,**k):
        if not (R0 and logs and "loss" in logs):return
        now=time.time();tok_now=getattr(s,"num_input_tokens_seen",0)
        dtok=(tok_now-self.ltok)*WORLD;dt=now-self.lt;tps=dtok/dt if dt>0 else 0
        self.lt=now;self.ltok=tok_now
        vram=torch.cuda.max_memory_allocated()/1e9
        log("step %-6d loss=%-7s lr=%-9s tok/s(2gpu)=%-6d peak_vram=%.1fGB"%(s.global_step,str(logs.get("loss")),str(logs.get("learning_rate")),tps,vram))
args=TrainingArguments(output_dir=OUT,per_device_train_batch_size=B,gradient_accumulation_steps=1,max_steps=STEPS,learning_rate=7e-5,warmup_ratio=0.02,lr_scheduler_type="cosine",optim="adamw_torch_fused",bf16=True,logging_steps=1,save_steps=100,save_total_limit=3,report_to="none",dataloader_num_workers=2,include_num_input_tokens_seen=True,ddp_find_unused_parameters=False,gradient_checkpointing=False)
tr=Trainer(model=model,args=args,train_dataset=ds,data_collator=Coll(tok),callbacks=[Stream()])
log("START sustained training (Ctrl-Stop / interrupt to stop; autosave every 100 steps -> %s)"%OUT)
t0=time.time()
try:
    tr.train()
except KeyboardInterrupt:
    log("INTERRUPTED by user")
finally:
    if R0:
        tr.save_model(OUT);tok.save_pretrained(OUT)
        log("saved adapter -> %s | total %.0fs | steps=%d"%(OUT,time.time()-t0,tr.state.global_step))
'''
open('/tmp/train_best.py','w').write(SCRIPT)
env=os.environ.copy()
env.update({'PYTHONPATH':'/tmp/hubfix:'+env.get('PYTHONPATH',''),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0,1','NCCL_DEBUG':'WARN','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','DATA_PATH':TRAIN,'MAX_LEN':'3072','BATCH':'24','MAX_STEPS':'1000000','OUT_DIR':'/tmp/gemma4_best_run'}); env.pop('RANK',None)
print(time.strftime('[%H:%M:%S]'),'launch sustained torchrun nproc=2 (model load ~3-6min, then streams every step; INTERRUPT to stop)...',flush=True)
proc=subprocess.Popen([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/train_best.py'],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
try:
    for line in proc.stdout:
        lines.append(line);s=line.rstrip()
        if s.startswith('[') or any(k in s for k in ('step ','START','saved','INTERRUPTED','Error','Traceback','OutOfMemory')):
            print(s,flush=True)
except KeyboardInterrupt:
    proc.terminate();print('[cell interrupted -> signaling training to stop & save]',flush=True)
proc.wait();open('/tmp/train_best_log.txt','w').write(''.join(lines))
print('[done] rc=',proc.returncode)

In [ ]:
# [WORKER-START] Persistent 2-GPU DDP worker: loads model ONCE, then serves jobs from a file queue (no reload per run)
import os, sys, subprocess, time, glob, signal
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'; os.makedirs(JOBS, exist_ok=True)
# kill previous worker if any
if 'WORKER_PROC' in globals() and globals()['WORKER_PROC'] and globals()['WORKER_PROC'].poll() is None:
    try: globals()['WORKER_PROC'].terminate(); time.sleep(3)
    except: pass
for f in glob.glob(CTL+'/*.log')+glob.glob(CTL+'/*.done')+glob.glob(JOBS+'/*')+glob.glob(CTL+'/STOP'): 
    try: os.remove(f)
    except: pass
PACK='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605'
DS=Path('/tmp/gemma4_quality4_dataset'); DS.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(DS)+'/train.jsonl'):
    session.sql(f"GET {PACK}/dataset_quality4 file://{DS} PATTERN='.*[.]jsonl' PARALLEL=16").collect()
TRAIN=str(DS/'train.jsonl')
SERVER=r'''
import os,sys,json,time,glob,gc
os.environ.setdefault("HF_HUB_OFFLINE","1");os.environ.setdefault("TRANSFORMERS_OFFLINE","1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True");os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
import torch, torch.distributed as dist
LR=int(os.environ.get("LOCAL_RANK","0"));WORLD=int(os.environ.get("WORLD_SIZE","1"));R0=(LR==0)
torch.cuda.set_device(LR)
CTL="/tmp/ddp_ctl";JOBS=CTL+"/jobs";DATA=os.environ["DATA_PATH"]
def slog(m):
    if R0:
        open(CTL+"/server.log","a").write(time.strftime("[%H:%M:%S] ")+m+"\n")
dist.init_process_group(backend="nccl")
import unsloth
import unsloth_zoo.temporary_patches.gemma4 as g4
if hasattr(g4,"_Gemma4KVSharedSafeProxy"):
    g4._Gemma4KVSharedSafeProxy.__getattr__=lambda self,name:getattr(object.__getattribute__(self,"_real"),name)
from unsloth import FastLanguageModel
from transformers import Trainer,TrainingArguments,TrainerCallback
from datasets import Dataset
MODEL_DIR="/tmp/google_gemma4_e4b_model";CAP=int(os.environ.get("CAP","4096"))
slog("loading model once (cap=%d)..."%CAP)
model,tok=FastLanguageModel.from_pretrained(model_name=MODEL_DIR,max_seq_length=CAP,load_in_4bit=False,load_in_16bit=True,full_finetuning=False,local_files_only=True)
if hasattr(tok,"tokenizer"):tok=tok.tokenizer
if tok.pad_token is None:tok.pad_token=tok.eos_token
model=FastLanguageModel.get_peft_model(model,r=32,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=64,lora_dropout=0.0,bias="none",use_rslora=True,use_gradient_checkpointing="unsloth")
for n,p in model.named_parameters():
    if any(k in n for k in ["vision_tower","audio_tower","embed_vision","embed_audio"]):p.requires_grad_(False)
SYS="Ban la AI chuyen sinh seed, mutate fuzz script va danh gia oracle cho PyTorch/TensorFlow."
def row_text(ex):
    t=ex.get("text")
    if t:return str(t)
    m=ex.get("messages")
    if m:
        if isinstance(m,str):
            try:m=json.loads(m)
            except:return ""
        try:return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=False)
        except:return ""
    p=ex.get("prompt") or ex.get("instruction") or ex.get("input");c=ex.get("completion") or ex.get("output") or ex.get("response")
    if p is not None and c is not None:
        try:return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":str(p)},{"role":"assistant","content":str(c)}],tokenize=False,add_generation_prompt=False)
        except:return ""
    return ""
texts=[]
for line in open(DATA):
    try:
        tt=row_text(json.loads(line))
        if tt:texts.append(tt)
    except:pass
slog("dataset rows=%d trainable=%.1fM"%(len(texts),sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6))
dscache={}
def make_ds(ML):
    if ML in dscache:return dscache[ML]
    def tk(b):
        o=tok(b["text"],add_special_tokens=False,truncation=True,max_length=ML);o["labels"]=[x[:] for x in o["input_ids"]];return o
    d=Dataset.from_dict({"text":texts}).map(tk,batched=True,remove_columns=["text"]);dscache[ML]=d;return d
class Coll:
    def __init__(self,t):self.t=t
    def __call__(self,feats):
        m=max(len(f["input_ids"]) for f in feats);pid=self.t.pad_token_id or 0
        ii=[];ll=[];am=[]
        for f in feats:
            ids=f["input_ids"];lab=f["labels"];pad=m-len(ids)
            ii.append(ids+[pid]*pad);ll.append(lab+[-100]*pad);am.append([1]*len(ids)+[0]*pad)
        return {"input_ids":torch.tensor(ii),"labels":torch.tensor(ll),"attention_mask":torch.tensor(am)}
coll=Coll(tok)
class CB(TrainerCallback):
    def __init__(self,lf):self.lf=lf;self.lt=time.time();self.lk=0
    def on_step_end(self,a,s,c,**k):
        if os.path.exists(CTL+"/STOP"):c.should_training_stop=True
    def on_log(self,a,s,c,logs=None,**k):
        if not(R0 and logs and "loss" in logs):return
        now=time.time();tn=getattr(s,"num_input_tokens_seen",0);d=(tn-self.lk)*WORLD;dt=now-self.lt;tps=d/dt if dt>0 else 0
        self.lt=now;self.lk=tn;v=torch.cuda.max_memory_allocated()/1e9
        open(self.lf,"a").write(time.strftime("[%H:%M:%S] ")+"step %-6d loss=%-7s lr=%-9s tok/s(2gpu)=%-6d vram=%.1fGB\n"%(s.global_step,str(logs.get("loss")),str(logs.get("learning_rate")),tps,v))
def run_job(job):
    jid=job["jobid"];ML=int(job["maxlen"]);B=int(job["batch"]);STEPS=int(job["steps"]);LRr=float(job.get("lr",7e-5));SAVE=bool(job.get("save",False));OUT=job.get("out","/tmp/gemma4_best_run")
    lf=CTL+"/"+jid+".log";ds=make_ds(ML)
    if R0:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"START job=%s maxlen=%d batch=%d eff=%d steps=%d\n"%(jid,ML,B,B*WORLD,STEPS))
    torch.cuda.reset_peak_memory_stats()
    args=TrainingArguments(output_dir=OUT,per_device_train_batch_size=B,gradient_accumulation_steps=1,max_steps=STEPS,learning_rate=LRr,warmup_ratio=0.02,lr_scheduler_type="cosine",optim="adamw_torch_fused",bf16=True,logging_steps=1,save_strategy="no",report_to="none",dataloader_num_workers=2,include_num_input_tokens_seen=True,ddp_find_unused_parameters=False,gradient_checkpointing=False)
    tr=Trainer(model=model,args=args,train_dataset=ds,data_collator=coll,callbacks=[CB(lf)])
    t0=time.time();err=None
    try:res=tr.train()
    except Exception as e:err=repr(e)[:300]
    dt=time.time()-t0
    if R0:
        if err is None:
            if SAVE:tr.save_model(OUT);tok.save_pretrained(OUT)
            tn=getattr(tr.state,"num_input_tokens_seen",0);v=torch.cuda.max_memory_allocated()/1e9
            open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"DONE "+json.dumps({"steps":tr.state.global_step,"sec":round(dt,1),"loss":round(float(res.training_loss),4),"tok_s_2gpu":round(tn*WORLD/dt) if dt else 0,"peak_vram_gb":round(v,1),"saved":SAVE,"out":OUT})+"\n")
        else:
            open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"ERROR "+err+"\n")
    del tr;gc.collect();torch.cuda.empty_cache()
    try:
        if R0 and os.path.exists(CTL+"/STOP"):os.remove(CTL+"/STOP")
    except:pass
    if R0:open(CTL+"/"+jid+".done","w").write("1")
slog("READY")
processed=set()
while True:
    nxt=None
    for fp in sorted(glob.glob(JOBS+"/*.json")):
        if fp not in processed:nxt=fp;break
    if nxt is None:
        time.sleep(1);continue
    processed.add(nxt)
    try:job=json.load(open(nxt))
    except:continue
    dist.barrier()
    if job.get("cmd")=="shutdown":
        slog("shutdown");break
    run_job(job)
    dist.barrier()
'''
open('/tmp/ddp_server.py','w').write(SERVER)
open(CTL+'/server.log','w').write('')
env=os.environ.copy()
env.update({'PYTHONPATH':'/tmp/hubfix:'+env.get('PYTHONPATH',''),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0,1','NCCL_DEBUG':'WARN','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','DATA_PATH':TRAIN,'CAP':'4096'}); env.pop('RANK',None)
proc=subprocess.Popen([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/ddp_server.py'],env=env,stdout=open(CTL+'/stdout.log','w'),stderr=subprocess.STDOUT)
globals()['WORKER_PROC']=proc
print(time.strftime('[%H:%M:%S]'),'worker launching (one-time model load ~3-6min)... waiting for READY',flush=True)
t0=time.time();ready=False
while time.time()-t0<720:
    if proc.poll() is not None:
        print('[worker EXITED early] tail stdout:',flush=True);print(open(CTL+'/stdout.log').read()[-1500:]);break
    log=open(CTL+'/server.log').read()
    if 'READY' in log:
        print(open(CTL+'/server.log').read().strip(),flush=True);print(time.strftime('[%H:%M:%S]'),'WORKER READY -> use [WORKER-RUN] to submit jobs (no reload)',flush=True);ready=True;break
    time.sleep(3)
if not ready and proc.poll() is None:print('[still loading; re-run a quick check of /tmp/ddp_ctl/server.log]',flush=True)

In [ ]:
# [WORKER-RUN] Submit a training job to the resident worker (NO model reload). Edit params below + re-run anytime.
# For "train until I stop": set JOB_STEPS huge and use [WORKER-STOP] to end gracefully (autosaves if JOB_SAVE=True).
import json, os, time
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'; os.makedirs(JOBS, exist_ok=True)
# ===== EDIT THESE =====
JOB_MAXLEN = 3072
JOB_BATCH  = 24
JOB_STEPS  = 50          # set large (e.g. 1000000) to run until [WORKER-STOP]
JOB_LR     = 7e-5
JOB_SAVE   = False       # True -> save adapter to JOB_OUT when job ends
JOB_OUT    = '/tmp/gemma4_best_run'
# ======================
assert ('WORKER_PROC' in globals() and globals()['WORKER_PROC'].poll() is None), 'Worker not running -> run [WORKER-START] first'
jid='job_%d'%int(time.time())
job={'jobid':jid,'maxlen':JOB_MAXLEN,'batch':JOB_BATCH,'steps':JOB_STEPS,'lr':JOB_LR,'save':JOB_SAVE,'out':JOB_OUT}
lf=CTL+'/'+jid+'.log'; donef=CTL+'/'+jid+'.done'
tmp=JOBS+'/'+jid+'.json.tmp'; open(tmp,'w').write(json.dumps(job)); os.rename(tmp,JOBS+'/'+jid+'.json')
print(time.strftime('[%H:%M:%S]'),'submitted',jid,job,'(streaming; no reload)',flush=True)
seen=0
try:
    while not os.path.exists(donef):
        if os.path.exists(lf):
            d=open(lf).read()
            if len(d)>seen:print(d[seen:],end='',flush=True);seen=len(d)
        time.sleep(0.5)
    if os.path.exists(lf):
        d=open(lf).read()
        if len(d)>seen:print(d[seen:],end='',flush=True)
    print('\n[job finished]',flush=True)
except KeyboardInterrupt:
    print('\n[stopped tailing - job keeps running in worker; use WORKER-STOP to end it]',flush=True)

In [ ]:
# [WORKER-STOP] Gracefully stop the CURRENTLY running job (it will finish current step, save if JOB_SAVE=True). Worker stays alive for next job.
import os, time
CTL='/tmp/ddp_ctl'
open(CTL+'/STOP','w').write('1')
print(time.strftime('[%H:%M:%S]'),'STOP flag set -> current job will stop at next step boundary (worker stays resident).',flush=True)
# To fully shut down the worker (free GPU), uncomment:
# import json
# open(CTL+'/jobs/zzzz_shutdown.json','w').write(json.dumps({'cmd':'shutdown','jobid':'shutdown'}))
# print('shutdown job queued')

In [ ]:
# [WORKER-START v2] Upgraded persistent 2-GPU worker: LoRA r=64/alpha=128 + per-job group_by_length / neftune / weight_decay / epochs
import os, sys, subprocess, time, glob
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'; os.makedirs(JOBS, exist_ok=True)
if 'WORKER_PROC' in globals() and globals()['WORKER_PROC'] and globals()['WORKER_PROC'].poll() is None:
    print('stopping previous worker...'); 
    try: globals()['WORKER_PROC'].terminate(); time.sleep(4)
    except: pass
for f in glob.glob(CTL+'/*.log')+glob.glob(CTL+'/*.done')+glob.glob(JOBS+'/*')+glob.glob(CTL+'/STOP'):
    try: os.remove(f)
    except: pass
PACK='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605'
DS=Path('/tmp/gemma4_quality4_dataset'); DS.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(DS)+'/train.jsonl'):
    session.sql(f"GET {PACK}/dataset_quality4 file://{DS} PATTERN='.*[.]jsonl' PARALLEL=16").collect()
TRAIN=str(DS/'train.jsonl')
SERVER=r'''
import os,sys,json,time,glob,gc
os.environ.setdefault("HF_HUB_OFFLINE","1");os.environ.setdefault("TRANSFORMERS_OFFLINE","1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True");os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
import torch, torch.distributed as dist
LR=int(os.environ.get("LOCAL_RANK","0"));WORLD=int(os.environ.get("WORLD_SIZE","1"));R0=(LR==0)
torch.cuda.set_device(LR)
CTL="/tmp/ddp_ctl";JOBS=CTL+"/jobs";DATA=os.environ["DATA_PATH"]
def slog(m):
    if R0:open(CTL+"/server.log","a").write(time.strftime("[%H:%M:%S] ")+m+"\n")
dist.init_process_group(backend="nccl")
import unsloth
import unsloth_zoo.temporary_patches.gemma4 as g4
if hasattr(g4,"_Gemma4KVSharedSafeProxy"):
    g4._Gemma4KVSharedSafeProxy.__getattr__=lambda self,name:getattr(object.__getattribute__(self,"_real"),name)
from unsloth import FastLanguageModel
from transformers import Trainer,TrainingArguments,TrainerCallback
from datasets import Dataset
MODEL_DIR="/tmp/google_gemma4_e4b_model";CAP=int(os.environ.get("CAP","4096"))
RR=int(os.environ.get("LORA_R","64"));AA=int(os.environ.get("LORA_ALPHA","128"))
slog("loading model once (cap=%d, LoRA r=%d alpha=%d)..."%(CAP,RR,AA))
model,tok=FastLanguageModel.from_pretrained(model_name=MODEL_DIR,max_seq_length=CAP,load_in_4bit=False,load_in_16bit=True,full_finetuning=False,local_files_only=True)
if hasattr(tok,"tokenizer"):tok=tok.tokenizer
if tok.pad_token is None:tok.pad_token=tok.eos_token
model=FastLanguageModel.get_peft_model(model,r=RR,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=AA,lora_dropout=0.0,bias="none",use_rslora=True,use_gradient_checkpointing="unsloth")
for n,p in model.named_parameters():
    if any(k in n for k in ["vision_tower","audio_tower","embed_vision","embed_audio"]):p.requires_grad_(False)
SYS="Ban la AI chuyen sinh seed, mutate fuzz script va danh gia oracle cho PyTorch/TensorFlow."
def row_text(ex):
    t=ex.get("text")
    if t:return str(t)
    m=ex.get("messages")
    if m:
        if isinstance(m,str):
            try:m=json.loads(m)
            except:return ""
        try:return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=False)
        except:return ""
    p=ex.get("prompt") or ex.get("instruction") or ex.get("input");c=ex.get("completion") or ex.get("output") or ex.get("response")
    if p is not None and c is not None:
        try:return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":str(p)},{"role":"assistant","content":str(c)}],tokenize=False,add_generation_prompt=False)
        except:return ""
    return ""
texts=[]
for line in open(DATA):
    try:
        tt=row_text(json.loads(line))
        if tt:texts.append(tt)
    except:pass
slog("dataset rows=%d trainable=%.1fM"%(len(texts),sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6))
dscache={}
def make_ds(ML):
    if ML in dscache:return dscache[ML]
    def tk(b):
        o=tok(b["text"],add_special_tokens=False,truncation=True,max_length=ML);o["labels"]=[x[:] for x in o["input_ids"]];o["length"]=[len(x) for x in o["input_ids"]];return o
    d=Dataset.from_dict({"text":texts}).map(tk,batched=True,remove_columns=["text"]);dscache[ML]=d;return d
class Coll:
    def __init__(self,t):self.t=t
    def __call__(self,feats):
        m=max(len(f["input_ids"]) for f in feats);pid=self.t.pad_token_id or 0
        ii=[];ll=[];am=[]
        for f in feats:
            ids=f["input_ids"];lab=f["labels"];pad=m-len(ids)
            ii.append(ids+[pid]*pad);ll.append(lab+[-100]*pad);am.append([1]*len(ids)+[0]*pad)
        return {"input_ids":torch.tensor(ii),"labels":torch.tensor(ll),"attention_mask":torch.tensor(am)}
coll=Coll(tok)
class CB(TrainerCallback):
    def __init__(self,lf):self.lf=lf;self.lt=time.time();self.lk=0
    def on_step_end(self,a,s,c,**k):
        if os.path.exists(CTL+"/STOP"):c.should_training_stop=True
    def on_log(self,a,s,c,logs=None,**k):
        if not(R0 and logs and "loss" in logs):return
        now=time.time();tn=getattr(s,"num_input_tokens_seen",0);d=(tn-self.lk)*WORLD;dt=now-self.lt;tps=d/dt if dt>0 else 0
        self.lt=now;self.lk=tn;v=torch.cuda.max_memory_allocated()/1e9
        open(self.lf,"a").write(time.strftime("[%H:%M:%S] ")+"step %-6d loss=%-7s lr=%-9s tok/s(2gpu)=%-6d vram=%.1fGB\n"%(s.global_step,str(logs.get("loss")),str(logs.get("learning_rate")),tps,v))
def run_job(job):
    jid=job["jobid"];ML=int(job["maxlen"]);B=int(job["batch"]);STEPS=int(job.get("steps",0));EP=float(job.get("epochs",0))
    LRr=float(job.get("lr",7e-5));SAVE=bool(job.get("save",False));OUT=job.get("out","/tmp/gemma4_best_run")
    GBL=bool(job.get("group_by_length",True));NEFT=float(job.get("neftune",0));WD=float(job.get("weight_decay",0.01));WU=float(job.get("warmup_ratio",0.03));SCH=job.get("sched","cosine");GA=int(job.get("grad_acc",1))
    lf=CTL+"/"+jid+".log";ds=make_ds(ML)
    if R0:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"START job=%s maxlen=%d batch=%d eff=%d steps=%d ep=%s gbl=%s neft=%s\n"%(jid,ML,B,B*WORLD*GA,STEPS,EP,GBL,NEFT))
    torch.cuda.reset_peak_memory_stats()
    kw=dict(output_dir=OUT,per_device_train_batch_size=B,gradient_accumulation_steps=GA,learning_rate=LRr,warmup_ratio=WU,lr_scheduler_type=SCH,weight_decay=WD,optim="adamw_torch_fused",bf16=True,logging_steps=1,save_strategy="no",report_to="none",dataloader_num_workers=2,include_num_input_tokens_seen=True,ddp_find_unused_parameters=False,gradient_checkpointing=False,group_by_length=GBL)
    if NEFT>0:kw["neftune_noise_alpha"]=NEFT
    if EP>0:kw["num_train_epochs"]=EP
    else:kw["max_steps"]=STEPS
    args=TrainingArguments(**kw)
    tr=Trainer(model=model,args=args,train_dataset=ds,data_collator=coll,callbacks=[CB(lf)])
    t0=time.time();err=None;res=None
    try:res=tr.train()
    except Exception as e:err=repr(e)[:300]
    dt=time.time()-t0
    if R0:
        if err is None:
            if SAVE:tr.save_model(OUT);tok.save_pretrained(OUT)
            tn=getattr(tr.state,"num_input_tokens_seen",0);v=torch.cuda.max_memory_allocated()/1e9
            open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"DONE "+json.dumps({"steps":tr.state.global_step,"sec":round(dt,1),"loss":round(float(res.training_loss),4),"tok_s_2gpu":round(tn*WORLD/dt) if dt else 0,"peak_vram_gb":round(v,1),"saved":SAVE,"out":OUT})+"\n")
        else:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"ERROR "+err+"\n")
    del tr;gc.collect();torch.cuda.empty_cache()
    try:
        if R0 and os.path.exists(CTL+"/STOP"):os.remove(CTL+"/STOP")
    except:pass
    if R0:open(CTL+"/"+jid+".done","w").write("1")
slog("READY")
processed=set()
while True:
    nxt=None
    for fp in sorted(glob.glob(JOBS+"/*.json")):
        if fp not in processed:nxt=fp;break
    if nxt is None:
        time.sleep(1);continue
    processed.add(nxt)
    try:job=json.load(open(nxt))
    except:continue
    dist.barrier()
    if job.get("cmd")=="shutdown":
        slog("shutdown");break
    run_job(job)
    dist.barrier()
'''
open('/tmp/ddp_server.py','w').write(SERVER)
open(CTL+'/server.log','w').write('')
env=os.environ.copy()
env.update({'PYTHONPATH':'/tmp/hubfix:'+env.get('PYTHONPATH',''),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0,1','NCCL_DEBUG':'WARN','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','DATA_PATH':TRAIN,'CAP':'4096','LORA_R':'64','LORA_ALPHA':'128'}); env.pop('RANK',None)
proc=subprocess.Popen([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/ddp_server.py'],env=env,stdout=open(CTL+'/stdout.log','w'),stderr=subprocess.STDOUT)
globals()['WORKER_PROC']=proc
print(time.strftime('[%H:%M:%S]'),'worker v2 launching (one-time reload ~3-6min, r=64)... waiting READY',flush=True)
t0=time.time();ready=False
while time.time()-t0<720:
    if proc.poll() is not None:
        print('[worker EXITED] tail:',flush=True);print(open(CTL+'/stdout.log').read()[-1500:]);break
    if 'READY' in open(CTL+'/server.log').read():
        print(open(CTL+'/server.log').read().strip(),flush=True);print(time.strftime('[%H:%M:%S]'),'WORKER v2 READY',flush=True);ready=True;break
    time.sleep(3)
if not ready and proc.poll() is None:print('[still loading; check /tmp/ddp_ctl/server.log]',flush=True)

In [ ]:
# [WORKER-SWEEP v2] mini batch sweep on resident worker (r64 + group_by_length). No reload between configs.
import json, os, time
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'
assert ('WORKER_PROC' in globals() and globals()['WORKER_PROC'].poll() is None), 'run [WORKER-START v2] first'
GRID=[(3072,16),(3072,20),(3072,24),(3072,28),(3072,32),(4096,16),(4096,20)]
STEPS=12; results=[]
for ML,B in GRID:
    jid='sw_%d'%int(time.time()*1000)
    job={'jobid':jid,'maxlen':ML,'batch':B,'steps':STEPS,'group_by_length':True,'neftune':0,'save':False}
    lf=CTL+'/'+jid+'.log'; donef=CTL+'/'+jid+'.done'
    tmp=JOBS+'/'+jid+'.json.tmp'; open(tmp,'w').write(json.dumps(job)); os.rename(tmp,JOBS+'/'+jid+'.json')
    print(time.strftime('[%H:%M:%S]'),'CONFIG maxlen=%d batch=%d eff=%d -> %d steps...'%(ML,B,B*2,STEPS),flush=True)
    t0=time.time()
    while not os.path.exists(donef):
        if time.time()-t0>600: print('  timeout'); break
        time.sleep(0.5)
    d=open(lf).read() if os.path.exists(lf) else ''
    rec=None
    for line in d.splitlines():
        if 'DONE ' in line:
            try: rec=json.loads(line.split('DONE ',1)[1]); rec.update({'maxlen':ML,'batch':B,'eff_batch':B*2})
            except: pass
        if 'ERROR ' in line:
            print('  -> ERROR/OOM at batch=%d maxlen=%d (stop pushing higher)'%(B,ML),flush=True); rec='OOM'
    if rec=='OOM': break
    if rec:
        results.append(rec); print('  -> tok/s(2gpu)=%-6d peak_vram=%-5.1fGB loss=%.3f (%.1fs)'%(rec['tok_s_2gpu'],rec['peak_vram_gb'],rec['loss'],rec['sec']),flush=True)
results.sort(key=lambda r:r['tok_s_2gpu'],reverse=True)
print('\n===== RANKED (r64 + group_by_length) =====',flush=True)
for r in results: print(json.dumps({k:r[k] for k in ('maxlen','batch','eff_batch','tok_s_2gpu','peak_vram_gb','loss','sec')}),flush=True)
if results: print('\nBEST:',json.dumps({k:results[0][k] for k in ('maxlen','batch','tok_s_2gpu','peak_vram_gb')}),flush=True)

In [ ]:
# [WORKER-RUN v2] optimized training on resident worker. Edit params + run. group_by_length + NEFTune for quality.
import json, os, time
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'
# ===== EDIT =====
JOB_MAXLEN=3072
JOB_BATCH=24
JOB_EPOCHS=0          # >0 uses epochs (overrides steps). e.g. 2 epochs
JOB_STEPS=200         # used when JOB_EPOCHS=0
JOB_LR=7e-5
JOB_GBL=True          # group_by_length: less padding -> faster
JOB_NEFT=5.0          # NEFTune noise alpha (quality). 0 to disable
JOB_WD=0.01
JOB_SAVE=True
JOB_OUT='/tmp/gemma4_best_run'
# ================
assert ('WORKER_PROC' in globals() and globals()['WORKER_PROC'].poll() is None), 'run [WORKER-START v2] first'
jid='run_%d'%int(time.time())
job={'jobid':jid,'maxlen':JOB_MAXLEN,'batch':JOB_BATCH,'epochs':JOB_EPOCHS,'steps':JOB_STEPS,'lr':JOB_LR,'group_by_length':JOB_GBL,'neftune':JOB_NEFT,'weight_decay':JOB_WD,'save':JOB_SAVE,'out':JOB_OUT}
lf=CTL+'/'+jid+'.log'; donef=CTL+'/'+jid+'.done'
tmp=JOBS+'/'+jid+'.json.tmp'; open(tmp,'w').write(json.dumps(job)); os.rename(tmp,JOBS+'/'+jid+'.json')
print(time.strftime('[%H:%M:%S]'),'submitted',jid,job,flush=True)
seen=0
try:
    while not os.path.exists(donef):
        if os.path.exists(lf):
            x=open(lf).read()
            if len(x)>seen:print(x[seen:],end='',flush=True);seen=len(x)
        time.sleep(0.5)
    x=open(lf).read()
    if len(x)>seen:print(x[seen:],end='',flush=True)
    print('\n[job finished]',flush=True)
except KeyboardInterrupt:
    print('\n[stopped tailing; job runs in worker. Use WORKER-STOP to end+save]',flush=True)

In [ ]:
import os, time
CTL='/tmp/ddp_ctl'
print('WORKER alive:', ('WORKER_PROC' in globals() and globals()['WORKER_PROC'].poll() is None))
print('server.log:'); print(open(CTL+'/server.log').read())
if 'READY' not in open(CTL+'/server.log').read():
    print('not ready yet; tail stdout:'); print(open(CTL+'/stdout.log').read()[-700:])

In [ ]:
# [WORKER-START v3] robust persistent 2-GPU worker (r64). Filters TrainingArguments by signature; bad job won't kill worker.
import os, sys, subprocess, time, glob
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'; os.makedirs(JOBS, exist_ok=True)
if 'WORKER_PROC' in globals() and globals()['WORKER_PROC'] and globals()['WORKER_PROC'].poll() is None:
    print('stopping previous worker...');
    try: globals()['WORKER_PROC'].terminate(); time.sleep(4)
    except: pass
for f in glob.glob(CTL+'/*.log')+glob.glob(CTL+'/*.done')+glob.glob(JOBS+'/*')+glob.glob(CTL+'/STOP'):
    try: os.remove(f)
    except: pass
PACK='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605'
DS=Path('/tmp/gemma4_quality4_dataset'); DS.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(DS)+'/train.jsonl'):
    session.sql(f"GET {PACK}/dataset_quality4 file://{DS} PATTERN='.*[.]jsonl' PARALLEL=16").collect()
TRAIN=str(DS/'train.jsonl')
SERVER=r'''
import os,sys,json,time,glob,gc,inspect
os.environ.setdefault("HF_HUB_OFFLINE","1");os.environ.setdefault("TRANSFORMERS_OFFLINE","1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True");os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
import torch, torch.distributed as dist
LR=int(os.environ.get("LOCAL_RANK","0"));WORLD=int(os.environ.get("WORLD_SIZE","1"));R0=(LR==0)
torch.cuda.set_device(LR)
CTL="/tmp/ddp_ctl";JOBS=CTL+"/jobs";DATA=os.environ["DATA_PATH"]
def slog(m):
    if R0:open(CTL+"/server.log","a").write(time.strftime("[%H:%M:%S] ")+m+"\n")
dist.init_process_group(backend="nccl")
import unsloth
import unsloth_zoo.temporary_patches.gemma4 as g4
if hasattr(g4,"_Gemma4KVSharedSafeProxy"):
    g4._Gemma4KVSharedSafeProxy.__getattr__=lambda self,name:getattr(object.__getattribute__(self,"_real"),name)
from unsloth import FastLanguageModel
from transformers import Trainer,TrainingArguments,TrainerCallback
from datasets import Dataset
TA_VALID=set(inspect.signature(TrainingArguments.__init__).parameters)
MODEL_DIR="/tmp/google_gemma4_e4b_model";CAP=int(os.environ.get("CAP","4096"))
RR=int(os.environ.get("LORA_R","64"));AA=int(os.environ.get("LORA_ALPHA","128"))
slog("loading model once (cap=%d, LoRA r=%d alpha=%d)..."%(CAP,RR,AA))
model,tok=FastLanguageModel.from_pretrained(model_name=MODEL_DIR,max_seq_length=CAP,load_in_4bit=False,load_in_16bit=True,full_finetuning=False,local_files_only=True)
if hasattr(tok,"tokenizer"):tok=tok.tokenizer
if tok.pad_token is None:tok.pad_token=tok.eos_token
model=FastLanguageModel.get_peft_model(model,r=RR,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=AA,lora_dropout=0.0,bias="none",use_rslora=True,use_gradient_checkpointing="unsloth")
for n,p in model.named_parameters():
    if any(k in n for k in ["vision_tower","audio_tower","embed_vision","embed_audio"]):p.requires_grad_(False)
SYS="Ban la AI chuyen sinh seed, mutate fuzz script va danh gia oracle cho PyTorch/TensorFlow."
def row_text(ex):
    t=ex.get("text")
    if t:return str(t)
    m=ex.get("messages")
    if m:
        if isinstance(m,str):
            try:m=json.loads(m)
            except:return ""
        try:return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=False)
        except:return ""
    p=ex.get("prompt") or ex.get("instruction") or ex.get("input");c=ex.get("completion") or ex.get("output") or ex.get("response")
    if p is not None and c is not None:
        try:return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":str(p)},{"role":"assistant","content":str(c)}],tokenize=False,add_generation_prompt=False)
        except:return ""
    return ""
texts=[]
for line in open(DATA):
    try:
        tt=row_text(json.loads(line))
        if tt:texts.append(tt)
    except:pass
slog("dataset rows=%d trainable=%.1fM"%(len(texts),sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6))
dscache={}
def make_ds(ML):
    if ML in dscache:return dscache[ML]
    def tk(b):
        o=tok(b["text"],add_special_tokens=False,truncation=True,max_length=ML);o["labels"]=[x[:] for x in o["input_ids"]];return o
    d=Dataset.from_dict({"text":texts}).map(tk,batched=True,remove_columns=["text"]);dscache[ML]=d;return d
class Coll:
    def __init__(self,t):self.t=t
    def __call__(self,feats):
        m=max(len(f["input_ids"]) for f in feats);pid=self.t.pad_token_id or 0
        ii=[];ll=[];am=[]
        for f in feats:
            ids=f["input_ids"];lab=f["labels"];pad=m-len(ids)
            ii.append(ids+[pid]*pad);ll.append(lab+[-100]*pad);am.append([1]*len(ids)+[0]*pad)
        return {"input_ids":torch.tensor(ii),"labels":torch.tensor(ll),"attention_mask":torch.tensor(am)}
coll=Coll(tok)
class CB(TrainerCallback):
    def __init__(self,lf):self.lf=lf;self.lt=time.time();self.lk=0
    def on_step_end(self,a,s,c,**k):
        if os.path.exists(CTL+"/STOP"):c.should_training_stop=True
    def on_log(self,a,s,c,logs=None,**k):
        if not(R0 and logs and "loss" in logs):return
        now=time.time();tn=getattr(s,"num_input_tokens_seen",0);d=(tn-self.lk)*WORLD;dt=now-self.lt;tps=d/dt if dt>0 else 0
        self.lt=now;self.lk=tn;v=torch.cuda.max_memory_allocated()/1e9
        open(self.lf,"a").write(time.strftime("[%H:%M:%S] ")+"step %-6d loss=%-7s lr=%-9s tok/s(2gpu)=%-6d vram=%.1fGB\n"%(s.global_step,str(logs.get("loss")),str(logs.get("learning_rate")),tps,v))
def run_job(job):
    jid=job["jobid"];lf=CTL+"/"+jid+".log"
    try:
        ML=int(job["maxlen"]);B=int(job["batch"]);STEPS=int(job.get("steps",0));EP=float(job.get("epochs",0))
        LRr=float(job.get("lr",7e-5));SAVE=bool(job.get("save",False));OUT=job.get("out","/tmp/gemma4_best_run")
        NEFT=float(job.get("neftune",0));WD=float(job.get("weight_decay",0.01));WU=float(job.get("warmup_ratio",0.03));SCH=job.get("sched","cosine");GA=int(job.get("grad_acc",1))
        ds=make_ds(ML)
        if R0:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"START job=%s maxlen=%d batch=%d eff=%d steps=%d ep=%s neft=%s\n"%(jid,ML,B,B*WORLD*GA,STEPS,EP,NEFT))
        torch.cuda.reset_peak_memory_stats()
        kw=dict(output_dir=OUT,per_device_train_batch_size=B,gradient_accumulation_steps=GA,learning_rate=LRr,warmup_ratio=WU,lr_scheduler_type=SCH,weight_decay=WD,optim="adamw_torch_fused",bf16=True,logging_steps=1,save_strategy="no",report_to="none",dataloader_num_workers=2,include_num_input_tokens_seen=True,ddp_find_unused_parameters=False,gradient_checkpointing=False)
        if NEFT>0:kw["neftune_noise_alpha"]=NEFT
        if EP>0:kw["num_train_epochs"]=EP
        else:kw["max_steps"]=STEPS
        kw={k:v for k,v in kw.items() if k in TA_VALID}
        args=TrainingArguments(**kw)
        tr=Trainer(model=model,args=args,train_dataset=ds,data_collator=coll,callbacks=[CB(lf)])
        t0=time.time();res=tr.train();dt=time.time()-t0
        if R0:
            if SAVE:tr.save_model(OUT);tok.save_pretrained(OUT)
            tn=getattr(tr.state,"num_input_tokens_seen",0);v=torch.cuda.max_memory_allocated()/1e9
            open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"DONE "+json.dumps({"steps":tr.state.global_step,"sec":round(dt,1),"loss":round(float(res.training_loss),4),"tok_s_2gpu":round(tn*WORLD/dt) if dt else 0,"peak_vram_gb":round(v,1),"saved":SAVE,"out":OUT})+"\n")
        del tr
    except Exception as e:
        if R0:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"ERROR "+repr(e)[:400]+"\n")
    finally:
        gc.collect();torch.cuda.empty_cache()
        try:
            if R0 and os.path.exists(CTL+"/STOP"):os.remove(CTL+"/STOP")
        except:pass
        if R0:open(CTL+"/"+jid+".done","w").write("1")
slog("READY")
processed=set()
while True:
    nxt=None
    for fp in sorted(glob.glob(JOBS+"/*.json")):
        if fp not in processed:nxt=fp;break
    if nxt is None:
        time.sleep(1);continue
    processed.add(nxt)
    try:job=json.load(open(nxt))
    except:continue
    dist.barrier()
    if job.get("cmd")=="shutdown":
        slog("shutdown");break
    run_job(job)
    dist.barrier()
'''
open('/tmp/ddp_server.py','w').write(SERVER)
open(CTL+'/server.log','w').write('')
env=os.environ.copy()
env.update({'PYTHONPATH':'/tmp/hubfix:'+env.get('PYTHONPATH',''),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0,1','NCCL_DEBUG':'WARN','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','DATA_PATH':TRAIN,'CAP':'4096','LORA_R':'64','LORA_ALPHA':'128'}); env.pop('RANK',None)
proc=subprocess.Popen([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/ddp_server.py'],env=env,stdout=open(CTL+'/stdout.log','w'),stderr=subprocess.STDOUT)
globals()['WORKER_PROC']=proc
print(time.strftime('[%H:%M:%S]'),'worker v3 launching (one-time reload ~3-6min, r=64, robust)... waiting READY',flush=True)
t0=time.time();ready=False
while time.time()-t0<720:
    if proc.poll() is not None:
        print('[worker EXITED] tail:',flush=True);print(open(CTL+'/stdout.log').read()[-1500:]);break
    if 'READY' in open(CTL+'/server.log').read():
        print(open(CTL+'/server.log').read().strip(),flush=True);print(time.strftime('[%H:%M:%S]'),'WORKER v3 READY',flush=True);ready=True;break
    time.sleep(3)
if not ready and proc.poll() is None:print('[still loading; check /tmp/ddp_ctl/server.log]',flush=True)

In [ ]:
# [WORKER-START v4] robust r64 worker WITH checkpointing (save_strategy=steps) + resume_from_checkpoint + epochs + neftune
import os, sys, subprocess, time, glob
from pathlib import Path
from snowflake.snowpark.context import get_active_session
session=get_active_session()
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'; os.makedirs(JOBS, exist_ok=True)
if 'WORKER_PROC' in globals() and globals()['WORKER_PROC'] and globals()['WORKER_PROC'].poll() is None:
    print('stopping previous worker...');
    try: globals()['WORKER_PROC'].terminate(); time.sleep(4)
    except: pass
for f in glob.glob(CTL+'/*.log')+glob.glob(CTL+'/*.done')+glob.glob(JOBS+'/*')+glob.glob(CTL+'/STOP'):
    try: os.remove(f)
    except: pass
PACK='@ML_DB.MODELS.FUZZ_DATASET_STAGE/google_gemma4_e4b_unsloth_blackwell_ddp_pack_20260605'
DS=Path('/tmp/gemma4_quality4_dataset'); DS.mkdir(parents=True, exist_ok=True)
if not glob.glob(str(DS)+'/train.jsonl'):
    session.sql(f"GET {PACK}/dataset_quality4 file://{DS} PATTERN='.*[.]jsonl' PARALLEL=16").collect()
TRAIN=str(DS/'train.jsonl')
SERVER=r'''
import os,sys,json,time,glob,gc,inspect
os.environ.setdefault("HF_HUB_OFFLINE","1");os.environ.setdefault("TRANSFORMERS_OFFLINE","1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True");os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
import torch, torch.distributed as dist
LR=int(os.environ.get("LOCAL_RANK","0"));WORLD=int(os.environ.get("WORLD_SIZE","1"));R0=(LR==0)
torch.cuda.set_device(LR)
CTL="/tmp/ddp_ctl";JOBS=CTL+"/jobs";DATA=os.environ["DATA_PATH"]
def slog(m):
    if R0:open(CTL+"/server.log","a").write(time.strftime("[%H:%M:%S] ")+m+"\n")
dist.init_process_group(backend="nccl")
import unsloth
import unsloth_zoo.temporary_patches.gemma4 as g4
if hasattr(g4,"_Gemma4KVSharedSafeProxy"):
    g4._Gemma4KVSharedSafeProxy.__getattr__=lambda self,name:getattr(object.__getattribute__(self,"_real"),name)
from unsloth import FastLanguageModel
from transformers import Trainer,TrainingArguments,TrainerCallback
from datasets import Dataset
TA_VALID=set(inspect.signature(TrainingArguments.__init__).parameters)
MODEL_DIR="/tmp/google_gemma4_e4b_model";CAP=int(os.environ.get("CAP","4096"))
RR=int(os.environ.get("LORA_R","64"));AA=int(os.environ.get("LORA_ALPHA","128"))
slog("loading model once (cap=%d, LoRA r=%d alpha=%d)..."%(CAP,RR,AA))
model,tok=FastLanguageModel.from_pretrained(model_name=MODEL_DIR,max_seq_length=CAP,load_in_4bit=False,load_in_16bit=True,full_finetuning=False,local_files_only=True)
if hasattr(tok,"tokenizer"):tok=tok.tokenizer
if tok.pad_token is None:tok.pad_token=tok.eos_token
model=FastLanguageModel.get_peft_model(model,r=RR,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=AA,lora_dropout=0.0,bias="none",use_rslora=True,use_gradient_checkpointing="unsloth")
for n,p in model.named_parameters():
    if any(k in n for k in ["vision_tower","audio_tower","embed_vision","embed_audio"]):p.requires_grad_(False)
SYS="Ban la AI chuyen sinh seed, mutate fuzz script va danh gia oracle cho PyTorch/TensorFlow."
def row_text(ex):
    t=ex.get("text")
    if t:return str(t)
    m=ex.get("messages")
    if m:
        if isinstance(m,str):
            try:m=json.loads(m)
            except:return ""
        try:return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=False)
        except:return ""
    p=ex.get("prompt") or ex.get("instruction") or ex.get("input");c=ex.get("completion") or ex.get("output") or ex.get("response")
    if p is not None and c is not None:
        try:return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":str(p)},{"role":"assistant","content":str(c)}],tokenize=False,add_generation_prompt=False)
        except:return ""
    return ""
texts=[]
for line in open(DATA):
    try:
        tt=row_text(json.loads(line))
        if tt:texts.append(tt)
    except:pass
slog("dataset rows=%d trainable=%.1fM"%(len(texts),sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6))
dscache={}
def make_ds(ML):
    if ML in dscache:return dscache[ML]
    def tk(b):
        o=tok(b["text"],add_special_tokens=False,truncation=True,max_length=ML);o["labels"]=[x[:] for x in o["input_ids"]];return o
    d=Dataset.from_dict({"text":texts}).map(tk,batched=True,remove_columns=["text"]);dscache[ML]=d;return d
class Coll:
    def __init__(self,t):self.t=t
    def __call__(self,feats):
        m=max(len(f["input_ids"]) for f in feats);pid=self.t.pad_token_id or 0
        ii=[];ll=[];am=[]
        for f in feats:
            ids=f["input_ids"];lab=f["labels"];pad=m-len(ids)
            ii.append(ids+[pid]*pad);ll.append(lab+[-100]*pad);am.append([1]*len(ids)+[0]*pad)
        return {"input_ids":torch.tensor(ii),"labels":torch.tensor(ll),"attention_mask":torch.tensor(am)}
coll=Coll(tok)
class CB(TrainerCallback):
    def __init__(self,lf):self.lf=lf;self.lt=time.time();self.lk=0
    def on_step_end(self,a,s,c,**k):
        if os.path.exists(CTL+"/STOP"):c.should_training_stop=True
    def on_save(self,a,s,c,**k):
        if R0:open(self.lf,"a").write(time.strftime("[%H:%M:%S] ")+"** checkpoint saved @ step %d (resumable)\n"%s.global_step)
    def on_log(self,a,s,c,logs=None,**k):
        if not(R0 and logs and "loss" in logs):return
        now=time.time();tn=getattr(s,"num_input_tokens_seen",0);d=(tn-self.lk)*WORLD;dt=now-self.lt;tps=d/dt if dt>0 else 0
        self.lt=now;self.lk=tn;v=torch.cuda.max_memory_allocated()/1e9
        ep=getattr(s,"epoch",0) or 0
        open(self.lf,"a").write(time.strftime("[%H:%M:%S] ")+"step %-6d ep=%.2f loss=%-7s lr=%-9s tok/s(2gpu)=%-6d vram=%.1fGB\n"%(s.global_step,ep,str(logs.get("loss")),str(logs.get("learning_rate")),tps,v))
def run_job(job):
    jid=job["jobid"];lf=CTL+"/"+jid+".log"
    try:
        ML=int(job["maxlen"]);B=int(job["batch"]);STEPS=int(job.get("steps",0));EP=float(job.get("epochs",0))
        LRr=float(job.get("lr",7e-5));SAVE=bool(job.get("save",True));OUT=job.get("out","/tmp/gemma4_quality_run")
        NEFT=float(job.get("neftune",0));WD=float(job.get("weight_decay",0.01));WU=float(job.get("warmup_ratio",0.03));SCH=job.get("sched","cosine");GA=int(job.get("grad_acc",1))
        SS=int(job.get("save_steps",50));SL=int(job.get("save_total_limit",5));RESUME=bool(job.get("resume",False))
        os.makedirs(OUT,exist_ok=True);ds=make_ds(ML)
        if R0:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"START job=%s maxlen=%d batch=%d eff=%d epochs=%s steps=%d neft=%s save_steps=%d resume=%s out=%s\n"%(jid,ML,B,B*WORLD*GA,EP,STEPS,NEFT,SS,RESUME,OUT))
        torch.cuda.reset_peak_memory_stats()
        kw=dict(output_dir=OUT,per_device_train_batch_size=B,gradient_accumulation_steps=GA,learning_rate=LRr,warmup_ratio=WU,lr_scheduler_type=SCH,weight_decay=WD,optim="adamw_torch_fused",bf16=True,logging_steps=1,save_strategy="steps",save_steps=SS,save_total_limit=SL,save_safetensors=True,report_to="none",dataloader_num_workers=2,include_num_input_tokens_seen=True,ddp_find_unused_parameters=False,gradient_checkpointing=False)
        if NEFT>0:kw["neftune_noise_alpha"]=NEFT
        if EP>0:kw["num_train_epochs"]=EP
        else:kw["max_steps"]=STEPS
        kw={k:v for k,v in kw.items() if k in TA_VALID}
        args=TrainingArguments(**kw)
        tr=Trainer(model=model,args=args,train_dataset=ds,data_collator=coll,callbacks=[CB(lf)])
        rc=None
        if RESUME:
            cks=[p for p in glob.glob(OUT+"/checkpoint-*") if p.rsplit("-",1)[-1].isdigit()]
            cks=sorted(cks,key=lambda p:int(p.rsplit("-",1)[-1]))
            if cks:rc=cks[-1]
            if R0 and rc:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"RESUME from "+rc+"\n")
        t0=time.time();res=tr.train(resume_from_checkpoint=rc);dt=time.time()-t0
        if R0:
            if SAVE:tr.save_model(OUT);tok.save_pretrained(OUT)
            tn=getattr(tr.state,"num_input_tokens_seen",0);v=torch.cuda.max_memory_allocated()/1e9
            open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"DONE "+json.dumps({"steps":tr.state.global_step,"epoch":round(float(getattr(tr.state,"epoch",0) or 0),3),"sec":round(dt,1),"loss":round(float(res.training_loss),4),"tok_s_2gpu":round(tn*WORLD/dt) if dt else 0,"peak_vram_gb":round(v,1),"out":OUT})+"\n")
        del tr
    except Exception as e:
        if R0:open(lf,"a").write(time.strftime("[%H:%M:%S] ")+"ERROR "+repr(e)[:400]+"\n")
    finally:
        gc.collect();torch.cuda.empty_cache()
        try:
            if R0 and os.path.exists(CTL+"/STOP"):os.remove(CTL+"/STOP")
        except:pass
        if R0:open(CTL+"/"+jid+".done","w").write("1")
slog("READY")
processed=set()
while True:
    nxt=None
    for fp in sorted(glob.glob(JOBS+"/*.json")):
        if fp not in processed:nxt=fp;break
    if nxt is None:
        time.sleep(1);continue
    processed.add(nxt)
    try:job=json.load(open(nxt))
    except:continue
    dist.barrier()
    if job.get("cmd")=="shutdown":
        slog("shutdown");break
    run_job(job)
    dist.barrier()
'''
open('/tmp/ddp_server.py','w').write(SERVER)
open(CTL+'/server.log','w').write('')
env=os.environ.copy()
env.update({'PYTHONPATH':'/tmp/hubfix:'+env.get('PYTHONPATH',''),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0,1','NCCL_DEBUG':'WARN','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','DATA_PATH':TRAIN,'CAP':'4096','LORA_R':'64','LORA_ALPHA':'128'}); env.pop('RANK',None)
proc=subprocess.Popen([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/ddp_server.py'],env=env,stdout=open(CTL+'/stdout.log','w'),stderr=subprocess.STDOUT)
globals()['WORKER_PROC']=proc
print(time.strftime('[%H:%M:%S]'),'worker v4 launching (checkpointing+resume; one-time reload ~10-12min)... waiting READY',flush=True)
t0=time.time();ready=False
while time.time()-t0<1000:
    if proc.poll() is not None:
        print('[worker EXITED] tail:',flush=True);print(open(CTL+'/stdout.log').read()[-1500:]);break
    if 'READY' in open(CTL+'/server.log').read():
        print(open(CTL+'/server.log').read().strip(),flush=True);print(time.strftime('[%H:%M:%S]'),'WORKER v4 READY (checkpointing on)',flush=True);ready=True;break
    time.sleep(3)
if not ready and proc.poll() is None:print('[still loading; re-run [DIAG] to check server.log]',flush=True)

In [ ]:
# [RUN-QUALITY] highest-quality multi-epoch training w/ checkpoints. Resumable. Edit + run; interrupt anytime ([WORKER-STOP]).
import json, os, time
CTL='/tmp/ddp_ctl'; JOBS=CTL+'/jobs'
# ===== EDIT =====
Q_MAXLEN=4096; Q_BATCH=20      # highest-quality config (long context)
Q_EPOCHS=3                     # multiple epochs
Q_LR=7e-5; Q_NEFT=5.0          # NEFTune for quality
Q_SAVE_STEPS=50                # checkpoint every 50 steps (resumable)
Q_SAVE_LIMIT=8
Q_RESUME=False                 # set True to continue from latest checkpoint in Q_OUT
Q_OUT='/tmp/gemma4_quality_run'
# ================
assert ('WORKER_PROC' in globals() and globals()['WORKER_PROC'].poll() is None), 'run [WORKER-START v4] first'
jid='q_%d'%int(time.time())
job={'jobid':jid,'maxlen':Q_MAXLEN,'batch':Q_BATCH,'epochs':Q_EPOCHS,'steps':0,'lr':Q_LR,'neftune':Q_NEFT,'save_steps':Q_SAVE_STEPS,'save_total_limit':Q_SAVE_LIMIT,'resume':Q_RESUME,'save':True,'out':Q_OUT}
lf=CTL+'/'+jid+'.log'; donef=CTL+'/'+jid+'.done'
tmp=JOBS+'/'+jid+'.json.tmp'; open(tmp,'w').write(json.dumps(job)); os.rename(tmp,JOBS+'/'+jid+'.json')
print(time.strftime('[%H:%M:%S]'),'submitted',jid,job,flush=True)
seen=0
try:
    while not os.path.exists(donef):
        if os.path.exists(lf):
            x=open(lf).read()
            if len(x)>seen:print(x[seen:],end='',flush=True);seen=len(x)
        time.sleep(0.5)
    x=open(lf).read()
    if len(x)>seen:print(x[seen:],end='',flush=True)
    print('\n[job finished]',flush=True)
except KeyboardInterrupt:
    print('\n[stopped tailing; job keeps running. Use [WORKER-STOP] to end+save, or re-run this cell to keep watching]',flush=True)

In [ ]:
# [SAVE-TO-STAGE] Persist trained adapter + checkpoints to stage (survives service suspend / cross-session recovery).
import os, glob, time
from snowflake.snowpark.context import get_active_session
session=get_active_session()
SRC=os.environ.get('Q_OUT','/tmp/gemma4_quality_run')   # change if different out dir
STAGE='@ML_DB.MODELS.FUZZ_DATASET_STAGE/gemma4_quality_run_ckpt'
assert os.path.isdir(SRC), 'no trained dir at '+SRC
# upload final adapter files (top level) and the latest checkpoint dir
def put(localfile, dest):
    session.sql(f"PUT 'file://{localfile}' {dest} AUTO_COMPRESS=FALSE OVERWRITE=TRUE PARALLEL=8").collect()
top=[f for f in glob.glob(SRC+'/*') if os.path.isfile(f)]
for f in top: put(f, STAGE+'/adapter'); print('put', os.path.basename(f))
cks=sorted([p for p in glob.glob(SRC+'/checkpoint-*') if p.rsplit('-',1)[-1].isdigit()], key=lambda p:int(p.rsplit('-',1)[-1]))
if cks:
    latest=cks[-1]; name=os.path.basename(latest)
    for f in glob.glob(latest+'/*'):
        if os.path.isfile(f): put(f, STAGE+'/'+name); 
    print('put latest checkpoint:', name)
session.sql('ALTER STAGE ML_DB.MODELS.FUZZ_DATASET_STAGE REFRESH').collect()
print('DONE -> ', STAGE)

## [ENV-2GPU] Re-create 2-GPU Blackwell environment (new account, 2026-06-06)

New account was reset (no `BLACKWELL_2GPU_POOL`, no service). Recreating the compute environment for Gemma4 Unsloth 2-GPU DDP training.

- **Instance family:** `GPU_R6K_G1_48` = **2 × NVIDIA RTX PRO 6000 (Blackwell)**, 96 GB VRAM each (192 GB total), 44 vCPU, 490 GiB RAM.
- **Pool:** `BLACKWELL_2GPU_POOL` (1 node, AUTO_SUSPEND 600s, AUTO_RESUME on).
- Next: GPU notebook **service** on this pool → connect kernel → rebuild `/tmp` env (installer + hub fix + datasets4 + sitecustomize) → persistent 2-GPU worker.

_Record-only cell (project log). Existing cells are not modified._

In [ ]:
%%sql -r blackwell_pool_desc
-- [ENV-2GPU] Reproducible DDL: 2-GPU Blackwell pool
CREATE COMPUTE POOL IF NOT EXISTS BLACKWELL_2GPU_POOL
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = GPU_R6K_G1_48          -- 2x RTX PRO 6000 Blackwell, 96GB each
  AUTO_RESUME = TRUE
  AUTO_SUSPEND_SECS = 600
  COMMENT = 'Gemma4 Unsloth 2-GPU DDP training (2x RTX PRO 6000 Blackwell, 192GB)';

DESCRIBE COMPUTE POOL BLACKWELL_2GPU_POOL;

In [ ]:
# [VERIFY-2GPU] Confirm 2x RTX PRO 6000 Blackwell visible on the new service
import subprocess, datetime, sys
def ts():
    return datetime.datetime.now().strftime('%H:%M:%S')
print(f'[{ts()}] python   :', sys.version.split()[0], flush=True)
out = subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total,driver_version','--format=csv,noheader'], capture_output=True, text=True)
print(f'[{ts()}] nvidia-smi:\n' + out.stdout + (out.stderr or ''), flush=True)
try:
    import torch
    print(f'[{ts()}] torch     :', torch.__version__, '| cuda', torch.version.cuda, flush=True)
    print(f'[{ts()}] device_cnt:', torch.cuda.device_count(), flush=True)
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'[{ts()}]   gpu{i}: {p.name} cc{p.major}.{p.minor} {p.total_memory/1e9:.1f}GB', flush=True)
except Exception as e:
    print(f'[{ts()}] torch not in base kernel ({e}); set up in /tmp venv step.', flush=True)
print(f'[{ts()}] ENV READY -> 2-GPU Blackwell service connected.', flush=True)

In [ ]:
# [DIAG] Kiem tra moi truong hien tai (sau khi service moi) - stream timestamp
import os, sys, glob, subprocess, datetime, shutil
t0 = datetime.datetime.now()
def ts():
    d = datetime.datetime.now()
    return f"[{d.strftime('%H:%M:%S')} +{(d-t0).total_seconds():5.1f}s]"
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print(ts(), 'HOST/py :', sys.version.split()[0], '|', sh('hostname'), flush=True)
print(ts(), 'GPU     :', flush=True)
print(sh("nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv,noheader"), flush=True)

print(ts(), 'DISK /tmp:', sh('df -h /tmp | tail -1'), flush=True)
for p in ['/tmp/google_gemma4_e4b_model', '/tmp/gemma4_quality_run', '/tmp/gemma4_best_run', '/tmp/gemma4_google_unsloth_2gpu_lora']:
    if os.path.isdir(p):
        n = len(glob.glob(p+'/**/*', recursive=True))
        sz = sh(f'du -sh {p} 2>/dev/null').split()[0] if os.path.isdir(p) else '-'
        print(ts(), f'  EXISTS {p}  ({n} files, {sz})', flush=True)
    else:
        print(ts(), f'  MISSING {p}', flush=True)

model_ok = os.path.exists('/tmp/google_gemma4_e4b_model/model.safetensors') or bool(glob.glob('/tmp/google_gemma4_e4b_model/*.safetensors'))
print(ts(), 'MODEL merged present:', model_ok, flush=True)

print(ts(), 'KEY PYTHON PKGS (venv if any):', flush=True)
for mod in ['torch','transformers','huggingface_hub','datasets','trl','unsloth','unsloth_zoo','bitsandbytes','peft','pyarrow','triton','accelerate']:
    v = sh(f'{sys.executable} -c "import {mod},sys; print({mod}.__version__)" 2>/dev/null') or 'NOT INSTALLED'
    print(ts(), f'  {mod:18s} {v}', flush=True)

print(ts(), 'VENV dir /tmp/venv exists:', os.path.isdir('/tmp/venv'), flush=True)
print(ts(), 'DATASET /tmp train.jsonl:', sh('ls -la /tmp/train.jsonl 2>/dev/null') or 'MISSING', flush=True)
print(ts(), 'DONE diag.', flush=True)

In [ ]:
# [DIAG2] Lay version nhanh qua importlib.metadata (khong import nang)
import importlib.metadata as m, datetime
def ts():
    return '['+datetime.datetime.now().strftime('%H:%M:%S')+']'
for p in ['torch','transformers','huggingface_hub','datasets','trl','tokenizers','safetensors','numpy','pyarrow','xformers']:
    try:
        print(ts(), f'{p:18s}', m.version(p), flush=True)
    except Exception:
        print(ts(), f'{p:18s} NOT INSTALLED', flush=True)
print(ts(), 'done', flush=True)

In [ ]:
# [SETUP-1] Tai support pack + model_snapshot tu stage, soi noi dung (stream timestamp)
import os, time, subprocess, datetime
from snowflake.snowpark.context import get_active_session
session = get_active_session()
t0 = time.time()
def ts():
    return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:5.1f}s]'
def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout

STAGE = '@ML_DB.MODELS.FUZZ_DATASET_STAGE'
PACK  = 'fuzz_dataset_stage/google_gemma4_e4b_unsloth_full_quality_pack_20260606'
os.makedirs('/tmp/pack/support', exist_ok=True)
os.makedirs('/tmp/pack/model_snapshot', exist_ok=True)

print(ts(), 'GET support tar.gz (601MB)...', flush=True)
session.file.get(f'{STAGE}/{PACK}/support/gemma4_full_support_pack.tar.gz', '/tmp/pack/support/')
print(ts(), 'GET model_snapshot config files...', flush=True)
session.file.get(f'{STAGE}/{PACK}/model_snapshot/', '/tmp/pack/model_snapshot/')

print(ts(), 'downloaded sizes:', flush=True)
print(sh('ls -la /tmp/pack/support/ /tmp/pack/model_snapshot/'), flush=True)

print(ts(), 'TAR contents (top-level dirs + counts):', flush=True)
print(sh("tar -tzf /tmp/pack/support/gemma4_full_support_pack.tar.gz | sed 's#/.*##' | sort | uniq -c | head -50"), flush=True)
print(ts(), 'TAR: any .safetensors / .whl / .py (sample):', flush=True)
print(sh("tar -tzf /tmp/pack/support/gemma4_full_support_pack.tar.gz | grep -E '\\.(safetensors|whl|py)$' | head -60"), flush=True)
print(ts(), 'DONE setup-1.', flush=True)

In [ ]:
# [SETUP-1b] Path dung: bo tien to ten-stage 'fuzz_dataset_stage/' khi GET
import os, time, subprocess, datetime
from snowflake.snowpark.context import get_active_session
session = get_active_session()
t0 = time.time()
def ts():
    return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:5.1f}s]'
def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout

STAGE = '@ML_DB.MODELS.FUZZ_DATASET_STAGE'
PACK  = 'google_gemma4_e4b_unsloth_full_quality_pack_20260606'
os.makedirs('/tmp/pack/support', exist_ok=True)
os.makedirs('/tmp/pack/model_snapshot', exist_ok=True)

print(ts(), 'GET support tar.gz (601MB)...', flush=True)
r = session.file.get(f'{STAGE}/{PACK}/support/gemma4_full_support_pack.tar.gz', '/tmp/pack/support/')
print(ts(), 'got:', [(x.file, x.size) for x in r], flush=True)
print(ts(), 'GET model_snapshot/ ...', flush=True)
r2 = session.file.get(f'{STAGE}/{PACK}/model_snapshot/', '/tmp/pack/model_snapshot/')
print(ts(), 'model_snapshot files:', [x.file for x in r2], flush=True)

print(ts(), 'local sizes:', flush=True)
print(sh('ls -la /tmp/pack/support/ /tmp/pack/model_snapshot/'), flush=True)
print(ts(), 'TAR top-level (count by first path segment):', flush=True)
print(sh("tar -tzf /tmp/pack/support/gemma4_full_support_pack.tar.gz | sed 's#/.*##' | sort | uniq -c"), flush=True)
print(ts(), 'TAR weights/wheels/scripts sample:', flush=True)
print(sh("tar -tzf /tmp/pack/support/gemma4_full_support_pack.tar.gz | grep -E '\\.(safetensors|whl|py|json)$' | head -80"), flush=True)
print(ts(), 'DONE setup-1b.', flush=True)

In [ ]:
# [SETUP-2] Giai nen support pack + doc manifest/installer de biet env-paths contract
import os, time, subprocess, datetime, json
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:5.1f}s]'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout

print(ts(),'extract support tar -> /tmp/pack ...', flush=True)
sh('cd /tmp/pack && tar -xzf support/gemma4_full_support_pack.tar.gz')
print(ts(),'tree:', flush=True)
print(sh('cd /tmp/pack && ls -la && echo --- && ls -la scripts meta dataset wheels | head -60'), flush=True)

for mf in ['/tmp/pack/meta/full_pack_manifest.json','/tmp/pack/meta/model_manifest.json']:
    if os.path.exists(mf):
        print(ts(),'===',mf,'===', flush=True)
        try: print(json.dumps(json.load(open(mf)), indent=2)[:2500], flush=True)
        except Exception as e: print('  (read err)',e, flush=True)

print(ts(),'=== installer script (full) ===', flush=True)
print(sh('cat /tmp/pack/scripts/SNOWFLAKE_INSTALL_AND_PROBE_FULL_PACK.py'), flush=True)
print(ts(),'DONE setup-2.', flush=True)

In [ ]:
# [SETUP-3] Kiem tra train script da nhung fix chua (khong ton GPU)
import subprocess, datetime
def ts(): return '['+datetime.datetime.now().strftime('%H:%M:%S')+']'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
TS='/tmp/pack/scripts/train_unsloth_gemma4_2gpu_ddp.py'
print(ts(),'lines:', sh(f'wc -l {TS}').strip(), flush=True)
for pat in ['from_pretrained','tokenizer','gradient_checkpoint','group_by_length','neftune','num_kv_shared','proxy','os.environ.get','MAX_LEN|BATCH|LORA_R|MAX_STEPS|GRAD_ACC|EPOCH','save_strategy|resume']:
    print(ts(),f'--- grep "{pat}" ---', flush=True)
    print(sh(f"grep -nE '{pat}' {TS}"), flush=True)
print(ts(),'DONE setup-3.', flush=True)

In [ ]:
# [SETUP-4] Chay installer goi cua ban (tu GET model + cai wheels offline + probe Unsloth)
import time, datetime, io, sys
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
print(ts(),'START installer SNOWFLAKE_INSTALL_AND_PROBE_FULL_PACK.py', flush=True)
code = open('/tmp/pack/scripts/SNOWFLAKE_INSTALL_AND_PROBE_FULL_PACK.py').read()
g = {'__name__':'__main__'}
try:
    exec(code, g)
    print(ts(),'installer finished OK', flush=True)
except SystemExit as e:
    print(ts(),'installer SystemExit', e, flush=True)
except Exception as e:
    import traceback; traceback.print_exc()
    print(ts(),'installer FAILED:', repr(e), flush=True)

In [ ]:
# [SETUP-5] Cai wheels offline --no-deps (bo click-resolve), bo xformers/datasets5, giu base networking
import os, glob, subprocess, sys, time, datetime
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
PY=sys.executable
WH='/tmp/gemma4_full_support/wheels'
if not os.path.isdir(WH): WH='/tmp/pack/wheels'
print(ts(),'wheelhouse:',WH,'py:',PY, flush=True)

EXCL_SUB=['xformers','datasets-','click-','certifi','urllib3','idna-','charset','requests-','protobuf','numpy-','pyarrow','safetensors-','tokenizers-']
def bad_plat(b):
    return ('cp311' in b) or ('cp310' in b) or ('cp39' in b) or ('py39' in b) or ('cp38' in b)
sel=[]
for w in sorted(glob.glob(WH+'/*.whl')):
    b=os.path.basename(w)
    if any(s in b for s in EXCL_SUB):  continue
    if bad_plat(b):                    continue
    sel.append(w)
print(ts(),f'installing {len(sel)} wheels:', flush=True)
for w in sel: print('   ', os.path.basename(w), flush=True)

cmd=[PY,'-m','pip','install','--no-index','--no-deps','--upgrade','--find-links',WH]+sel
print(ts(),'pip ...', flush=True)
p=subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-3000:], flush=True)
if p.returncode!=0:
    print(ts(),'PIP STDERR:', p.stderr[-3000:], flush=True)
print(ts(),'pip rc=',p.returncode, flush=True)

In [ ]:
# [SETUP-6] Test import unsloth/transformers/datasets/trl + AutoConfig (subprocess, stream realtime)
import subprocess, sys, datetime, time
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
code = r'''
import time, sys
t=time.time()
def p(*a): print(f"[+{time.time()-t:5.1f}s]",*a,flush=True)
import os
p("importing unsloth (heavy, patches transformers)...")
import unsloth; p("unsloth", unsloth.__version__)
from unsloth import FastLanguageModel; p("FastLanguageModel ok")
import transformers; p("transformers", transformers.__version__)
import huggingface_hub as h; p("hub", h.__version__)
import datasets; p("datasets", datasets.__version__)
import trl; p("trl", trl.__version__)
import peft, bitsandbytes; p("peft", peft.__version__, "| bnb", bitsandbytes.__version__)
from transformers import AutoConfig
c=AutoConfig.from_pretrained("/tmp/google_gemma4_e4b_model")
p("AutoConfig model_type=", getattr(c,"model_type","?"))
p("ALL IMPORTS OK")
'''
proc = subprocess.Popen([sys.executable,'-u','-c',code], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(ts(),'subprocess started pid', proc.pid, flush=True)
for line in proc.stdout:
    print(ts(), line.rstrip(), flush=True)
proc.wait()
print(ts(),'rc=', proc.returncode, flush=True)

In [ ]:
# [SETUP-7] Tao ban va COPY cua train script (KHONG sua goc). 2 fix tu phien truoc.
import datetime, re, os
def ts(): return '['+datetime.datetime.now().strftime('%H:%M:%S')+']'
SRC='/tmp/gemma4_full_support/scripts/train_unsloth_gemma4_2gpu_ddp.py'
if not os.path.exists(SRC): SRC='/tmp/pack/scripts/train_unsloth_gemma4_2gpu_ddp.py'
DST='/tmp/train_unsloth_gemma4_2gpu_ddp_PATCHED.py'
txt=open(SRC).read()
orig=txt

# fix1: gradient checkpointing -> 'unsloth' (chong OOM o batch/seq cao)
txt=txt.replace('use_gradient_checkpointing=False','use_gradient_checkpointing="unsloth"')

# fix2: unwrap text tokenizer neu la Gemma4Processor (tranh fetch_images)
anchor='if tokenizer.pad_token is None:'
if anchor in txt and 'getattr(tokenizer, "tokenizer"' not in txt:
    txt=txt.replace(anchor, 'tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # PATCH: unwrap Gemma4Processor\n'+anchor, 1)

open(DST,'w').write(txt)
print(ts(),'patched ->',DST, flush=True)
print(ts(),'changed grad_ckpt:', orig.count('use_gradient_checkpointing=False'),'->', txt.count('use_gradient_checkpointing="unsloth"'), flush=True)
import subprocess
print(ts(),'verify lines:', flush=True)
print(subprocess.run(['grep','-nE','use_gradient_checkpointing|getattr\\(tokenizer|pad_token is None',DST],capture_output=True,text=True).stdout, flush=True)
print(ts(),'DONE setup-7.', flush=True)

In [ ]:
# [STRESS] Vat triet de + chat luong cao nhat: 4096x20 r64 alpha128, 20 step, torchrun 2-GPU
# Stream realtime tung dong + timestamp. INTERRUPT de dung.
import os, sys, time, datetime, subprocess, signal
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'

SCRIPT='/tmp/train_unsloth_gemma4_2gpu_ddp_PATCHED.py'
env=dict(os.environ)
env.update({
  'MODEL_DIR':'/tmp/google_gemma4_e4b_model',
  'DATA_PATH':'/tmp/gemma4_full_support/dataset/train.jsonl',
  'OUT_DIR':'/tmp/gemma4_stress_q4096x20_r64',
  'MAX_LEN':'4096','BATCH':'20','GRAD_ACC':'1','MAX_STEPS':'20',
  'LORA_R':'64','LORA_ALPHA':'128','LR':'7e-5',
  'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','TOKENIZERS_PARALLELISM':'false','PYTHONUNBUFFERED':'1',
})
cmd=['torchrun','--standalone','--nproc_per_node=2',SCRIPT]
print(ts(),'launch:', ' '.join(f'{k}={env[k]}' for k in ['MAX_LEN','BATCH','MAX_STEPS','LORA_R','LORA_ALPHA']), flush=True)
print(ts(),'cmd:', ' '.join(cmd), flush=True)
print(ts(),'(model load ~3-6 phut roi moi stream step; INTERRUPT de dung)', flush=True)

proc=subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(ts(), line.rstrip(), flush=True)
    proc.wait()
    print(ts(),'EXIT rc=',proc.returncode, flush=True)
except KeyboardInterrupt:
    print(ts(),'INTERRUPT -> terminating torchrun...', flush=True)
    proc.send_signal(signal.SIGINT); time.sleep(5)
    if proc.poll() is None: proc.terminate(); time.sleep(3)
    if proc.poll() is None: proc.kill()
    print(ts(),'stopped.', flush=True)

In [ ]:
# [STRESS-2] Fix: dung VENV python lam launcher (torchrun PATH tro nham /opt python -> thieu unsloth)
import os, sys, time, datetime, subprocess, signal
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
SCRIPT='/tmp/train_unsloth_gemma4_2gpu_ddp_PATCHED.py'
env=dict(os.environ)
env.update({
  'MODEL_DIR':'/tmp/google_gemma4_e4b_model',
  'DATA_PATH':'/tmp/gemma4_full_support/dataset/train.jsonl',
  'OUT_DIR':'/tmp/gemma4_stress_q4096x20_r64',
  'MAX_LEN':'4096','BATCH':'20','GRAD_ACC':'1','MAX_STEPS':'20',
  'LORA_R':'64','LORA_ALPHA':'128','LR':'7e-5',
  'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','TOKENIZERS_PARALLELISM':'false','PYTHONUNBUFFERED':'1',
})
cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',SCRIPT]
print(ts(),'py launcher:', sys.executable, flush=True)
print(ts(),'launch 4096x20 r64 a128 steps20 | model load ~3-6min then stream. INTERRUPT de dung.', flush=True)
proc=subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(ts(), line.rstrip(), flush=True)
    proc.wait(); print(ts(),'EXIT rc=',proc.returncode, flush=True)
except KeyboardInterrupt:
    print(ts(),'INTERRUPT -> stop torchrun...', flush=True)
    proc.send_signal(signal.SIGINT); time.sleep(5)
    if proc.poll() is None: proc.terminate(); time.sleep(3)
    if proc.poll() is None: proc.kill()
    print(ts(),'stopped.', flush=True)

In [ ]:
# [STRESS-3] Chay lai, luu full log /tmp/stress.log, stream dong quan trong, in tail de thay loi that
import os, sys, time, datetime, subprocess, signal, re
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
SCRIPT='/tmp/train_unsloth_gemma4_2gpu_ddp_PATCHED.py'; LOG='/tmp/stress.log'
env=dict(os.environ); env.update({
  'MODEL_DIR':'/tmp/google_gemma4_e4b_model','DATA_PATH':'/tmp/gemma4_full_support/dataset/train.jsonl',
  'OUT_DIR':'/tmp/gemma4_stress_q4096x20_r64','MAX_LEN':'4096','BATCH':'20','GRAD_ACC':'1','MAX_STEPS':'20',
  'LORA_R':'64','LORA_ALPHA':'128','LR':'7e-5','HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1',
  'TOKENIZERS_PARALLELISM':'false','PYTHONUNBUFFERED':'1','TORCHELASTIC_ERROR_FILE':'/tmp/torch_err.json'})
KEY=re.compile(r'(loss|step|frize|froze|trainable|START|READY|Error|error|Traceback|Exception|OOM|CUDA|num_kv|RuntimeError|AttributeError|[Ii]mage|VRAM|world=|saved|SUMMARY|\{)')
cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',SCRIPT]
print(ts(),'launch (filtered stream). INTERRUPT de dung.', flush=True)
lines=[]
with open(LOG,'w') as f:
    proc=subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            f.write(line); f.flush(); lines.append(line.rstrip())
            if KEY.search(line): print(ts(), line.rstrip(), flush=True)
        proc.wait()
    except KeyboardInterrupt:
        print(ts(),'INTERRUPT', flush=True); proc.send_signal(signal.SIGINT); time.sleep(5)
        if proc.poll() is None: proc.terminate(); time.sleep(3)
        if proc.poll() is None: proc.kill()
print(ts(),'EXIT rc=',proc.returncode, flush=True)
print(ts(),'===== TAIL 50 =====', flush=True)
for l in lines[-50:]: print('   ',l, flush=True)

In [ ]:
# [DIAG-ERR] Lay dung traceback tu log (output nho, khong bi truncate)
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print('=== matches error/traceback/cuda/oom ===', flush=True)
print(sh("grep -niE 'error|traceback|exception|runtimeerror|out of memory|oom|cuda|image|assert|nan' /tmp/stress.log | head -60"), flush=True)
print('=== first Traceback block (40 lines after) ===', flush=True)
print(sh("grep -n -m1 'Traceback' /tmp/stress.log"), flush=True)
print(sh("awk '/Traceback/{f=1} f{print; n++} n>45{exit}' /tmp/stress.log"), flush=True)

In [ ]:
# [STRESS-4] Quality-max OOM-safe: 4096x16 r64 a128 + PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
import os, sys, time, datetime, subprocess, signal, re
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
SCRIPT='/tmp/train_unsloth_gemma4_2gpu_ddp_PATCHED.py'; LOG='/tmp/stress4.log'
env=dict(os.environ); env.update({
  'MODEL_DIR':'/tmp/google_gemma4_e4b_model','DATA_PATH':'/tmp/gemma4_full_support/dataset/train.jsonl',
  'OUT_DIR':'/tmp/gemma4_stress_q4096x16_r64','MAX_LEN':'4096','BATCH':'16','GRAD_ACC':'1','MAX_STEPS':'20',
  'LORA_R':'64','LORA_ALPHA':'128','LR':'7e-5','HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1',
  'TOKENIZERS_PARALLELISM':'false','PYTHONUNBUFFERED':'1','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'})
KEY=re.compile(r'(loss|step|froze|trainable|START|READY|Error|Traceback|Exception|OutOfMemory|out of memory|RuntimeError|VRAM|world=|saved|SUMMARY|examples|it/s|s/it|\{)')
cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',SCRIPT]
print(ts(),'launch 4096x16 r64 a128 steps20 expandable. model load ~3min. INTERRUPT de dung.', flush=True)
lines=[]
with open(LOG,'w') as f:
    proc=subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            f.write(line); f.flush(); lines.append(line.rstrip())
            if KEY.search(line): print(ts(), line.rstrip()[:200], flush=True)
        proc.wait()
    except KeyboardInterrupt:
        print(ts(),'INTERRUPT', flush=True); proc.send_signal(signal.SIGINT); time.sleep(5)
        if proc.poll() is None: proc.terminate(); time.sleep(3)
        if proc.poll() is None: proc.kill()
print(ts(),'EXIT rc=',proc.returncode, flush=True)
print(ts(),'=== TAIL 30 ===', flush=True)
for l in lines[-30:]: print('   ',l[:200], flush=True)

In [ ]:
# In tron script da va theo khoi 40 dong de khong bi truncate
lines=open('/tmp/train_unsloth_gemma4_2gpu_ddp_PATCHED.py').read().split('\n')
N=len(lines)
for a in range(0,N,40):
    print(f'\n=========== LINES {a+1}..{min(a+40,N)} ===========', flush=True)
    for i in range(a,min(a+40,N)):
        print(f'{i+1:3d}| {lines[i]}', flush=True)

In [ ]:
lines=open('/tmp/train_unsloth_gemma4_2gpu_ddp_PATCHED.py').read().split('\n')
N=len(lines); CH=28
for a in range(0,N,CH):
    blk='\n'.join(f'{i+1:3d}| {lines[i]}' for i in range(a,min(a+CH,N)))
    print(f'===== {a+1}..{min(a+CH,N)} =====\n'+blk, flush=True)

In [ ]:
# [WORKER-START] Bat worker thuong tru 2-GPU: load model+LoRA+dataset 1 lan, roi nhan job qua file-queue.
# Co dinh MAX_LEN=4096, r=64, alpha=128 (quality). Job doi: batch/grad_acc/steps/lr/out/save (KHONG reload).
import os, sys, time, datetime, subprocess, signal
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'

SERVER = r'''
import os, sys, json, time, glob
from pathlib import Path
os.environ.setdefault("HF_HUB_OFFLINE","1"); os.environ.setdefault("TRANSFORMERS_OFFLINE","1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM","false"); os.environ.setdefault("CUDA_DEVICE_MAX_CONNECTIONS","1")
import torch, torch.distributed as dist
from datasets import load_dataset
from transformers import Trainer, TrainingArguments, TrainerCallback
import unsloth
try:
    import unsloth_zoo.temporary_patches.gemma4 as g4patch
    if hasattr(g4patch, "_Gemma4KVSharedSafeProxy"):
        def _pt(self,name): return getattr(object.__getattribute__(self,"_real"),name)
        g4patch._Gemma4KVSharedSafeProxy.__getattr__=_pt
except Exception as e:
    print("proxy patch warning",repr(e),flush=True)
from unsloth import FastLanguageModel
MODEL_DIR=os.environ.get("MODEL_DIR","/tmp/google_gemma4_e4b_model")
DATA_PATH=os.environ.get("DATA_PATH","/tmp/gemma4_full_support/dataset/train.jsonl")
MAX_LEN=int(os.environ.get("MAX_LEN","4096")); LORA_R=int(os.environ.get("LORA_R","64")); LORA_ALPHA=int(os.environ.get("LORA_ALPHA","128"))
SYSTEM_PROMPT=os.environ.get("SYSTEM_PROMPT","You are an assistant for defensive fuzzing of PyTorch/TensorFlow APIs.")
rank=int(os.environ.get("RANK","0")); local_rank=int(os.environ.get("LOCAL_RANK","0"))
if torch.cuda.is_available(): torch.cuda.set_device(local_rank)
def log(x):
    if rank==0: print(f"[{time.strftime('%H:%M:%S')}] {x}",flush=True)
log(f"loading model={MODEL_DIR} max_len={MAX_LEN} r={LORA_R} a={LORA_ALPHA}")
model,tokenizer=FastLanguageModel.from_pretrained(model_name=MODEL_DIR,max_seq_length=MAX_LEN,load_in_4bit=False,load_in_16bit=True,full_finetuning=False,local_files_only=True)
tokenizer=getattr(tokenizer,"tokenizer",tokenizer)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=FastLanguageModel.get_peft_model(model,r=LORA_R,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=LORA_ALPHA,lora_dropout=0.0,bias="none",use_rslora=True,use_gradient_checkpointing="unsloth")
bad=["vision_tower","audio_tower","embed_vision","embed_audio"]; froze=0
for n,p in model.named_parameters():
    if any(k in n for k in bad):
        if p.requires_grad: froze+=1
        p.requires_grad_(False)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"froze={froze} trainable={trainable/1e6:.1f}M")
def split_row(ex):
    msgs=ex.get("messages")
    if msgs:
        if isinstance(msgs,str): msgs=json.loads(msgs)
        la=None
        for i in range(len(msgs)-1,-1,-1):
            if msgs[i].get("role")=="assistant": la=i; break
        if la is not None:
            pm=msgs[:la]; completion=str(msgs[la].get("content",""))
            prompt=tokenizer.apply_chat_template(pm,tokenize=False,add_generation_prompt=True)
            return prompt,completion,False
        return "",tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=False),True
    prompt=ex.get("prompt") or ex.get("instruction") or ex.get("input")
    completion=ex.get("completion") or ex.get("output") or ex.get("response")
    if prompt is not None and completion is not None:
        pm=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":str(prompt)}]
        return tokenizer.apply_chat_template(pm,tokenize=False,add_generation_prompt=True),str(completion),False
    return "",str(ex.get("text") or ""),True
def tok_one(ex):
    p,c,ft=split_row(ex)
    if ft:
        ids=tokenizer(c,add_special_tokens=False)["input_ids"][:MAX_LEN]; labels=ids.copy()
    else:
        pids=tokenizer(p,add_special_tokens=False)["input_ids"]; cids=tokenizer(c,add_special_tokens=False)["input_ids"]+[tokenizer.eos_token_id]
        if len(pids)+len(cids)>MAX_LEN:
            kp=max(0,MAX_LEN-len(cids)); pids=pids[-kp:] if kp else []; cids=cids[:MAX_LEN-len(pids)]
        ids=pids+cids; labels=[-100]*len(pids)+cids
    return {"input_ids":ids,"labels":labels,"attention_mask":[1]*len(ids),"n_tokens":len(ids)}
class PadCollator:
    def __init__(self,tk): self.tk=tk
    def __call__(self,fts):
        ml=max(len(f["input_ids"]) for f in fts); pid=self.tk.pad_token_id
        ii,ll,am=[],[],[]
        for f in fts:
            n=len(f["input_ids"]); pad=ml-n
            ii.append(f["input_ids"]+[pid]*pad); ll.append(f["labels"]+[-100]*pad); am.append(f["attention_mask"]+[0]*pad)
        return {"input_ids":torch.tensor(ii,dtype=torch.long),"labels":torch.tensor(ll,dtype=torch.long),"attention_mask":torch.tensor(am,dtype=torch.long)}
log("loading dataset")
ds=load_dataset("json",data_files={"train":DATA_PATH})["train"]
ds=ds.map(tok_one,remove_columns=ds.column_names,desc="tok")
ds=ds.filter(lambda x: len(x["input_ids"])>0 and any(v!=-100 for v in x["labels"]),desc="flt")
log(f"dataset rows={len(ds)} avg_tokens={sum(ds['n_tokens'])/max(1,len(ds)):.1f}")
WDIR="/tmp/worker"; JOBS=WDIR+"/jobs"; RES=WDIR+"/results"
os.makedirs(JOBS,exist_ok=True); os.makedirs(RES,exist_ok=True)
def ready_ids(): return sorted(int(Path(p).stem) for p in glob.glob(JOBS+"/*.ready"))
last=max(ready_ids(),default=0)
dev=torch.device(f"cuda:{local_rank}")
class StopCB(TrainerCallback):
    def __init__(self,jid): self.flags=[WDIR+"/STOP",WDIR+f"/STOP_{jid}"]
    def on_step_end(self,a,s,c,**k):
        loc=1.0 if any(os.path.exists(f) for f in self.flags) else 0.0
        if dist.is_initialized():
            t=torch.tensor([loc],device=dev); dist.all_reduce(t,op=dist.ReduceOp.MAX); loc=t.item()
        if loc>0: c.should_training_stop=True
        return c
log(f"WORKER READY (resident) last_job={last} world={torch.cuda.device_count()}")
while True:
    todo=[i for i in ready_ids() if i>last]
    if not todo: time.sleep(2); continue
    jid=todo[0]
    try: spec=json.load(open(JOBS+f"/{jid}.json"))
    except Exception: time.sleep(1); continue
    BATCH=int(spec.get("batch",16)); GACC=int(spec.get("grad_acc",1)); STEPS=int(spec.get("max_steps",20))
    LR=float(spec.get("lr",7e-5)); OUT=spec.get("out_dir",f"/tmp/gemma4_job_{jid}"); SAVE=bool(spec.get("save",False)); SS=int(spec.get("save_steps",50))
    logf=JOBS+f"/{jid}.log"
    def jlog(x):
        if rank==0:
            open(logf,"a").write(f"[{time.strftime('%H:%M:%S')}] {x}\n"); print(f"[job{jid}] {x}",flush=True)
    class LogCB(TrainerCallback):
        def on_log(self,a,s,c,logs=None,**k):
            if rank==0 and logs:
                d={kk:(round(vv,4) if isinstance(vv,float) else vv) for kk,vv in logs.items()}; d["step"]=s.global_step
                jlog(json.dumps(d))
            return c
    torch.cuda.reset_peak_memory_stats()
    args=TrainingArguments(output_dir=OUT,bf16=True,tf32=True,per_device_train_batch_size=BATCH,gradient_accumulation_steps=GACC,max_steps=STEPS,learning_rate=LR,warmup_ratio=0.03,lr_scheduler_type="cosine",optim="adamw_torch_fused",logging_steps=1,save_steps=(SS if SAVE else 10**9),save_total_limit=2,report_to="none",remove_unused_columns=False,ddp_find_unused_parameters=False,dataloader_num_workers=2,dataloader_pin_memory=True)
    jlog(f"START batch={BATCH} gacc={GACC} steps={STEPS} lr={LR} out={OUT} save={SAVE}")
    tt=time.time()
    try:
        tr=Trainer(model=model,args=args,train_dataset=ds,data_collator=PadCollator(tokenizer),callbacks=[StopCB(jid),LogCB()])
        res=tr.train(); secs=time.time()-tt
        if rank==0:
            if SAVE: tr.save_model(OUT); tokenizer.save_pretrained(OUT)
            vram=torch.cuda.max_memory_allocated()/1024**3
            summ={"job":jid,"steps":res.global_step,"train_seconds":round(secs,1),"loss":round(res.training_loss,4),"max_vram_gb":round(vram,1),"out_dir":OUT,"saved":SAVE}
            json.dump(summ,open(RES+f"/{jid}.done","w"),indent=2); jlog("DONE "+json.dumps(summ))
    except Exception as e:
        import traceback; tb=traceback.format_exc()
        if rank==0:
            json.dump({"job":jid,"error":repr(e)},open(RES+f"/{jid}.done","w")); jlog("ERROR "+repr(e)); open(logf,"a").write(tb)
    if dist.is_initialized(): dist.barrier()
    for f in [WDIR+"/STOP",WDIR+f"/STOP_{jid}"]:
        if os.path.exists(f):
            try: os.remove(f)
            except: pass
    last=jid
'''
os.makedirs('/tmp/worker', exist_ok=True)
open('/tmp/worker_server.py','w').write(SERVER)
env=dict(os.environ); env.update({'MODEL_DIR':'/tmp/google_gemma4_e4b_model','DATA_PATH':'/tmp/gemma4_full_support/dataset/train.jsonl','MAX_LEN':'4096','LORA_R':'64','LORA_ALPHA':'128','HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1'})
LOG='/tmp/worker/server.log'; open(LOG,'w').close()
cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/worker_server.py']
proc=subprocess.Popen(cmd, env=env, stdout=open(LOG,'a'), stderr=subprocess.STDOUT, text=True)
open('/tmp/worker/server.pid','w').write(str(proc.pid))
print(ts(),'worker launched pid',proc.pid,'| waiting for READY (model load ~3min)...', flush=True)
import re
pos=0; ready=False; deadline=time.time()+540
while time.time()<deadline:
    if proc.poll() is not None:
        print(ts(),'WORKER PROCESS EXITED rc=',proc.returncode,'(see tail below)', flush=True); break
    with open(LOG) as f: f.seek(pos); new=f.read(); pos=f.tell()
    for ln in new.splitlines():
        if re.search(r'READY|froze=|dataset rows=|loading model|loading dataset|Error|Traceback|OutOfMemory|warning', ln):
            print(ts(), ln[:200], flush=True)
        if 'WORKER READY' in ln: ready=True
    if ready: break
    time.sleep(2)
print(ts(),'READY' if ready else 'NOT READY (check tail)', flush=True)
print(ts(),'--- server.log tail ---', flush=True)
print(subprocess.run(['tail','-n','12',LOG],capture_output=True,text=True).stdout, flush=True)

In [ ]:
# [WORKER-RUN] Gui 1 job toi worker (KHONG reload). Sua params ben duoi roi Run.
# 'Train den khi dung': dat JOB_STEPS rat lon + JOB_SAVE=True, roi chay [WORKER-STOP] de dung+luu.
JOB_BATCH      = 16
JOB_GRAD_ACC   = 1
JOB_STEPS      = 20
JOB_LR         = 7e-5
JOB_OUT        = '/tmp/gemma4_worker_run'
JOB_SAVE       = False
JOB_SAVE_STEPS = 50

import os, glob, json, time, datetime
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
WDIR='/tmp/worker'; JOBS=WDIR+'/jobs'; RES=WDIR+'/results'
if not os.path.exists(WDIR+'/server.pid'):
    print('WORKER chua chay. Chay [WORKER-START] truoc.'); raise SystemExit
os.makedirs(JOBS,exist_ok=True); os.makedirs(RES,exist_ok=True)
ids=[int(os.path.basename(p)[:-5]) for p in glob.glob(JOBS+'/*.json')]
jid=(max(ids)+1) if ids else 1
spec=dict(batch=JOB_BATCH,grad_acc=JOB_GRAD_ACC,max_steps=JOB_STEPS,lr=JOB_LR,out_dir=JOB_OUT,save=JOB_SAVE,save_steps=JOB_SAVE_STEPS)
json.dump(spec,open(JOBS+f'/{jid}.json','w')); open(JOBS+f'/{jid}.ready','w').close()
print(ts(),f'submitted job {jid}:',spec, flush=True)
print(ts(),'streaming (INTERRUPT chi dung xem, job van chay; dung han bang [WORKER-STOP])', flush=True)
logf=JOBS+f'/{jid}.log'; donef=RES+f'/{jid}.done'; pos=0
try:
    while True:
        if os.path.exists(logf):
            with open(logf) as f: f.seek(pos); new=f.read(); pos=f.tell()
            for ln in new.splitlines(): print(ts(), ln, flush=True)
        if os.path.exists(donef):
            time.sleep(0.5)
            with open(logf) as f: f.seek(pos); print(f.read(), flush=True)
            print(ts(),'=== JOB DONE ===', flush=True); print(open(donef).read(), flush=True); break
        time.sleep(1)
except KeyboardInterrupt:
    print(ts(),'stopped watching (job tiep tuc chay tren worker).', flush=True)

In [ ]:
# [WORKER-STOP] Dung job dang chay (graceful, luu neu save=True). Worker van song de chay job khac.
# De TAT han worker + giai phong GPU (het ton credit): dat SHUTDOWN=True.
SHUTDOWN = False
import os, time, datetime, signal
def ts(): return '['+datetime.datetime.now().strftime('%H:%M:%S')+']'
WDIR='/tmp/worker'
open(WDIR+'/STOP','w').close()
print(ts(),'wrote STOP flag -> job se dung o step ke tiep (graceful).', flush=True)
if SHUTDOWN:
    try:
        pid=int(open(WDIR+'/server.pid').read()); os.kill(pid, signal.SIGINT); time.sleep(3)
        try: os.kill(pid, signal.SIGTERM)
        except: pass
        print(ts(),'worker shutdown signal sent pid',pid,'(GPU se giai phong).', flush=True)
    except Exception as e:
        print(ts(),'shutdown err',repr(e), flush=True)

In [ ]:
# [WORKER-RUN b8] Job batch=8 @ MAX_LEN=4096 (worst-case logits ~17GB, an toan). Khong reload.
JOB_BATCH, JOB_GRAD_ACC, JOB_STEPS, JOB_LR = 8, 1, 20, 7e-5
JOB_OUT, JOB_SAVE, JOB_SAVE_STEPS = '/tmp/gemma4_worker_b8', False, 50
import os, glob, json, time, datetime
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
WDIR='/tmp/worker'; JOBS=WDIR+'/jobs'; RES=WDIR+'/results'
ids=[int(os.path.basename(p)[:-5]) for p in glob.glob(JOBS+'/*.json')]
jid=(max(ids)+1) if ids else 1
spec=dict(batch=JOB_BATCH,grad_acc=JOB_GRAD_ACC,max_steps=JOB_STEPS,lr=JOB_LR,out_dir=JOB_OUT,save=JOB_SAVE,save_steps=JOB_SAVE_STEPS)
json.dump(spec,open(JOBS+f'/{jid}.json','w')); open(JOBS+f'/{jid}.ready','w').close()
print(ts(),f'submitted job {jid}:',spec, flush=True)
logf=JOBS+f'/{jid}.log'; donef=RES+f'/{jid}.done'; pos=0
try:
    while True:
        if os.path.exists(logf):
            with open(logf) as f: f.seek(pos); new=f.read(); pos=f.tell()
            for ln in new.splitlines(): print(ts(), ln, flush=True)
        if os.path.exists(donef):
            time.sleep(0.5)
            with open(logf) as f: f.seek(pos); print(f.read(), flush=True)
            print(ts(),'=== JOB DONE ===', flush=True); print(open(donef).read(), flush=True); break
        time.sleep(1)
except KeyboardInterrupt:
    print(ts(),'stopped watching (job van chay).', flush=True)

In [ ]:
# [WORKER-RESTART] Kill worker cu (giai phong GPU) + don job + bat lai o MAX_LEN=2048 (on dinh, batch<=16 an toan)
import os, sys, time, datetime, subprocess, signal, glob, re
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
WDIR='/tmp/worker'
# 1) kill old
sh('pkill -9 -f worker_server.py'); sh('pkill -9 -f torch.distributed.run'); time.sleep(4)
print(ts(),'after kill, GPU mem:', flush=True)
print(sh('nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv,noheader'), flush=True)
# 2) clean job queue (giu lai log cu? xoa de sach)
for d in ['jobs','results']:
    for f in glob.glob(f'{WDIR}/{d}/*'):
        try: os.remove(f)
        except: pass
for f in glob.glob(f'{WDIR}/STOP*'):
    try: os.remove(f)
    except: pass
# 3) relaunch MAX_LEN=2048
env=dict(os.environ); env.update({'MODEL_DIR':'/tmp/google_gemma4_e4b_model','DATA_PATH':'/tmp/gemma4_full_support/dataset/train.jsonl','MAX_LEN':'2048','LORA_R':'64','LORA_ALPHA':'128','HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','PYTHONUNBUFFERED':'1'})
LOG=WDIR+'/server.log'; open(LOG,'w').close()
cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2','/tmp/worker_server.py']
proc=subprocess.Popen(cmd, env=env, stdout=open(LOG,'a'), stderr=subprocess.STDOUT, text=True, start_new_session=True)
open(WDIR+'/server.pid','w').write(str(proc.pid))
print(ts(),'relaunched pid',proc.pid,'MAX_LEN=2048 | waiting READY...', flush=True)
pos=0; ready=False; deadline=time.time()+540
while time.time()<deadline:
    if proc.poll() is not None:
        print(ts(),'EXITED rc=',proc.returncode, flush=True); break
    with open(LOG) as f: f.seek(pos); new=f.read(); pos=f.tell()
    for ln in new.splitlines():
        if re.search(r'READY|froze=|dataset rows=|loading|Error|Traceback|OutOfMemory', ln): print(ts(), ln[:200], flush=True)
        if 'WORKER READY' in ln: ready=True
    if ready: break
    time.sleep(2)
print(ts(),'READY' if ready else 'NOT READY', flush=True)

In [ ]:
# [WORKER-SWEEP] Tang BATCH de vat GPU, giu MAX_LEN=2048 r64 (chat luong khong doi). KHONG reload.
# Dung ngay neu OOM (de khong hong worker). Do VRAM + samples/s + ~tok/s.
BATCHES=[24,32,40]; STEPS=15; LR=7e-5; AVG_TOK=437.0; WORLD=2
import os, glob, json, time, datetime
t0=time.time()
def ts(): return f'[{datetime.datetime.now().strftime("%H:%M:%S")} +{time.time()-t0:6.1f}s]'
WDIR='/tmp/worker'; JOBS=WDIR+'/jobs'; RES=WDIR+'/results'
if not os.path.exists(WDIR+'/server.pid'): print('worker chua chay'); raise SystemExit
rows=[]
for b in BATCHES:
    ids=[int(os.path.basename(p)[:-5]) for p in glob.glob(JOBS+'/*.json')]
    jid=(max(ids)+1) if ids else 1
    spec=dict(batch=b,grad_acc=1,max_steps=STEPS,lr=LR,out_dir=f'/tmp/gemma4_sweep_b{b}',save=False,save_steps=10**9)
    json.dump(spec,open(JOBS+f'/{jid}.json','w')); open(JOBS+f'/{jid}.ready','w').close()
    print(ts(),f'--- submit batch={b} (job {jid}) ---', flush=True)
    logf=JOBS+f'/{jid}.log'; donef=RES+f'/{jid}.done'; pos=0; oom=False
    while True:
        if os.path.exists(logf):
            with open(logf) as f: f.seek(pos); new=f.read(); pos=f.tell()
            for ln in new.splitlines():
                if '"step":' in ln or 'START' in ln or 'ERROR' in ln or 'DONE' in ln: print(ts(), ln[-140:], flush=True)
        if os.path.exists(donef):
            s=json.load(open(donef))
            if 'error' in s:
                print(ts(),'OOM/ERROR @batch',b,'->',s['error'][:120], flush=True); oom=True
            else:
                sps=(b*WORLD*s['steps'])/s['train_seconds']; toks=sps*AVG_TOK
                rows.append((b,s['max_vram_gb'],round(sps,2),int(toks),s['loss']))
                print(ts(),f"OK batch={b} VRAM={s['max_vram_gb']}GB samples/s={sps:.2f} ~tok/s={int(toks)} loss={s['loss']}", flush=True)
            break
        time.sleep(1)
    if oom: print(ts(),'STOP sweep (worker can restart sau OOM).', flush=True); break
print('\n'+ts(),'===== RANK (tok/s) =====', flush=True)
for r in sorted(rows,key=lambda x:-x[3]):
    print(f'  batch={r[0]:>3}  VRAM={r[1]:>5}GB  samples/s={r[2]:>6}  ~tok/s={r[3]:>7}  loss={r[4]}', flush=True)